# Reproducible original PriceTracker baseline

The original fast-renderer TypeScript sources, compiled JavaScript, npm dependencies and
third-party licenses are embedded in this notebook. No PriceTracker checkout or npm installation
is needed. Node.js executes the original code; Python evaluates results and creates Excel reports.

Default methods: `original_fast` (all page products), `original_fast_query` (original query filter).
Both reproduce the methods in `original_comparison.xlsx`. B1/B2 below are retained only as optional
historical controls; they are not the default baseline for the paper.

Reference environment: Python 3.13.13, Node.js 24.6.0. Place the notebook next to `industrial/`
and `synthetic/`, then Restart Kernel → Run All. Results: `baseline_results/determined/`.


In [1]:
%pip install "pandas==3.0.6" "beautifulsoup4==4.15.0" "lxml==6.1.3" "openpyxl==3.1.5"


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from __future__ import annotations

from pathlib import Path
from urllib.parse import urljoin, urlsplit
from difflib import SequenceMatcher
import hashlib
import html as html_std
import importlib.util
import json
import math
import re
import zipfile
import xml.etree.ElementTree as ET

import pandas as pd
from bs4 import BeautifulSoup

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)

from datetime import datetime, timezone
import uuid

from time import perf_counter
import traceback


## 1. Configuration

The benchmark is self-contained. Put the notebook in the benchmark directory with this layout:

```text
benchmark/
├─ baseline_determined.ipynb
├─ config/requirements-benchmark.lock.txt     # pinned environment
├─ scripts/parse_logs_to_excel_v2.py          # optional parity check only
├─ synthetic/
│  ├─ html/
│  └─ [dind.xlsx | synthetic_ground_truth.xlsx | synthetic_ground_truth.csv | ...]
└─ industrial/
   ├─ html/
   └─ dind.xlsx
```

The notebook automatically finds the **nearest directory containing both `synthetic/html/` and `industrial/html/`**. Normally this is simply the notebook's working directory.

It does not depend on environment variables or on the original PriceTracker repository. If `scripts/parse_logs_to_excel_v2.py` is present, it is used only for an optional runtime parity check; the evaluation contract is already embedded in the notebook.

Outputs are written to `baseline_results/determined/` next to the datasets.

Synthetic data has two separate splits sharing synthetic/html: synthetic uses
synthetic_ground_truth.csv + synthetic_pages.csv (20 clean pages); robustness
uses robustness_ground_truth.csv + robustness_pages.csv (200 mutations).
RUN_DATASETS=None runs everything. RUN_DATASETS=("synthetic", "robustness")
reruns those splits and retains verified industrial results in the combined files.


In [3]:
# ----------------------------------------
# Self-contained benchmark configuration
# ----------------------------------------

def _find_benchmark_root(start: Path | None = None) -> Path:
    """Find the nearest folder that contains both benchmark HTML directories."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "synthetic" / "html").is_dir()
            and (candidate / "industrial" / "html").is_dir()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not find a benchmark root containing both "
        "'synthetic/html/' and 'industrial/html/'. "
        "Place this notebook next to the synthetic/ and industrial/ folders "
        "or start Jupyter from that directory."
    )


BENCHMARK_ROOT = _find_benchmark_root()

DATASETS = {
    "synthetic": {
        "root": BENCHMARK_ROOT / "synthetic",
        "html_dir": BENCHMARK_ROOT / "synthetic" / "html",
        "reference_file": None,  # auto-discovered under synthetic/
    },
    "robustness": {
        "root": BENCHMARK_ROOT / "synthetic",
        "html_dir": BENCHMARK_ROOT / "synthetic" / "html",
        "reference_file": BENCHMARK_ROOT / "synthetic" / "robustness_ground_truth.csv",
    },
    "industrial": {
        "root": BENCHMARK_ROOT / "industrial",
        "html_dir": BENCHMARK_ROOT / "industrial" / "html",
        "reference_file": None,  # auto-discovered under industrial/
    },
}

OUTPUT_DIR = BENCHMARK_ROOT / "baseline_results" / "determined"

# Synthetic v2: rerun the two updated splits. Set None to run all datasets.
# 20 clean pages / 420 products; 200 mutations / 4200 products.
# Other datasets' verified saved results are retained in the combined CSVs.
RUN_DATASETS = ("synthetic", "robustness")

# Exact original parser modes used in original_comparison.xlsx.
VARIANTS = ("original_fast", "original_fast_query")
# Optional historical controls: append "b1_jsonld", "b2_css" or "b3_regex".
ORIGINAL_RUNTIME_DIR = BENCHMARK_ROOT / ".benchmark_runtime"
# False always performs fresh extraction. True resumes verified input/code checkpoints.
RESUME_EXTRACTION = False

NAME_THRESHOLD = 0.85
PRICE_REL_TOL = 0.01
MAX_CANDIDATES_PER_HTML = 5000
EXCLUDE_DERIVED_ARTIFACT_HTML = True

# Pages that remain completely unannotated are excluded from metric computation,
# while extraction predictions are still saved.
FAIL_ON_INCOMPLETE_GT = False

# The current industrial GT was reviewed by the user, although most rows still
# carry in_progress. "labeled" accepts those rows; "done" requires review status.
# Explicit pending/todo/new/skip rows never become labels merely by having values.
ANNOTATION_POLICY = "labeled"  # "labeled" or "done"
BENCHMARK_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]

# Optional parser-contract file. It is NOT required for the benchmark.
# If a copy is placed next to this notebook, a parity check is performed.
PARSER_CONTRACT_FILE: Path | None = None

print("BENCHMARK_ROOT =", BENCHMARK_ROOT)
print("OUTPUT_DIR =", OUTPUT_DIR)
for name, cfg in DATASETS.items():
    print(f"{name}: HTML={cfg['html_dir']} reference={cfg['reference_file'] or 'AUTO'}")


BENCHMARK_ROOT = D:\Pet_Codes\science\article-price-tracker
OUTPUT_DIR = D:\Pet_Codes\science\article-price-tracker\baseline_results\determined
synthetic: HTML=D:\Pet_Codes\science\article-price-tracker\synthetic\html reference=AUTO
robustness: HTML=D:\Pet_Codes\science\article-price-tracker\synthetic\html reference=D:\Pet_Codes\science\article-price-tracker\synthetic\robustness_ground_truth.csv
industrial: HTML=D:\Pet_Codes\science\article-price-tracker\industrial\html reference=AUTO


## 2. Shared evaluation contract

Name matching uses SequenceMatcher >= 0.85; prices use ±1% tolerance.
URL identity additionally treats www and the bare host as equivalent. Query,
fragment and trailing slash are removed; other subdomains remain distinct.
The optional parser parity check compares URLs under this application policy.
HTML identity preserves the relative path, including subdirectories.


In [4]:

SRC_URL_RE = re.compile(r"^https?://(?:www\.)?([^/]+)")
JSONLD_RE = re.compile(
    r'<script[^>]*type\s*=\s*["\']application/ld\+json["\'][^>]*>(.*?)</script>',
    re.DOTALL | re.IGNORECASE,
)
BENCHMARK_FILENAME_RE = re.compile(
    r"^(?P<source>.+)-(?P<run>run-(?:[0-9a-f]+|none\d+))(?:-(?P<copy>\d+))?$",
    re.IGNORECASE,
)


def _clean_text(value: object) -> str:
    if value is None or value is pd.NA or (isinstance(value, float) and math.isnan(value)):
        return ""
    text = html_std.unescape(str(value))
    text = text.replace("\u00a0", " ").replace("\u202f", " ").replace("\u2009", " ")
    return re.sub(r"\s+", " ", text).strip()


def _is_blank(x: object) -> bool:
    if x is None:
        return True
    if pd.isna(x):
        return True
    return str(x).strip() == ""


def _first_nonempty(*values: object) -> str:
    for value in values:
        text = _clean_text(value)
        if text:
            return text
    return ""


def normalize_source(s: str | None) -> str | None:
    # Exact contract from parse_logs_to_excel_v2.py.
    if _is_blank(s):
        return None
    s = str(s).strip().strip("'").strip('"').lower()
    m = SRC_URL_RE.match(s)
    if m:
        s = m.group(1)
    if s.startswith("www."):
        s = s[4:]
    return s


def _norm_url(u: object) -> str | None:
    # Application identity: www and bare host are the same product domain.
    # Keep the legacy case/query policy; do not rewrite other subdomains.
    if _is_blank(u):
        return None
    u = str(u).strip().split("?")[0].split("#")[0].rstrip("/").lower()
    try:
        parts = urlsplit(u)
    except ValueError:
        return None
    if parts.netloc.startswith("www."):
        u = parts._replace(netloc=parts.netloc[4:]).geturl()
    return u or None


def _resolve_norm_url(value: object, base_url: str | None = None) -> str | None:
    if _is_blank(value):
        return None
    return _norm_url(urljoin(base_url or "", str(value).strip()))


def _name_match(a: str | None, b: str | None, threshold: float = 0.85) -> bool:
    if not a or not b:
        return False
    return SequenceMatcher(None, a.lower().strip(), b.lower().strip()).ratio() >= threshold


def _price_match(p1: object, p2: object, tol: float = 0.01) -> bool:
    if p1 is None or p2 is None or p1 == "" or p2 == "":
        return False
    try:
        a, b = float(p1), float(p2)
    except (ValueError, TypeError):
        return False
    if b == 0:
        return abs(a) < 1e-9
    return abs(a - b) / abs(b) <= tol


def parse_price(value: object) -> float | None:
    if value is None:
        return None
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        try:
            v = float(value)
            return v if math.isfinite(v) and v >= 0 else None
        except (TypeError, ValueError):
            return None
    text = _clean_text(value)
    if not text:
        return None
    text = re.sub(r"(?i)(?:₽|руб(?:\.|лей|ля)?|р\.)", "", text)
    text = text.replace("\u00a0", "").replace("\u202f", "").replace("\u2009", "").replace(" ", "")
    m = re.search(r"-?\d[\d.,]*", text)
    if not m:
        return None
    raw = m.group(0)
    if "," in raw and "." in raw:
        if raw.rfind(",") > raw.rfind("."):
            raw = raw.replace(".", "").replace(",", ".")
        else:
            raw = raw.replace(",", "")
    elif "," in raw:
        parts = raw.split(",")
        raw = "".join(parts) if len(parts[-1]) == 3 else ".".join(parts)
    elif raw.count(".") > 1:
        raw = raw.replace(".", "")
    try:
        v = float(raw)
    except ValueError:
        return None
    return v if math.isfinite(v) and v >= 0 else None


def _parse_binary_annotation(x: object) -> bool | None:
    if _is_blank(x):
        return None
    if isinstance(x, bool):
        return x
    try:
        return bool(int(float(x)))
    except (TypeError, ValueError):
        s = str(x).strip().casefold()
        if s in {"1", "true", "yes", "y", "да"}:
            return True
        if s in {"0", "false", "no", "n", "нет"}:
            return False
    return None


def normalize_html_key(value: object) -> str:
    if _is_blank(value):
        return ""
    value = str(value).replace("\\", "/").rstrip("/")
    value = value.removeprefix("./").casefold()
    return value.removeprefix("html/")


def parse_benchmark_filename(path_or_name: object) -> dict[str, str | None]:
    name = str(path_or_name).replace("\\", "/").split("/")[-1]
    stem = Path(name).stem
    m = BENCHMARK_FILENAME_RE.match(stem)
    if not m:
        return {"source": None, "run_id": None}
    return {
        "source": normalize_source(m.group("source")),
        "run_id": m.group("run").lower(),
    }


def host_from_url(url: object) -> str | None:
    if _is_blank(url):
        return None
    try:
        host = urlsplit(str(url)).hostname
    except Exception:
        return None
    return normalize_source(host)


def detect_derived_collection_summary(html_text: str) -> bool:
    head = html_text[:200_000].casefold()
    return (
        "collection artifact" in head
        and "<th>product</th>" in head
        and "<th>price</th>" in head
        and "<th>url</th>" in head
    )


def _coerce_cell(raw: str):
    if raw == "":
        return ""
    try:
        if re.fullmatch(r"-?\d+", raw):
            return int(raw)
        if re.fullmatch(r"-?(?:\d+\.\d*|\d*\.\d+)(?:[eE][+-]?\d+)?", raw):
            return float(raw)
    except ValueError:
        pass
    return raw


def _column_index(cell_ref: str) -> int:
    letters = re.match(r"[A-Z]+", cell_ref or "")
    if not letters:
        return 0
    n = 0
    for ch in letters.group(0):
        n = n * 26 + (ord(ch) - ord("A") + 1)
    return n - 1


def _read_xlsx_sheet_without_openpyxl(path: Path, wanted_sheet: str) -> pd.DataFrame:
    ns = {"x": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    rel_ns = {"r": "http://schemas.openxmlformats.org/package/2006/relationships"}
    office_rel = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"

    with zipfile.ZipFile(path) as z:
        workbook = ET.fromstring(z.read("xl/workbook.xml"))
        rels = ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))
        rel_map = {rel.attrib["Id"]: rel.attrib["Target"] for rel in rels.findall("r:Relationship", rel_ns)}
        sheet_path = None
        available = []
        for sheet in workbook.findall("x:sheets/x:sheet", ns):
            name = sheet.attrib.get("name", "")
            available.append(name)
            if name.casefold() == wanted_sheet.casefold():
                rid = sheet.attrib.get(f"{{{office_rel}}}id")
                target = rel_map.get(rid or "")
                if target:
                    target = target.lstrip("/")
                    sheet_path = target if target.startswith("xl/") else "xl/" + target
                break
        if sheet_path is None:
            raise ValueError(f"Sheet {wanted_sheet!r} not found; available={available}")

        shared = []
        if "xl/sharedStrings.xml" in z.namelist():
            root = ET.fromstring(z.read("xl/sharedStrings.xml"))
            for si in root.findall("x:si", ns):
                shared.append("".join((n.text or "") for n in si.findall(".//x:t", ns)))

        root = ET.fromstring(z.read(sheet_path))
        rows = []
        for row in root.findall(".//x:sheetData/x:row", ns):
            cells = {}
            max_idx = -1
            for cell in row.findall("x:c", ns):
                idx = _column_index(cell.attrib.get("r", ""))
                max_idx = max(max_idx, idx)
                ctype = cell.attrib.get("t")
                inline = cell.findall("x:is//x:t", ns)
                vnode = cell.find("x:v", ns)
                if inline:
                    value = "".join((n.text or "") for n in inline)
                else:
                    raw = vnode.text if vnode is not None and vnode.text is not None else ""
                    if ctype == "s":
                        try:
                            value = shared[int(raw)]
                        except (ValueError, IndexError):
                            value = raw
                    elif ctype == "b":
                        value = raw == "1"
                    else:
                        value = _coerce_cell(raw)
                cells[idx] = value
            if max_idx >= 0:
                rows.append([cells.get(i, "") for i in range(max_idx + 1)])

    if not rows:
        return pd.DataFrame()
    width = max(len(r) for r in rows)
    rows = [r + [""] * (width - len(r)) for r in rows]
    headers = [str(x).strip() for x in rows[0]]
    seen = {}
    clean_headers = []
    for i, h in enumerate(headers):
        h = h or f"column_{i+1}"
        count = seen.get(h, 0)
        seen[h] = count + 1
        clean_headers.append(h if count == 0 else f"{h}_{count+1}")
    return pd.DataFrame(rows[1:], columns=clean_headers)


def _read_reference_sheet(path: Path, sheet_name: str = "GroundTruth") -> pd.DataFrame:
    if path.suffix.casefold() == ".csv":
        return pd.read_csv(path)
    if path.suffix.casefold() != ".xlsx":
        raise ValueError(f"Unsupported reference format: {path.suffix}")
    try:
        with pd.ExcelFile(path) as xls:
            actual = next((s for s in xls.sheet_names if s.casefold() == sheet_name.casefold()), None)
            if actual is None:
                raise ValueError(f"Sheet {sheet_name!r} not found; available={xls.sheet_names}")
            return pd.read_excel(xls, sheet_name=actual)
    except ImportError:
        return _read_xlsx_sheet_without_openpyxl(path, sheet_name)


def discover_dataset_reference(dataset_root: Path) -> Path | None:
    preferred = (
        "dind.xlsx",
        "groundtruth.xlsx",
        "reference.xlsx",
        "synthetic_ground_truth.xlsx",
        "industrial_ground_truth.xlsx",
        "synthetic_ground_truth.csv",
        "industrial_ground_truth.csv",
        "groundtruth.csv",
    )
    for name in preferred:
        p = dataset_root / name
        if p.exists():
            return p
    candidates = []
    for pattern in ("*.xlsx", "*.csv"):
        for p in dataset_root.glob(pattern):
            if p.name.startswith("~$"):
                continue
            if re.search(r"ground|truth|dind|reference|annotation", p.name, re.I):
                candidates.append(p)
    return sorted(candidates, key=lambda p: (p.suffix != ".xlsx", p.name.casefold()))[0] if candidates else None


def discover_parser_contract_file(root: Path) -> Path | None:
    for name in ("parse_logs_to_excel_v2.py", "parse_logs_to_excel_v3.py"):
        p = root / name
        if p.exists():
            return p
    for p in root.rglob("parse_logs_to_excel_v2.py"):
        if ".venv" not in p.parts and "node_modules" not in p.parts:
            return p
    return None


def validate_parser_contract(path: Path | None) -> dict:
    if path is None or not Path(path).exists():
        return {"available": False, "path": str(path or ""), "ok": None, "mismatches": []}
    spec = importlib.util.spec_from_file_location("_baseline_reference_contract", str(path))
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)

    mismatches = []
    source_cases = [None, "bigam.ru", "WWW.DNS-SHOP.RU", "https://www.mvideo.ru/search"]
    url_cases = [None, "https://www.bigam.ru/product/123/", "https://demo.ru/p/1/?utm=x#frag"]
    name_cases = [("Дрель Alpha 500", "дрель alpha 500"), ("A-B", "A B"), (None, "x")]
    price_cases = [(10050, 10000), (10101, 10000), (0, 0), (None, 1)]

    for x in source_cases:
        if normalize_source(x) != module.normalize_source(x):
            mismatches.append(("normalize_source", x, normalize_source(x), module.normalize_source(x)))
    for x in url_cases:
        # URL identity deliberately extends the parser contract with www folding.
        if _norm_url(x) != _norm_url(module._norm_url(x)):
            mismatches.append(("_norm_url", x, _norm_url(x), module._norm_url(x)))
    for a, b in name_cases:
        if _name_match(a, b, NAME_THRESHOLD) != module._name_match(a, b, NAME_THRESHOLD):
            mismatches.append(("_name_match", (a, b), _name_match(a, b), module._name_match(a, b)))
    for a, b in price_cases:
        if _price_match(a, b, PRICE_REL_TOL) != module._price_match(a, b, PRICE_REL_TOL):
            mismatches.append(("_price_match", (a, b), _price_match(a, b), module._price_match(a, b)))
    return {"available": True, "path": str(path), "ok": not mismatches,
            "url_policy": "legacy rules plus www equivalence and trailing-slash normalization",
            "mismatches": mismatches}


if PARSER_CONTRACT_FILE is None:
    PARSER_CONTRACT_FILE = discover_parser_contract_file(BENCHMARK_ROOT)

contract_check = validate_parser_contract(PARSER_CONTRACT_FILE)
print("Parser contract:", json.dumps(contract_check, ensure_ascii=False, indent=2))


Parser contract: {
  "available": true,
  "path": "D:\\Pet_Codes\\science\\article-price-tracker\\scripts\\parse_logs_to_excel_v2.py",
  "ok": true,
  "url_policy": "legacy rules plus www equivalence and trailing-slash normalization",
  "mismatches": []
}


## 3. Dataset manifests and GroundTruth

When the reference workbook has a Pages sheet, it defines the corpus. Extra
HTML files are ignored; missing files remain in the manifest with missing_html.
Without Pages, HTML is discovered recursively. Relative paths remain unique.

GroundTruth accepts html_file, page_id or synthetic mutation_id. Original
entity_id/gt_id and additional classification columns are retained. Missing
identifiers in legacy synthetic references receive deterministic fallback IDs.

ANNOTATION_POLICY='labeled' includes the current user-reviewed in_progress
annotations. Use 'done' to require done/reviewed/complete/completed statuses.
Explicit pending/todo/new/unannotated rows are never scored. A partially
annotated page is excluded as a whole; skip/excluded excludes its whole page.

Industrial price 0 means absent. A blank price is absent only with an explicit
price state/confirmation or the previously recorded USER_REVIEW: no price note.
Otherwise its state is unannotated. Synthetic zero remains a numeric price.


In [5]:
def _canonical_columns(raw: pd.DataFrame) -> dict[str, str]:
    return {re.sub(r"[^a-z0-9]+", "", str(c).casefold()): c for c in raw.columns}


def _pick_column(raw: pd.DataFrame, aliases: list[str]) -> pd.Series | None:
    canonical = _canonical_columns(raw)
    for alias in aliases:
        key = re.sub(r"[^a-z0-9]+", "", alias.casefold())
        if key in canonical:
            return raw[canonical[key]]
    return None


def _identity(prefix: str, *parts: object) -> str:
    value = "\x1f".join("" if _is_blank(p) else str(p) for p in parts)
    return prefix + hashlib.sha256(value.encode("utf-8")).hexdigest()[:20]


def read_groundtruth(path: Path, sheet_name: str = "GroundTruth", *,
                     zero_price_is_missing: bool = False,
                     annotation_policy: str = ANNOTATION_POLICY) -> pd.DataFrame:
    if annotation_policy not in {"labeled", "done"}:
        raise ValueError(f"Unknown annotation policy: {annotation_policy}")
    raw = _read_reference_sheet(path, sheet_name)

    def col(aliases, default=None):
        result = _pick_column(raw, aliases)
        return result if result is not None else pd.Series([default] * len(raw), index=raw.index, dtype=object)

    # Keep classification/cluster columns and the original annotations for joins.
    gt = raw.copy()
    for name, aliases in {
        "gt_id": ["gt_id"], "page_id": ["page_id", "mutation_id", "base_page_id"],
        "html_file": ["html_file", "benchmark_html_file", "file", "filename"],
        "entity_id": ["entity_id"], "source": ["source", "source_name", "supplier", "site"],
        "run_id": ["run_id", "run", "collection_run_id"], "query": ["query"],
        "annotation_status": ["annotation_status", "status"], "is_product": ["is_product"],
        "gt_name": ["name_gt", "gt_name", "ground_truth_name", "product_name_gt", "product_name"],
        "gt_price": ["price_gt", "gt_price", "ground_truth_price", "product_price"],
        "product_url": ["product_url_gt", "product_url", "canonical_url", "url"],
        "notes": ["notes"],
    }.items():
        gt[name] = col(aliases)
    missing_html = gt.html_file.map(_is_blank)
    has_page = ~gt.page_id.map(_is_blank)
    gt.loc[missing_html & has_page, "html_file"] = gt.loc[missing_html & has_page, "page_id"].astype(str) + ".html"
    gt["html_key"] = gt.html_file.map(normalize_html_key)
    parsed = gt.html_file.map(parse_benchmark_filename)
    gt["source"] = [normalize_source(src) or p["source"] for src, p in zip(gt.source, parsed)]
    gt["run_id"] = [("" if _is_blank(r) else str(r).strip()) or p["run_id"] or "" for r, p in zip(gt.run_id, parsed)]
    gt["gt_name"] = gt.gt_name.map(lambda x: "" if _is_blank(x) else str(x).strip())
    price = pd.to_numeric(gt.gt_price, errors="coerce")
    price = price.where(price.map(lambda p: pd.notna(p) and math.isfinite(p) and p >= 0))
    zero_absent = price.eq(0) & zero_price_is_missing
    gt["gt_price"] = price.mask(zero_absent)
    gt["product_url"] = gt.product_url.map(_norm_url)
    gt["is_product"] = gt.is_product.map(_parse_binary_annotation)
    gt["annotation_status"] = gt.annotation_status.map(lambda x: "" if _is_blank(x) else str(x).strip().casefold())
    gt["_excluded"] = gt.annotation_status.isin({"skip", "excluded"})
    reviewed = {"done", "reviewed", "complete", "completed"}
    pending = {"pending", "todo", "unannotated", "new"}

    def annotated(row):
        status = row.annotation_status
        if row._excluded or status in pending:
            return False
        if annotation_policy == "done":
            return status in reviewed
        return bool(status in reviewed | {"in_progress", "annotated"}
                    or (not status and (row.is_product is not None or row.gt_name
                                        or pd.notna(row.gt_price) or row.product_url)))

    gt["_annotated"] = pd.Series([annotated(r) for _, r in gt.iterrows()], index=gt.index, dtype=bool)
    gt["_positive"] = pd.Series([
        bool(r._annotated and (r.is_product is True or (r.is_product is not False and
             (r.gt_name or pd.notna(r.gt_price) or r.product_url))))
        for _, r in gt.iterrows()
    ], index=gt.index, dtype=bool)

    explicit_state = col(["gt_price_state", "price_state_gt", "price_state"])
    confirmed = col(["price_absence_confirmed"]).map(_parse_binary_annotation).eq(True)
    # These notes were written by the prior, explicitly user-confirmed GT update.
    confirmed |= gt.notes.fillna("").astype(str).str.contains("USER_REVIEW: no price", regex=False)
    states = []
    for i in gt.index:
        state = "" if _is_blank(explicit_state.loc[i]) else str(explicit_state.loc[i]).strip().casefold()
        inferred = "present" if pd.notna(gt.at[i, "gt_price"]) else (
            "absent" if zero_absent.loc[i] or confirmed.loc[i] else "unannotated")
        state = state or inferred
        if state not in {"present", "absent", "unannotated"}:
            raise ValueError(f"Unknown price state at GT row {i}: {state}")
        if (state == "present") != pd.notna(gt.at[i, "gt_price"]):
            raise ValueError(f"Price/state conflict at GT row {i}: {state}")
        states.append(state)
    gt["gt_price_state"] = pd.Series(states, index=gt.index, dtype=object)
    gt["gt_id"] = gt.gt_id.astype(object)
    gt["entity_id"] = gt.entity_id.astype(object)
    gt["_eval_product_key"] = gt.product_url.astype(object)
    for i in gt.index:
        if gt.at[i, "_positive"] and _is_blank(gt.at[i, "entity_id"]):
            gt.at[i, "entity_id"] = _identity("entity-", gt.at[i, "html_key"], gt.at[i, "product_url"] or f"row-{i}")
        if _is_blank(gt.at[i, "gt_id"]):
            entity = gt.at[i, "entity_id"]
            gt.at[i, "gt_id"] = _identity("gt-", gt.at[i, "html_key"], entity if not _is_blank(entity) else f"row-{i}")
        if gt.at[i, "_positive"] and _is_blank(gt.at[i, "product_url"]):
            gt.at[i, "_eval_product_key"] = "__gt_missing_url__:" + str(gt.at[i, "entity_id"])
    gt.attrs.update(source_path=str(path), reference_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
                    annotation_policy=annotation_policy, zero_price_is_missing=zero_price_is_missing)
    return gt.reset_index(drop=True)


def read_pages_metadata(reference_file: Path | None) -> dict[str, dict] | None:
    if reference_file is None:
        return None
    if reference_file.suffix.casefold() == ".csv":
        # Generated corpora have an authoritative companion *_pages.csv.
        stem = reference_file.stem
        if not stem.endswith("_ground_truth"):
            return None
        pages_file = reference_file.with_name(stem.removesuffix("_ground_truth") + "_pages.csv")
        if not pages_file.is_file():
            return None
        pages = pd.read_csv(pages_file)
    elif reference_file.suffix.casefold() == ".xlsx":
        try:
            pages = _read_reference_sheet(reference_file, "Pages")
        except ValueError as exc:
            if "not found; available=" in str(exc):
                return None
            raise
    else:
        return None
    file_col = _pick_column(pages, ["benchmark_html_file", "html_file"])
    if file_col is None:
        raise ValueError("Pages sheet is missing benchmark_html_file/html_file")
    mapping = {}
    for i, row in pages.iterrows():
        key = normalize_html_key(file_col.loc[i])
        if not key or key in mapping:
            raise ValueError(f"Empty or duplicate Pages HTML key: {key!r}")
        mapping[key] = row.to_dict()
    return mapping


def build_html_manifest(dataset_name: str, html_dir: Path, reference_file: Path | None = None) -> pd.DataFrame:
    columns = ["dataset", "page_id", "html_file", "html_key", "run_id", "source", "html_path",
               "page_url", "source_url", "query", "location", "route", "artifact_kind", "skip_reason",
               "content_sha256", "expected_sha256"]
    metadata = read_pages_metadata(reference_file)
    if metadata is not None:
        paths = [(html_dir / key, key, meta) for key, meta in metadata.items()]
    else:
        paths = [(p, normalize_html_key(p.relative_to(html_dir)), {})
                 for p in sorted([*html_dir.rglob("*.html"), *html_dir.rglob("*.htm")])]
    rows = []
    for html_path, html_key, meta in paths:
        if not html_path.resolve().is_relative_to(html_dir.resolve()):
            raise ValueError(f"HTML path outside dataset: {html_path}")
        parsed = parse_benchmark_filename(html_path.name)
        page_url = _first_nonempty(meta.get("page_url"), meta.get("source_url"))
        reason = ""
        if _parse_binary_annotation(meta.get("include")) is False:
            reason = _first_nonempty(meta.get("exclusion_reason"), "excluded_in_pages")
        exists = html_path.is_file()
        data = html_path.read_bytes() if exists else b""
        text = data.decode("utf-8", errors="replace")
        if not exists:
            reason = "missing_html"
        if not page_url and exists:
            page_url = document_canonical_url(BeautifulSoup(text, "lxml"), "")
        derived = detect_derived_collection_summary(text)
        if not reason and derived and EXCLUDE_DERIVED_ARTIFACT_HTML:
            reason = "derived_from_pipeline_snapshots"
        rows.append({
            "dataset": dataset_name, "page_id": _first_nonempty(meta.get("page_id"), html_key.removesuffix(html_path.suffix)),
            "html_file": "html/" + html_key, "html_key": html_key, "html_path": str(html_path),
            "run_id": _first_nonempty(meta.get("run_id"), parsed["run_id"]),
            "source": normalize_source(meta.get("source")) or parsed["source"] or host_from_url(page_url) or "unknown",
            "page_url": page_url, "source_url": _first_nonempty(meta.get("source_url"), page_url),
            "query": _first_nonempty(meta.get("query")), "location": _first_nonempty(meta.get("location")),
            "route": _first_nonempty(meta.get("route")),
            "artifact_kind": "derived_collection_summary" if derived else "raw_html", "skip_reason": reason,
            "content_sha256": hashlib.sha256(data).hexdigest() if exists else "",
            "expected_sha256": _first_nonempty(meta.get("sha256")),
        })
        rows[-1].update({"page_meta__" + str(k): v for k, v in meta.items()})
    counters = {}
    for row in rows:
        if not row["run_id"]:
            source = row["source"]
            counters[source] = counters.get(source, 0) + 1
            row["run_id"] = f"run-none{counters[source]}"
    extra_columns = sorted({k for r in rows for k in r if k not in columns})
    manifest = pd.DataFrame(rows, columns=columns + extra_columns)
    if manifest.html_key.duplicated().any() or manifest.page_id.duplicated().any():
        raise ValueError("Manifest requires unique html_key and page_id")
    manifest.attrs["selection"] = "Pages" if metadata is not None else "html_directory"
    return manifest


def align_groundtruth_to_manifest(gt: pd.DataFrame, manifest: pd.DataFrame) -> pd.DataFrame:
    out = gt.copy()
    if gt.empty or manifest.empty:
        return out
    if manifest.html_key.duplicated().any() or manifest.page_id.duplicated().any():
        raise ValueError("Duplicate HTML/page keys in manifest")
    meta = manifest.set_index("html_key")
    for i in out.index:
        key = out.at[i, "html_key"]
        if key not in meta.index:
            continue
        for col in ["html_file", "page_id", "run_id", "source", "query"]:
            if col in meta and _is_blank(out.at[i, col]):
                out.at[i, col] = meta.at[key, col]
        if str(out.at[i, "page_id"]) != str(meta.at[key, "page_id"]):
            raise ValueError(f"GT/manifest page_id conflict: {key}")
    return out


def evaluation_scope(gt: pd.DataFrame, manifest: pd.DataFrame):
    """Only available, fully annotated pages; skip excludes the whole page."""
    if manifest.html_key.duplicated().any():
        raise ValueError("Duplicate HTML keys in manifest")
    excluded = set(gt.loc[gt["_excluded"], "html_key"])
    active = manifest[manifest.skip_reason.fillna("").eq("") & ~manifest.html_key.isin(excluded)].copy()
    local_gt = gt[gt.html_key.isin(active.html_key)].copy()
    complete = local_gt.groupby("html_key")["_annotated"].all()
    keys = set(complete.index[complete])
    return local_gt[local_gt.html_key.isin(keys)].copy(), active[active.html_key.isin(keys)].copy()


def groundtruth_audit(gt: pd.DataFrame, manifest: pd.DataFrame) -> dict:
    scoped_gt, scored = evaluation_scope(gt, manifest)
    positive = scoped_gt[scoped_gt._positive]
    excluded = set(gt.loc[gt._excluded, "html_key"])
    active_keys = set(manifest.loc[manifest.skip_reason.fillna("").eq(""), "html_key"]) - excluded
    duplicate_keys = int(positive[positive.product_url.notna()].duplicated(["html_key", "product_url"], keep=False).sum())
    missing_name = sum(_is_blank(v) for v in positive.gt_name)
    missing_url = int(positive.product_url.isna().sum())
    unknown_price = int(positive.gt_price_state.eq("unannotated").sum())
    missing_files = manifest.loc[manifest.skip_reason.eq("missing_html"), "html_file"].tolist()
    hash_mismatch = []
    if {"content_sha256", "expected_sha256"} <= set(manifest.columns):
        hash_mismatch = manifest.loc[
            manifest.content_sha256.fillna("").ne("") & manifest.expected_sha256.fillna("").ne("")
            & manifest.content_sha256.ne(manifest.expected_sha256), "html_file"].tolist()
    ready = bool(active_keys and active_keys <= set(scored.html_key) and not
                 (missing_name or missing_url or unknown_price or duplicate_keys or hash_mismatch))
    outside = set(gt.loc[gt._annotated, "html_key"]) - set(manifest.html_key)
    return {
        "reference_rows": len(gt), "manifest_pages": len(manifest), "available_manifest_pages": len(active_keys),
        "annotated_pages": len(scored), "unannotated_manifest_pages": len(active_keys - set(scored.html_key)),
        "reference_pages_not_in_html_dir": len(outside), "positive_entities": len(positive),
        "negative_or_empty_rows": int((scoped_gt._annotated & ~scoped_gt._positive).sum()),
        "positive_missing_name_gt": missing_name, "positive_missing_price_gt": int(positive.gt_price.isna().sum()),
        "positive_confirmed_absent_price_gt": int(positive.gt_price_state.eq("absent").sum()),
        "positive_unannotated_price_gt": unknown_price, "positive_missing_product_url_gt": missing_url,
        "duplicate_positive_page_url_keys": duplicate_keys, "missing_html_files": missing_files,
        "html_hash_mismatches": hash_mismatch, "excluded_gt_pages": len(excluded),
        "annotation_policy": gt.attrs.get("annotation_policy", ANNOTATION_POLICY),
        "annotation_status_counts": gt.annotation_status.value_counts().to_dict(),
        "ready_for_available_pages": ready,
        "paper_ready_closed_world": ready and not missing_files and not outside,
    }


## 4. Embedded original parser and historical controls

The following cell embeds seven unmodified original TypeScript modules, their SHA256 checksums,
and a standalone JavaScript bundle with the exact dependency versions used by the reference run.
`prepare_original_runtime()` verifies and materializes these files in `.benchmark_runtime/`.
This folder is disposable and is recreated from notebook contents. No original repository is read.

The original parser includes its DOM/CSS, JSON-LD, embedded-state JSON, supplier profiles,
candidate selection, field cleaning and query relevance logic. The request limit remains 500.
The Python B1/B2/B3 functions are historical controls, activated only if explicitly added to VARIANTS.


In [6]:
# Frozen original PriceTracker parser. Sources below are byte-for-byte copies.
ORIGINAL_SOURCE_FILES = {'src/extractor.ts': 'import * as cheerio from "cheerio";\nimport type { AnyNode } from "domhandler";\n\nimport {\n  detectBlockedPage,\n  detectEmptySearchResults,\n} from "./blockDetection.js";\nimport type {\n  ExtractProductsRequest,\n  ExtractProductsResult,\n  ProductRow,\n  SearchProduct,\n  SupplierProfile,\n} from "./contracts.js";\nimport { getSupplierProfile } from "./profiles.js";\nimport {\n  absoluteUrl,\n  cleanText,\n  coercePrice,\n  firstAttr,\n  firstText,\n  looksLikeProductHref,\n  normalizeHostname,\n  ownOrDescendantText,\n  splitSelectorList,\n  cleanProductName,\n  normalizeCssSelectorForCheerio,\n} from "./utils.js";\nimport { discoverProductCandidates } from "./candidates.js";\nimport { filterRowsByQueryRelevance } from "./queryRelevance.js";\n\n// Селекторы расположены от специфичных и устойчивых к наиболее широким:\n// раннее совпадение уменьшает риск принять общий элемент списка за товар.\nconst genericSelectors = [\n  "a[mvid-product-card]",\n  "tr.with-hover",\n  ".itemlist tr",\n  "[data-id=\'product\']",\n  "[data-meta-product-id]",\n  "[itemtype*=\'Product\']",\n  ".search-results__item",\n  ".js-product",\n  ".product-item-container",\n  ".product-item-big-card",\n  ".product-list-item",\n\n  // Repeated generic ecommerce cards.\n  ".catalog__list-item.snippet",\n  ".catalog-product",\n  "[class*=\'catalog-product\' i]",\n  ".catalog__list-item.snippet",\n  ".promo-slider__item.snippet",\n  ".promo-slider__item.snippet",\n  ".snippet",\n  ".snippet__content",\n  "[class~=\'snippet\']",\n\n  ".product-card-list",\n  ".product-card",\n  "[class*=\'product-card__\']",\n\n  ".catalog-product",\n  "[class*=\'catalog-product\' i]",\n  ".catalog-item",\n  ".catalog_item",\n  ".item_block",\n  ".product-item",\n\n  // Broad selectors last.\n  "article",\n  "li",\n\n  "a[data-qa=\'product-name\']",\n  "[data-qa=\'product-price-current\']",\n  "a[data-qa=\'product-photo-click\']",\n  "[data-qa=\'product-add-to-cart-button\']",\n];\n\n/**\n * Извлекает товарные строки по явным селекторам полей.\n *\n * Селекторы полей берутся по приоритету: из запроса, из конфигурации source,\n * затем из профиля поставщика. Контейнеры карточек выбираются в том же порядке:\n * request.itemSelector, source.itemSelector, profile.itemSelector, затем body.\n * Это позволяет профильным пайплайнам извлекать товары даже тогда, когда API\n * не передал fieldSelectors вместе с source.\n * Из каждой карточки извлекаются обязательные название, цена и ссылка на товар.\n * Дополнительно, если селекторы доступны, подтягиваются изображение, доставка\n * и признак наличия. Строки без имени, цены или ссылки не возвращаются.\n *\n * @param $ Cheerio-документ уже загруженной HTML-страницы.\n * @param request URL страницы, source, явные селекторы и контекст извлечения.\n * @param profile Профиль поставщика с резервными itemSelector и fieldSelectors.\n * @returns Нормализованные товарные строки, найденные по селекторам.\n */\nexport function extractProductsFromHtml(\n  request: ExtractProductsRequest,\n): ExtractProductsResult {\n  const limit = Math.max(1, Math.min(500, request.limit ?? 48));\n  const sourceUrl = request.source?.url ?? request.pageUrl;\n  const profile = getSupplierProfile(request.pageUrl, sourceUrl);\n  const route = profile\n    ? `pipeline:${profile.domain}`\n    : `pipeline:generic:${normalizeHostname(sourceUrl) || "unknown"}`;\n  const debug: string[] = [`route=${route}`];\n  const blockedMarkers = detectBlockedPage(request.html, request.pageUrl);\n  if (blockedMarkers.length > 0) {\n    return buildResult({\n      status: "blocked",\n      route,\n      pageUrl: request.pageUrl,\n      rows: [],\n      blockedMarkers,\n      debug: [...debug, `blocked=${blockedMarkers.join(",")}`],\n    });\n  }\n\n  const $ = cheerio.load(request.html);\n\n  const selectorRows = extractRows($, request, profile, limit);\n\n  const shouldDiscoverCandidates =\n    selectorRows.length === 0 || request.includeCandidates === true;\n\n  debug.push(\n    shouldDiscoverCandidates\n      ? selectorRows.length === 0\n        ? "candidate_discovery=enabled_no_selector_rows"\n        : "candidate_discovery=enabled_for_onboarding_learning"\n      : "candidate_discovery=skipped_selector_rows_found",\n  );\n\n  const candidates = shouldDiscoverCandidates\n    ? discoverProductCandidates({\n        $,\n        pageUrl: request.pageUrl,\n        profile,\n        itemSelector: request.itemSelector ?? request.source?.itemSelector,\n        limit: Math.max(limit, 24),\n      })\n    : [];\n\n  // Важно: candidates используются как fallback rows только если быстрый selector path пустой.\n  // Иначе onboarding может использовать candidates для обучения selectors,\n  // но collection не получит ложные rows из candidate discovery.\n  const candidateRows =\n    selectorRows.length === 0\n      ? candidates\n          .filter((candidate) => candidate.missingFields.length === 0)\n          .map((candidate) => candidate.fields as ProductRow)\n      : [];\n\n  const extractedRows = uniqueRows([...selectorRows, ...candidateRows]).slice(\n    0,\n    limit,\n  );\n\n  const relevance = filterRowsByQueryRelevance(extractedRows, request.query);\n\n  const rows = relevance.rows;\n  const emptyReason =\n    rows.length === 0 ? detectEmptySearchResults(request.html) : undefined;\n  const products = buildProductsFromRows(rows);\n  const rejectedRows = countRejectedRows(rows);\n\n  const rejectReasons: Record<string, number> = {};\n  for (const candidate of candidates) {\n    if (!candidate.rejectReason) continue;\n    rejectReasons[candidate.rejectReason] =\n      (rejectReasons[candidate.rejectReason] ?? 0) + 1;\n  }\n\n  return {\n    status: products.length > 0 ? "ok" : "empty",\n    route,\n    pageUrl: request.pageUrl,\n    queryTags: relevance.queryTags,\n    rows,\n    products,\n    diagnostics: {\n      blockedMarkers,\n      emptyReason,\n      candidateCount: candidates.length,\n      rowCount: products.length,\n      rejectedRows,\n      queryRelevance: relevance.diagnostics,\n      debug: [\n        ...debug,\n        `selector_rows=${selectorRows.length}`,\n        `candidates=${candidates.length}`,\n        `rows_before_query_relevance=${extractedRows.length}`,\n        `rows_after_query_relevance=${rows.length}`,\n        `query_tags=${relevance.queryTags.map((tag) => tag.value).join("|")}`,\n        `query_relevance_rejected=${relevance.diagnostics.rejectedCount}`,\n        `products=${products.length}`,\n        ...(emptyReason ? [`empty_reason=${emptyReason}`] : []),\n      ],\n      candidates: request.includeCandidates\n        ? {\n            candidateCount: candidates.length,\n            validRowCount: rows.length,\n            rejectedCandidateCount: candidates.filter(\n              (candidate) => candidate.rejectReason,\n            ).length,\n            rejectReasons,\n            topCandidates: candidates.slice(0, 12),\n          }\n        : undefined,\n    },\n  };\n}\n\n/**\n * Получает товарные строки быстрыми селекторными и JSON-маршрутами.\n *\n * @param $ Документ Cheerio.\n * @param request Параметры извлечения.\n * @param profile Профиль известного поставщика.\n * @param limit Максимальное число строк.\n * @returns Уникальные строки в пределах лимита.\n */\nfunction extractRows(\n  $: cheerio.CheerioAPI,\n  request: ExtractProductsRequest,\n  profile: SupplierProfile | undefined,\n  limit: number,\n): ProductRow[] {\n  let explicitRows: ProductRow[] = [];\n  try {\n    explicitRows = extractRowsWithSelectors($, request, profile);\n  } catch {\n    explicitRows = [];\n  }\n  if (explicitRows.length > 0) return uniqueRows(explicitRows).slice(0, limit);\n\n  const selectors = [\n    ...splitSelectorList(request.itemSelector),\n    ...splitSelectorList(request.source?.itemSelector),\n    ...splitSelectorList(profile?.itemSelector),\n    ...genericSelectors,\n  ];\n  const rows: ProductRow[] = [];\n  const seenNodes = new Set<AnyNode>();\n\n  for (const selector of selectors) {\n    let nodes: cheerio.Cheerio<AnyNode>;\n    try {\n      nodes = $(selector);\n    } catch {\n      continue;\n    }\n    nodes.each((_, element) => {\n      if (rows.length >= limit) return false;\n      if (seenNodes.has(element)) return;\n      seenNodes.add(element);\n      const node = $(element);\n      const row = rowFromNode($, node, request.pageUrl, profile);\n      if (row) rows.push(row);\n      return undefined;\n    });\n    if (\n      rows.length >= Math.min(4, limit) &&\n      selector === profile?.itemSelector\n    ) {\n      break;\n    }\n  }\n\n  if (rows.length === 0) {\n    rows.push(\n      ...extractRowsFromEmbeddedJson(request.html, request.pageUrl, profile),\n    );\n  }\n\n  return uniqueRows(rows).slice(0, limit);\n}\n\n/**\n * Извлекает строки строго по селекторам, сохранённым для источника.\n *\n * @param $ Документ Cheerio.\n * @param request Запрос с item- и field-селекторами.\n * @returns Строки, в которых одновременно найдены имя, цена и URL.\n */\nfunction extractRowsWithSelectors(\n  $: cheerio.CheerioAPI,\n  request: ExtractProductsRequest,\n  profile?: SupplierProfile,\n): ProductRow[] {\n  const fieldSelectors =\n    request.fieldSelectors ??\n    request.source?.fieldSelectors ??\n    profile?.fieldSelectors;\n  if (!fieldSelectors || Object.keys(fieldSelectors).length === 0) return [];\n  const itemSelectors = splitSelectorList(\n    request.itemSelector ??\n      request.source?.itemSelector ??\n      profile?.itemSelector ??\n      "body",\n  );\n  const roots = itemSelectors.length > 0 ? itemSelectors : ["body"];\n  const rows: ProductRow[] = [];\n\n  for (const selector of roots) {\n    let rootNodes: cheerio.Cheerio<AnyNode>;\n\n    try {\n      rootNodes = $(normalizeCssSelectorForCheerio(selector));\n    } catch {\n      continue;\n    }\n\n    rootNodes.each((_, element) => {\n      const root = $(element);\n      const fields: Record<string, string> = {};\n      for (const [fieldName, expression] of Object.entries(fieldSelectors)) {\n        const value = extractSelectorExpression(\n          $,\n          root,\n          expression,\n          request.pageUrl,\n        );\n        if (value) fields[fieldName] = value;\n      }\n      const price = coercePrice(fields.price ?? fields.old_price);\n      const name = cleanProductName(fields.name);\n      const productUrl = absoluteUrl(\n        request.pageUrl,\n        fields.product_url ?? fields.url,\n      );\n      if (name && price && productUrl) {\n        const row: ProductRow = { name, price, product_url: productUrl };\n\n        const imageUrl = absoluteUrl(\n          request.pageUrl,\n          fields.image_url?.split(/\\s+/, 1)[0],\n        );\n        if (imageUrl) row.image_url = imageUrl;\n\n        const deliveryTime = cleanText(fields.delivery_time);\n        if (deliveryTime) row.delivery_time = deliveryTime.slice(0, 180);\n\n        const availabilityText = cleanText(\n          fields.availability ?? fields.delivery_time,\n        ).toLowerCase();\n\n        if (\n          /нет в наличии|нет на складе|нет в продаже|недоступен/.test(\n            availabilityText,\n          )\n        ) {\n          row.in_stock = false;\n        } else if (\n          /есть в наличии|в наличии|доступны|доступно|купить|в корзину/.test(\n            availabilityText,\n          )\n        ) {\n          row.in_stock = true;\n        }\n\n        rows.push(row);\n      }\n    });\n  }\n  return rows;\n}\n\n/**\n * Выполняет выражение поля относительно одной карточки.\n *\n * Поддерживаются обычный текст, `::text`, `::attr(name)` и сокращение\n * `selector@attribute`; несколько альтернатив проверяются по порядку.\n *\n * @param $ Документ Cheerio.\n * @param root Корень карточки.\n * @param expression Выражение селектора.\n * @param pageUrl Базовый URL для ссылок и изображений.\n * @returns Первое непустое значение либо пустую строку.\n */\nfunction extractSelectorExpression(\n  $: cheerio.CheerioAPI,\n  root: cheerio.Cheerio<AnyNode>,\n  expression: string,\n  pageUrl: string,\n): string {\n  for (const rawPart of expression\n    .split(",")\n    .map((item) => item.trim())\n    .filter(Boolean)) {\n    const { selector, attr } = parseSelectorExpression(rawPart);\n    if (!selector) continue;\n\n    let target: cheerio.Cheerio<AnyNode>;\n    try {\n      target = root.find(selector).first();\n    } catch {\n      continue;\n    }\n\n    if (target.length === 0) continue;\n\n    const raw = attr ? target.attr(attr) : target.text();\n    const value = cleanText(raw);\n    if (!value) continue;\n\n    return attr === "href" || attr === "src"\n      ? absoluteUrl(pageUrl, value)\n      : value;\n  }\n\n  return "";\n}\n\n/**\n * Разбирает расширенное выражение селектора на CSS и имя атрибута.\n *\n * @param expression Настройка поля источника.\n * @returns Нормализованный CSS-селектор и пустой атрибут для текста либо имя\n * читаемого атрибута.\n */\nfunction parseSelectorExpression(expression: string): {\n  selector: string;\n  attr: string;\n} {\n  let value = expression.trim();\n\n  const attrPseudoMatch = value.match(/::attr\\(([^)]+)\\)\\s*$/i);\n  if (attrPseudoMatch) {\n    value = value.slice(0, attrPseudoMatch.index).trim();\n    return {\n      selector: normalizeCssSelectorForCheerio(value),\n      attr: attrPseudoMatch[1]?.trim() ?? "",\n    };\n  }\n\n  value = value.replace(/::text(?:\\(\\))?\\s*$/i, "").trim();\n\n  const atMatch = value.match(\n    /^(?<selector>.+?)@(?<attr>[A-Za-z_:][-A-Za-z0-9_:.]*)$/,\n  );\n  if (atMatch?.groups) {\n    return {\n      selector: normalizeCssSelectorForCheerio(atMatch.groups.selector.trim()),\n      attr: atMatch.groups.attr.trim(),\n    };\n  }\n\n  return {\n    selector: normalizeCssSelectorForCheerio(value),\n    attr: "",\n  };\n}\n\n/**\n * Отбрасывает строку, похожую на рейтинг, рекламу или элемент интерфейса.\n *\n * @param node Предполагаемая карточка.\n * @param name Извлечённое название.\n * @param price Извлечённая цена.\n * @returns `true`, если сочетание классов и текста не похоже на товар.\n */\nfunction isSuspiciousNonProductRow(\n  node: cheerio.Cheerio<AnyNode>,\n  name: string,\n  price: number,\n): boolean {\n  const className = cleanText(node.attr("class")).toLowerCase();\n  const zoneName = cleanText(node.attr("data-zone-name")).toLowerCase();\n  const normalizedName = cleanText(name).toLowerCase();\n  const text = ownOrDescendantText(node).toLowerCase();\n\n  const isRealCatalogRoot =\n    node.hasClass("catalog-product") ||\n    node.hasClass("snippet") ||\n    node.hasClass("product-card") ||\n    zoneName === "productsnippet" ||\n    Boolean(node.attr("data-id")) ||\n    Boolean(node.attr("data-product-id")) ||\n    Boolean(node.attr("data-offer-id"));\n\n  if (\n    /help-choosing|help|banner|promo|sidebar|aside|filter|pagination|breadcrumb/.test(\n      className,\n    )\n  ) {\n    return true;\n  }\n\n  if (\n    /затрудняетесь в выборе|помогут вам определиться|материалы, которые помогут/.test(\n      text,\n    )\n  ) {\n    return true;\n  }\n\n  if (\n    !isRealCatalogRoot &&\n    /rating|comment|review|opinion|question|stat|compare|wishlist|pagination|avail/.test(\n      className,\n    )\n  ) {\n    return true;\n  }\n\n  if (\n    /^\\d+(?:[.,]\\d+)?(?:\\s*\\|\\s*[\\d.,kк]+\\s*отзывов?)?$/.test(normalizedName)\n  ) {\n    return true;\n  }\n\n  if (\n    price < 50 &&\n    /отзыв|вопрос|коммент|рейтинг|надежность|сравнить|выбор/.test(\n      normalizedName,\n    )\n  ) {\n    return true;\n  }\n\n  return false;\n}\n\n/**\n * Строит товарную строку из одного найденного DOM-узла.\n *\n * @param $ Документ Cheerio.\n * @param node Исходный узел.\n * @param pageUrl Базовый URL.\n * @param profile Профиль поставщика.\n * @returns Полную строку с обязательными полями либо `null`.\n */\nfunction rowFromNode(\n  $: cheerio.CheerioAPI,\n  node: cheerio.Cheerio<AnyNode>,\n  pageUrl: string,\n  profile: SupplierProfile | undefined,\n): ProductRow | null {\n  const root = expandToLikelyProductContainer($, node, profile);\n\n  const link = bestProductLink($, root, profile);\n  const productUrl = link ? absoluteUrl(pageUrl, link.attr("href")) : "";\n  const name = extractName($, root, link);\n  const price = extractPrice($, root);\n  if (!name || price === null || !productUrl) return null;\n  if (isSuspiciousNonProductRow(root, name, price)) return null;\n\n  const row: ProductRow = {\n    name,\n    price,\n    product_url: productUrl,\n  };\n\n  const imageUrl = extractImageUrl(root, pageUrl);\n  if (imageUrl) row.image_url = imageUrl;\n\n  const deliveryTime = firstText($, root, [\n    ".delivery-info-widget",\n    ".delivery-info-widget__text",\n    ".order-avail-wrap_main",\n    ".catalog-product__avails",\n    "[data-auto=\'delivery-wrapper\']",\n    "[class*=\'delivery\']",\n    "[class*=\'availability\']",\n    ".scenario-availability-item",\n    "[data-qa=\'product-availability\']",\n  ]);\n  if (deliveryTime) row.delivery_time = deliveryTime;\n\n  const stock = extractStock(root);\n  if (stock !== undefined) row.in_stock = stock;\n\n  return row;\n}\n\n/**\n * Выбирает наиболее вероятную товарную ссылку внутри карточки.\n *\n * @param $ Документ Cheerio.\n * @param node Корень карточки.\n * @param profile Профиль с подсказками URL.\n * @returns Лучший элемент `<a>` либо `null`.\n */\nfunction bestProductLink(\n  $: cheerio.CheerioAPI,\n  node: cheerio.Cheerio<AnyNode>,\n  profile: SupplierProfile | undefined,\n): cheerio.Cheerio<AnyNode> | null {\n  let best: AnyNode | null = null;\n  let bestScore = Number.NEGATIVE_INFINITY;\n  const links = node.is("a[href]")\n    ? node.add(node.find("a[href]"))\n    : node.find("a[href]");\n  links.each((_, element) => {\n    const link = $(element);\n    const href = cleanText(link.attr("href"));\n    const absolute = absoluteUrl("https://example.test", href).toLowerCase();\n    const hints = profile?.productUrlHints ?? [];\n    if (\n      !looksLikeProductHref(href) &&\n      !hints.some((hint) => absolute.includes(hint))\n    ) {\n      return;\n    }\n    const label =\n      ownOrDescendantText(link) ||\n      cleanText(link.attr("title")) ||\n      cleanText(link.attr("aria-label"));\n    let score = 1;\n    if (label.length >= 8) score += 2;\n    if (hints.some((hint) => absolute.includes(hint))) score += 3;\n    if (cleanText(link.attr("data-auto")) === "snippet-link") score += 4;\n    if (cleanText(link.parent().attr("data-zone-name")) === "title") score += 2;\n    if (/title|name|product/i.test(cleanText(link.attr("class")))) score += 2;\n    if (score > bestScore) {\n      bestScore = score;\n      best = element;\n    }\n  });\n  return best ? $(best) : null;\n}\n\n/**\n * Извлекает и очищает название товара из наиболее надёжного источника.\n *\n * @param $ Документ Cheerio.\n * @param node Корень карточки.\n * @param link Уже выбранная товарная ссылка.\n * @returns Название длиной не более 240 символов.\n */\nfunction extractName(\n  $: cheerio.CheerioAPI,\n  node: cheerio.Cheerio<AnyNode>,\n  link: cheerio.Cheerio<AnyNode> | null,\n): string {\n  const fromSpecificSelectors = firstText($, node, [\n    // Stable product-title selectors first.\n    "a.catalog-product__name",\n    ".catalog-product__name",\n    "[data-auto=\'snippet-title\']",\n    "a[data-qa=\'product-name\']",\n    "[data-qa=\'product-name\']",\n\n    // Generic snippet cards.\n    "[itemprop=\'name\']",\n    "a.catalog-product__name",\n    ".catalog-product__name",\n    ".snippet__title",\n    ".snippet__title span",\n    "a.snippet__title",\n    "a.snippet__title span",\n    ".snippet-title",\n    ".snippet-title span",\n\n    // Generic product-card cards.\n    ".product-card__name",\n    "a.product-card__name",\n    ".product-card__title",\n    "a.product-card__title",\n  ]);\n\n  const fromGenericSelectors = firstText($, node, [\n    // Generic selectors only after product-specific selectors.\n    "[class*=\'product\'][class*=\'title\']",\n    "[class*=\'product\'][class*=\'name\']",\n    "[class*=\'goods\'][class*=\'title\']",\n    "[class*=\'goods\'][class*=\'name\']",\n  ]);\n\n  const value =\n    fromSpecificSelectors ||\n    fromGenericSelectors ||\n    cleanText(link?.attr("title")) ||\n    cleanText(link?.attr("aria-label")) ||\n    cleanText(node.attr("data-description")) ||\n    (link ? ownOrDescendantText(link) : "") ||\n    firstAttr(node, ["img[alt]"], "alt");\n\n  return cleanProductName(value)\n    .replace(/^Открыть карточку товара\\s+/i, "")\n    .replace(/^(купить|в корзину|подробнее)\\s+/i, "")\n    .slice(0, 240);\n}\n\n/**\n * Извлекает текущую цену, избегая старых цен, кредита и артикула.\n *\n * @param $ Документ Cheerio.\n * @param node Корень карточки.\n * @returns Положительную цену либо `null`.\n */\nfunction extractPrice(\n  $: cheerio.CheerioAPI,\n  node: cheerio.Cheerio<AnyNode>,\n): number | null {\n  const attrPrice =\n    cleanText(node.find("[data-meta-price]").first().attr("data-meta-price")) ||\n    ownOrDescendantText(\n      node.find("[data-auto=\'snippet-price-current\']").first(),\n    ) ||\n    cleanText(node.attr("data-retail-price")) ||\n    cleanText(node.attr("data-price")) ||\n    cleanText(\n      node.find(".js-product-price[data-price-without-nds]").first().text(),\n    ) ||\n    cleanText(\n      node\n        .find(".js-product-price[data-price-without-nds]")\n        .first()\n        .attr("data-price-without-nds"),\n    ) ||\n    cleanText(node.find("[class*=\'price\'][content]").first().attr("content")) ||\n    cleanText(\n      node.find("[content][itemprop=\'price\']").first().attr("content"),\n    ) ||\n    cleanText(node.find("meta[itemprop=\'price\']").first().attr("content"));\n  const attrParsed = coercePrice(attrPrice);\n  if (attrParsed !== null) return attrParsed;\n\n  const priceCandidates = node.find(\n    [\n      ".product-buy__price",\n      ".catalog-product__buy .product-buy__price",\n      "[data-auto=\'snippet-price-current\']",\n      ".snippet-price__value",\n      ".snippet__price .snippet-price__value",\n      ".product-card__price-current",\n      ".product-card__price-container .product-card__price-current",\n      ".current-price",\n      ".price-main",\n      ".product-list-item-price",\n      "[class*=\'price\']:not([class*=\'old\']):not([class*=\'credit\']):not([class*=\'month\'])",\n      "[data-qa*=\'price\']",\n    ].join(", "),\n  );\n  let fallbackPrice: number | null = null;\n  for (const element of priceCandidates.toArray()) {\n    const priceText = ownOrDescendantText($(element));\n    if (!priceText || /артикул|article|код товара/i.test(priceText)) continue;\n    const selectedParsed = coercePrice(priceText);\n    if (selectedParsed === null) continue;\n    if (/₽|руб|р\\.|price|цена/i.test(priceText)) return selectedParsed;\n    fallbackPrice ??= selectedParsed;\n  }\n  if (fallbackPrice !== null) return fallbackPrice;\n\n  const fullText = ownOrDescendantText(node);\n  if (/₽|руб|р\\.|цена/i.test(fullText)) {\n    return coercePrice(fullText);\n  }\n\n  return null;\n}\n\n/**\n * Получает первое настоящее изображение карточки.\n *\n * @param node Корень карточки.\n * @param pageUrl Базовый URL.\n * @returns Абсолютный URL либо пустую строку для placeholder-а.\n */\nfunction extractImageUrl(\n  node: cheerio.Cheerio<AnyNode>,\n  pageUrl: string,\n): string {\n  const image = node.find("img").first();\n  const value =\n    image.attr("src") ||\n    image.attr("data-src") ||\n    image.attr("data-original") ||\n    image.attr("srcset")?.split(",", 1)[0]?.trim().split(/\\s+/, 1)[0];\n  if (!value || /placeholder|loader|spinner/i.test(value)) return "";\n  return absoluteUrl(pageUrl, value);\n}\n\n/**\n * Определяет наличие товара по текстовым маркерам карточки.\n *\n * @param node Корень карточки.\n * @returns `true`, `false` или `undefined` при отсутствии уверенного признака.\n */\nfunction extractStock(node: cheerio.Cheerio<AnyNode>): boolean | undefined {\n  const text = ownOrDescendantText(node).toLowerCase();\n  if (/нет в наличии|нет на складе|out of stock|недоступен/.test(text))\n    return false;\n  if (/есть в наличии|в наличии|на складе|купить|в корзину/.test(text))\n    return true;\n  return undefined;\n}\n\n/**\n * Извлекает товары из встроенных JSON-данных страницы.\n *\n * Ozon обрабатывается отдельным быстрым парсером. Для остальных сайтов\n * объединяются JSON-LD, состояния популярных frontend-фреймворков и\n * совместимый fallback Citilink.\n *\n * @param html Полный HTML.\n * @param pageUrl URL страницы.\n * @param profile Профиль поставщика.\n * @returns Уникальные товарные строки.\n */\nfunction extractRowsFromEmbeddedJson(\n  html: string,\n  pageUrl: string,\n  profile: SupplierProfile | undefined,\n): ProductRow[] {\n  const rows: ProductRow[] = [];\n\n  // Ozon лучше оставить отдельным быстрым special-case:\n  // его HTML часто плохо парсится обычными DOM/JSON эвристиками.\n  if (profile?.domain === "ozon.ru") {\n    return extractOzonRows(html, pageUrl);\n  }\n\n  // Универсальный schema.org / JSON-LD fallback.\n  rows.push(...extractJsonLdProductRows(html, pageUrl));\n\n  // Универсальный fallback для __NEXT_DATA__, __NUXT__, __INITIAL_STATE__.\n  rows.push(...extractGenericStateRows(html, pageUrl));\n\n  // Старый Citilink-specific fallback оставляем, но больше не делаем early return\n  // для всех остальных сайтов.\n  if (profile?.domain === "citilink.ru") {\n    const productObjects =\n      html.match(/"products"\\s*:\\s*\\[[\\s\\S]{0,200000}?\\]/g) ?? [];\n    for (const productBlock of productObjects.slice(0, 3)) {\n      const productMatches = productBlock.matchAll(\n        /"name"\\s*:\\s*"([^"]{6,240})"[\\s\\S]{0,2500}?"price"\\s*:\\s*"?(\\d[\\d\\s.,]*)"?[\\s\\S]{0,2500}?"url"\\s*:\\s*"([^"]+)"/g,\n      );\n      for (const match of productMatches) {\n        const price = coercePrice(match[2]);\n        const name = cleanText(match[1]);\n        const productUrl = absoluteUrl(pageUrl, match[3].replace(/\\\\\\//g, "/"));\n        if (name && price && productUrl) {\n          rows.push({ name, price, product_url: productUrl });\n        }\n      }\n    }\n  }\n\n  return uniqueRows(rows);\n}\n\n/**\n * Извлекает карточки Ozon из повторяющихся фрагментов tile-разметки.\n *\n * @param html HTML выдачи Ozon.\n * @param pageUrl Базовый URL.\n * @returns Уникальные строки с названием, ценой и ссылкой.\n */\nfunction extractOzonRows(html: string, pageUrl: string): ProductRow[] {\n  const tileMarker =\n    /<div[^>]*class="[^"]*\\btile-root\\b[^"]*"[^>]*data-index="\\d+"[^>]*>|<div[^>]*data-index="\\d+"[^>]*class="[^"]*\\btile-root\\b[^"]*"[^>]*>/gi;\n  const productLinkMarker =\n    /<a[^>]*href="\\/product\\/[^"]+"[^>]*class="[^"]*\\btile-clickable-element\\b[^"]*"/gi;\n  const positions = [...html.matchAll(tileMarker)].map(\n    (match) => match.index ?? 0,\n  );\n  if (positions.length === 0) {\n    positions.push(\n      ...[...html.matchAll(productLinkMarker)].map((match) => match.index ?? 0),\n    );\n  }\n\n  const rows: ProductRow[] = [];\n  for (let index = 0; index < positions.length; index += 1) {\n    const start = positions[index] ?? 0;\n    const end = positions[index + 1] ?? html.length;\n    const chunk = html.slice(start, end);\n    const href = chunk.match(/<a[^>]*href="(?<value>\\/product\\/[^"]+)"/i)\n      ?.groups?.value;\n    const priceText = chunk.match(\n      /<span[^>]*tsHeadline500Medium[^>]*>(?<value>[^<]+)<\\/span>/i,\n    )?.groups?.value;\n    const titleMatches = [\n      ...chunk.matchAll(\n        /<span[^>]*tsBody500Medium[^>]*>(?<value>[^<]+)<\\/span>/gi,\n      ),\n    ]\n      .map((match) => htmlText(match.groups?.value))\n      .filter(Boolean);\n    const name = titleMatches.sort(\n      (left, right) => right.length - left.length,\n    )[0];\n    const price = coercePrice(htmlText(priceText));\n    if (!href || !name || price === null) continue;\n    rows.push({ name, price, product_url: absoluteUrl(pageUrl, href) });\n  }\n  return uniqueRows(rows);\n}\n\n/**\n * Декодирует HTML-фрагмент и возвращает его видимый текст.\n *\n * @param value Фрагмент HTML или `undefined`.\n * @returns Очищенный текст.\n */\nfunction htmlText(value: string | undefined): string {\n  if (!value) return "";\n  return cleanText(cheerio.load(`<span>${value}</span>`)("span").text());\n}\n\n/**\n * Удаляет точные дубли строк по URL, имени и цене.\n *\n * @param rows Исходные строки.\n * @returns Первый экземпляр каждой уникальной комбинации.\n */\nfunction uniqueRows(rows: ProductRow[]): ProductRow[] {\n  const result: ProductRow[] = [];\n  const seen = new Set<string>();\n  for (const row of rows) {\n    const key = `${row.product_url}|${row.name}|${row.price}`;\n    if (seen.has(key)) continue;\n    seen.add(key);\n    result.push(row);\n  }\n  return result;\n}\n\n/**\n * Преобразует внутренние строки в публичную модель результата поиска.\n *\n * @param rows Проверенные товарные строки.\n * @returns Товары с уникальными URL и положительными ценами.\n */\nfunction buildProductsFromRows(rows: ProductRow[]): SearchProduct[] {\n  const products: SearchProduct[] = [];\n  const seenUrls = new Set<string>();\n  for (const row of rows) {\n    if (!row.name || !row.product_url || !row.price || row.price <= 0) continue;\n    if (seenUrls.has(row.product_url)) continue;\n    seenUrls.add(row.product_url);\n    products.push({\n      name: row.name,\n      price: row.price,\n      url: row.product_url,\n      delivery_time: row.delivery_time,\n      rating: row.rating,\n      in_stock: row.in_stock,\n    });\n  }\n  return products;\n}\n\n/**\n * Подсчитывает строки с отсутствующими обязательными полями.\n *\n * @param rows Анализируемые строки.\n * @returns Счётчики отсутствующих имени, цены и URL.\n */\nfunction countRejectedRows(rows: ProductRow[]): Record<string, number> {\n  const counters = {\n    missing_name: 0,\n    missing_price: 0,\n    missing_url: 0,\n  };\n  for (const row of rows) {\n    if (!row.name) counters.missing_name += 1;\n    if (!row.price || row.price <= 0) counters.missing_price += 1;\n    if (!row.product_url) counters.missing_url += 1;\n  }\n  return counters;\n}\n\n/**\n * Создаёт ранний результат для заблокированной или пустой страницы.\n *\n * @param args Статус, маршрут, URL, строки и диагностические маркеры.\n * @returns Результат с пустым публичным списком продуктов.\n */\nfunction buildResult(args: {\n  status: "ok" | "empty" | "blocked";\n  route: string;\n  pageUrl: string;\n  rows: ProductRow[];\n  blockedMarkers: string[];\n  debug: string[];\n}): ExtractProductsResult {\n  return {\n    status: args.status,\n    route: args.route,\n    pageUrl: args.pageUrl,\n    rows: args.rows,\n    products: [],\n    diagnostics: {\n      blockedMarkers: args.blockedMarkers,\n      candidateCount: 0,\n      rowCount: 0,\n      rejectedRows: {},\n      debug: args.debug,\n    },\n  };\n}\n\n/**\n * Расширяет найденный элемент до контейнера, объединяющего поля одного товара.\n *\n * @param $ Документ Cheerio.\n * @param node Начальный узел.\n * @param profile Профиль поставщика.\n * @returns Первый предок со ссылкой, ценой и именем либо лучший просмотренный.\n */\nfunction expandToLikelyProductContainer(\n  $: cheerio.CheerioAPI,\n  node: cheerio.Cheerio<AnyNode>,\n  profile: SupplierProfile | undefined,\n): cheerio.Cheerio<AnyNode> {\n  let current = node;\n  let best = node;\n  let bestScore = -1;\n\n  for (let depth = 0; depth < 8; depth += 1) {\n    const text = ownOrDescendantText(current);\n    const hasProductLink = bestProductLink($, current, profile) !== null;\n\n    const hasPrice =\n      current.find(\n        [\n          "[data-qa=\'product-price-current\']",\n          "[data-auto=\'snippet-price-current\']",\n          "[data-qa*=\'price\']",\n          "[itemprop=\'price\']",\n          "[content][itemprop=\'price\']",\n          ".product-buy__price",\n          ".catalog-product__buy .product-buy__price",\n          "[data-price]",\n          ".snippet-price__value",\n          ".snippet__price .snippet-price__value",\n          ".product-card__price-current",\n          ".product-card__price-container .product-card__price-current",\n          ".price-value",\n          ".price_value",\n          ".current-price",\n          ".price-main",\n          ".Product__price",\n          ".v-product-price__value",\n          "[class*=\'price\']",\n        ].join(", "),\n      ).length > 0 || /₽|руб|р\\./i.test(text);\n\n    const hasName =\n      current.find(\n        [\n          "a[data-qa=\'product-name\']",\n          "[data-auto=\'snippet-title\']",\n          "[data-qa=\'product-name\']",\n          ".snippet__title",\n          "a.snippet__title",\n          ".snippet-title",\n          ".product-card__name",\n          "a.product-card__name",\n          ".product-card__title",\n          "a.product-card__title",\n          ".item-name-link",\n          ".item-name a",\n          ".name a",\n          ".product-title",\n          ".catalog-product__name",\n          "[class*=\'product\'][class*=\'name\']",\n          "[class*=\'product\'][class*=\'title\']",\n        ].join(", "),\n      ).length > 0;\n\n    const className = cleanText(current.attr("class")).toLowerCase();\n    let score = 0;\n    if (hasProductLink) score += 3;\n    if (hasPrice) score += 3;\n    if (hasName) score += 3;\n    if (\n      /(^|\\s)(snippet|product-card|catalog-product|product-item|catalog-item)(\\s|$)/.test(\n        className,\n      )\n    ) {\n      score += 2;\n    }\n\n    if (score > bestScore && text.length < 4500) {\n      best = current;\n      bestScore = score;\n    }\n\n    if (hasProductLink && hasPrice && hasName) {\n      return current;\n    }\n\n    const parent = current.parent();\n    if (parent.length === 0 || parent.is("body") || parent.is("html")) break;\n    current = parent;\n  }\n\n  return best;\n}\n\n/**\n * Извлекает товары из schema.org Product в JSON-LD.\n *\n * @param html HTML документа.\n * @param pageUrl Базовый URL.\n * @returns Уникальные строки с данными первого Offer каждого продукта.\n */\nfunction extractJsonLdProductRows(html: string, pageUrl: string): ProductRow[] {\n  const rows: ProductRow[] = [];\n  const scriptMatches = html.matchAll(\n    /<script[^>]+type=["\']application\\/ld\\+json["\'][^>]*>([\\s\\S]*?)<\\/script>/gi,\n  );\n\n  for (const match of scriptMatches) {\n    const raw = htmlText(match[1]);\n    if (!raw) continue;\n\n    let parsed: unknown;\n    try {\n      parsed = JSON.parse(raw);\n    } catch {\n      continue;\n    }\n\n    for (const item of flattenJsonLd(parsed)) {\n      if (!isRecord(item)) continue;\n\n      const type = item["@type"];\n      const types = Array.isArray(type)\n        ? type.map(String)\n        : [String(type ?? "")];\n      if (!types.some((value) => value.toLowerCase() === "product")) continue;\n\n      const name = cleanProductName(item.name);\n      const offers = item.offers;\n      const offer = Array.isArray(offers) ? offers[0] : offers;\n\n      const price =\n        isRecord(offer) && offer.price !== undefined\n          ? coercePrice(offer.price)\n          : null;\n\n      const url =\n        absoluteUrl(pageUrl, item.url) ||\n        (isRecord(offer) ? absoluteUrl(pageUrl, offer.url) : "");\n\n      if (name && price && url) {\n        const row: ProductRow = {\n          name,\n          price,\n          product_url: url,\n        };\n\n        const image = Array.isArray(item.image) ? item.image[0] : item.image;\n        const imageUrl = absoluteUrl(pageUrl, image);\n        if (imageUrl) row.image_url = imageUrl;\n\n        rows.push(row);\n      }\n    }\n  }\n\n  return uniqueRows(rows);\n}\n\n/**\n * Разворачивает массивы и `@graph` JSON-LD в линейный список значений.\n *\n * @param value Разобранный JSON-LD.\n * @returns Все доступные объекты верхнего уровня и графа.\n */\nfunction flattenJsonLd(value: unknown): unknown[] {\n  if (Array.isArray(value)) return value.flatMap(flattenJsonLd);\n  if (!isRecord(value)) return [value];\n\n  const graph = value["@graph"];\n  if (Array.isArray(graph)) return [value, ...graph.flatMap(flattenJsonLd)];\n\n  return [value];\n}\n\n/**\n * Проверяет, является ли значение непустым объектом с ключами.\n *\n * @param value Неизвестное значение.\n * @returns TypeScript type guard для `Record<string, unknown>`.\n */\nfunction isRecord(value: unknown): value is Record<string, unknown> {\n  return typeof value === "object" && value !== null;\n}\n\n/**\n * Извлекает товары из сериализованного состояния frontend-приложения.\n *\n * @param html HTML с `__NEXT_DATA__`, `__NUXT__` или похожими объектами.\n * @param pageUrl Базовый URL.\n * @returns До 500 уникальных товарных строк.\n */\nfunction extractGenericStateRows(html: string, pageUrl: string): ProductRow[] {\n  const rows: ProductRow[] = [];\n  const jsonChunks: string[] = [];\n\n  const nextMatch = html.match(\n    /<script[^>]+id=["\']__NEXT_DATA__["\'][^>]*>([\\s\\S]*?)<\\/script>/i,\n  );\n  if (nextMatch?.[1]) {\n    jsonChunks.push(htmlText(nextMatch[1]));\n  }\n\n  const stateMatches = html.matchAll(\n    /(?:window\\.)?(?:__NUXT__|__INITIAL_STATE__|__APOLLO_STATE__|__REDUX_STATE__|__PRELOADED_STATE__)\\s*=\\s*({[\\s\\S]{100,800000}?})\\s*;?\\s*<\\/script>/gi,\n  );\n\n  for (const match of stateMatches) {\n    if (match[1]) {\n      jsonChunks.push(match[1]);\n    }\n  }\n\n  for (const chunk of jsonChunks) {\n    let parsed: unknown;\n    try {\n      parsed = JSON.parse(chunk);\n    } catch {\n      continue;\n    }\n\n    for (const object of walkObjects(parsed, 25_000)) {\n      const row = rowFromUnknownObject(object, pageUrl);\n      if (row) {\n        rows.push(row);\n      }\n    }\n  }\n\n  return uniqueRows(rows).slice(0, 500);\n}\n\n/**\n * Итеративно обходит вложенные объекты без риска переполнить стек.\n *\n * @param value Корневое JSON-значение.\n * @param maxObjects Ограничение числа возвращаемых объектов.\n * @returns Найденные объекты в порядке обхода стека.\n */\nfunction walkObjects(\n  value: unknown,\n  maxObjects: number,\n): Record<string, unknown>[] {\n  const result: Record<string, unknown>[] = [];\n  const stack: unknown[] = [value];\n\n  while (stack.length > 0 && result.length < maxObjects) {\n    const current = stack.pop();\n\n    if (Array.isArray(current)) {\n      for (const item of current) {\n        stack.push(item);\n      }\n      continue;\n    }\n\n    if (!isRecord(current)) {\n      continue;\n    }\n\n    result.push(current);\n\n    for (const child of Object.values(current)) {\n      if (typeof child === "object" && child !== null) {\n        stack.push(child);\n      }\n    }\n  }\n\n  return result;\n}\n\n/**\n * Пытается распознать товар в произвольном объекте состояния страницы.\n *\n * @param object JSON-объект.\n * @param pageUrl Базовый URL.\n * @returns Строку только при наличии правдоподобных имени, цены и товарной\n * ссылки.\n */\nfunction rowFromUnknownObject(\n  object: Record<string, unknown>,\n  pageUrl: string,\n): ProductRow | null {\n  const name = cleanProductName(\n    firstUnknownValue(object, [\n      "name",\n      "title",\n      "productName",\n      "product_name",\n      "displayName",\n      "display_name",\n      "caption",\n      "label",\n    ]),\n  );\n\n  const price = coercePrice(\n    firstUnknownValue(object, [\n      "price",\n      "currentPrice",\n      "current_price",\n      "actualPrice",\n      "actual_price",\n      "salePrice",\n      "sale_price",\n      "priceValue",\n      "price_value",\n      "cost",\n      "amount",\n    ]),\n  );\n\n  const rawUrl = firstUnknownValue(object, [\n    "url",\n    "href",\n    "link",\n    "productUrl",\n    "product_url",\n    "detailUrl",\n    "detail_url",\n  ]);\n\n  const productUrl = absoluteUrl(pageUrl, rawUrl);\n\n  if (!name || price === null || !productUrl) {\n    return null;\n  }\n\n  if (name.length < 4 || name.length > 260) {\n    return null;\n  }\n\n  if (!looksLikeProductHref(productUrl)) {\n    return null;\n  }\n\n  const row: ProductRow = {\n    name,\n    price,\n    product_url: productUrl,\n  };\n\n  const rawImage = firstUnknownValue(object, [\n    "image",\n    "imageUrl",\n    "image_url",\n    "picture",\n    "pictureUrl",\n    "picture_url",\n    "photo",\n    "photoUrl",\n    "photo_url",\n    "thumbnail",\n    "thumbnailUrl",\n  ]);\n\n  const imageUrl = imageValueToUrl(rawImage, pageUrl);\n  if (imageUrl) {\n    row.image_url = imageUrl;\n  }\n\n  return row;\n}\n\n/**\n * Возвращает первое непустое значение среди синонимичных ключей.\n *\n * @param object Исследуемый объект.\n * @param keys Ключи в порядке приоритета.\n * @returns Найденное значение либо `undefined`.\n */\nfunction firstUnknownValue(\n  object: Record<string, unknown>,\n  keys: string[],\n): unknown {\n  for (const key of keys) {\n    const value = object[key];\n    if (value !== undefined && value !== null && value !== "") {\n      return value;\n    }\n  }\n  return undefined;\n}\n\n/**\n * Преобразует строку, массив или объект изображения в абсолютный URL.\n *\n * @param value Значение поля изображения.\n * @param pageUrl Базовый URL.\n * @returns Первый доступный URL либо пустую строку.\n */\nfunction imageValueToUrl(value: unknown, pageUrl: string): string {\n  if (typeof value === "string") {\n    return absoluteUrl(pageUrl, value);\n  }\n\n  if (Array.isArray(value)) {\n    for (const item of value) {\n      const url = imageValueToUrl(item, pageUrl);\n      if (url) return url;\n    }\n    return "";\n  }\n\n  if (isRecord(value)) {\n    return (\n      absoluteUrl(pageUrl, value.url) ||\n      absoluteUrl(pageUrl, value.src) ||\n      absoluteUrl(pageUrl, value.href)\n    );\n  }\n\n  return "";\n}\n', 'src/candidates.ts': 'import * as cheerio from "cheerio";\nimport type { AnyNode } from "domhandler";\n\nimport type {\n  ProductCandidate,\n  ProductCandidateFieldDiagnostics,\n  ProductRow,\n  SupplierProfile,\n} from "./contracts.js";\nimport {\n  absoluteUrl,\n  cleanProductName,\n  cleanText,\n  coercePrice,\n  looksLikeProductHref,\n  ownOrDescendantText,\n  splitSelectorList,\n} from "./utils.js";\n\nconst CANDIDATE_SELECTORS = [\n  // explicit ecommerce / product blocks\n  "[itemtype*=\'Product\']",\n  "[itemscope][itemtype*=\'schema.org/Product\']",\n  "[data-product-id]",\n  "[data-product]",\n  "[data-offer-id]",\n  "[data-sku]",\n  "[data-id=\'product\']",\n  "[data-entity=\'item\']",\n  "[data-testid*=\'product\' i]",\n  "[data-test*=\'product\' i]",\n  "[data-qa*=\'product\' i]",\n\n  // repeated card blocks first\n  ".catalog__list-item.snippet",\n  ".promo-slider__item.snippet",\n  ".snippet",\n  ".snippet__content",\n  "[class~=\'snippet\']",\n\n  // class-based generic\n  "[class*=\'product-card\' i]",\n  "[class*=\'product_card\' i]",\n  "[class*=\'productCard\']",\n  "[class*=\'product-item\' i]",\n  "[class*=\'product_item\' i]",\n  "[class*=\'catalog-item\' i]",\n  "[class*=\'catalog_item\' i]",\n  "[class*=\'goods-item\' i]",\n  "[class*=\'goods_item\' i]",\n  "[class*=\'item-card\' i]",\n  "[class*=\'card-item\' i]",\n  "[class*=\'snippet\' i]",\n  "[class*=\'tile\' i]",\n\n  // broad but useful; keep last\n  "article",\n  "li",\n  "tr",\n  ".item",\n  ".card",\n];\n\nconst TITLE_SELECTORS = [\n  "[itemprop=\'name\']",\n  ".snippet__title",\n  ".snippet__title span",\n  "a.snippet__title",\n  "a.snippet__title span",\n  ".snippet-title",\n  ".snippet-title span",\n  ".product-card__name",\n  "a.product-card__name",\n  ".product-card__title",\n  "a.product-card__title",\n  "[class*=\'title\' i]",\n  "[class*=\'name\' i]",\n  "[class*=\'product-name\' i]",\n  "[class*=\'product-title\' i]",\n  "[data-qa*=\'name\' i]",\n  "[data-testid*=\'name\' i]",\n  "h1",\n  "h2",\n  "h3",\n  "a[title]",\n  "img[alt]",\n];\nconst PRICE_SELECTORS = [\n  "[itemprop=\'price\']",\n  "[content][itemprop=\'price\']",\n  "[data-price]",\n  "[data-retail-price]",\n  "[data-meta-price]",\n  ".snippet-price__value",\n  ".snippet__price .snippet-price__value",\n  ".product-card__price-current",\n  ".product-card__price-container .product-card__price-current",\n  "[class*=\'price\' i]:not([class*=\'old\' i]):not([class*=\'credit\' i]):not([class*=\'month\' i])",\n  "[data-qa*=\'price\' i]",\n  "[data-testid*=\'price\' i]",\n];\n\nconst IMAGE_SELECTORS = [\n  "img[src]",\n  "img[data-src]",\n  "img[data-original]",\n  "img[srcset]",\n  "source[srcset]",\n];\n\nconst BAD_TEXT_RE =\n  /купить|в корзину|подробнее|сравнить|избранное|отзывы|обзор|быстрый просмотр|затрудняетесь|помогут.*определиться|материалы, которые помогут|выборе/i;\nconst PRICE_TEXT_RE =\n  /\\d{1,3}(?:[ \\u00a0\\u202f\\u2009]\\d{3})+(?:[.,]\\d+)?\\s*(?:₽|руб\\.?|р\\.?)|\\d+(?:[.,]\\d+)?\\s*(?:₽|руб\\.?|р\\.?)/i;\n\n/**\n * Находит и ранжирует DOM-узлы, похожие на самостоятельные карточки товара.\n *\n * Поиск объединяет пользовательский селектор, профиль поставщика и общие\n * ecommerce-селекторы. Каждый найденный узел расширяется до наиболее\n * вероятного корня карточки, получает поля и диагностическую оценку, после\n * чего дубликаты удаляются.\n *\n * @param args Документ Cheerio, URL страницы, профиль, селектор и лимит.\n * @returns Лучшие кандидаты с полями, оценками и причинами принятия/отказа.\n */\nexport function discoverProductCandidates(args: {\n  $: cheerio.CheerioAPI;\n  pageUrl: string;\n  profile?: SupplierProfile;\n  itemSelector?: string | null;\n  limit?: number;\n}): ProductCandidate[] {\n  const { $, pageUrl, profile } = args;\n  const limit = Math.max(1, Math.min(100, args.limit ?? 24));\n\n  const selectors = [\n    ...splitSelectorList(args.itemSelector),\n    ...splitSelectorList(profile?.itemSelector),\n    ...CANDIDATE_SELECTORS,\n  ];\n\n  const seen = new Set<AnyNode>();\n  const candidates: ProductCandidate[] = [];\n  let inspectedNodeCount = 0;\n  const maxInspectedNodes = 1500;\n\n  for (const selector of selectors) {\n    if (inspectedNodeCount >= maxInspectedNodes) break;\n\n    let nodes: cheerio.Cheerio<AnyNode>;\n    try {\n      nodes = $(selector);\n    } catch {\n      continue;\n    }\n\n    nodes.each((_, element) => {\n      if (inspectedNodeCount >= maxInspectedNodes) return false;\n      inspectedNodeCount += 1;\n\n      if (seen.has(element)) return;\n      seen.add(element);\n\n      const root = expandToCardRoot($, $(element), profile);\n      const identityNode = root.get(0);\n      if (!identityNode || seen.has(identityNode)) return;\n      seen.add(identityNode);\n\n      const candidate = buildCandidate({\n        $,\n        node: root,\n        pageUrl,\n        selector,\n        profile,\n        index: candidates.length,\n      });\n\n      if (candidate.score > 0) {\n        candidates.push(candidate);\n      }\n\n      if (candidates.length >= limit * 3) return false;\n      return undefined;\n    });\n  }\n\n  candidates.sort((left, right) => right.score - left.score);\n\n  return dedupeCandidates(candidates).slice(0, limit);\n}\n\n/**\n * Извлекает поля одного DOM-кандидата и рассчитывает его итоговую оценку.\n *\n * Наличие имени, цены и товарной ссылки повышает балл. Повторяемая структура\n * карточек даёт дополнительную уверенность, а слишком крупные, одиночные,\n * навигационные и рекламные контейнеры штрафуются или явно отклоняются.\n *\n * @param args Узел, документ, URL, исходный селектор, профиль и индекс.\n * @returns Полное диагностическое описание кандидата.\n */\nfunction buildCandidate(args: {\n  $: cheerio.CheerioAPI;\n  node: cheerio.Cheerio<AnyNode>;\n  pageUrl: string;\n  selector: string;\n  profile?: SupplierProfile;\n  index: number;\n}): ProductCandidate {\n  const { $, node, pageUrl, selector, profile, index } = args;\n\n  const fieldDiagnostics: Record<string, ProductCandidateFieldDiagnostics> = {};\n\n  const name = extractNameField($, node);\n  fieldDiagnostics.name = name.diagnostics;\n\n  const price = extractPriceField($, node);\n  fieldDiagnostics.price = price.diagnostics;\n\n  const productUrl = extractProductUrlField($, node, pageUrl, profile);\n  fieldDiagnostics.product_url = productUrl.diagnostics;\n\n  const imageUrl = extractImageField($, node, pageUrl);\n  fieldDiagnostics.image_url = imageUrl.diagnostics;\n\n  const stock = extractStockField(node);\n  fieldDiagnostics.in_stock = stock.diagnostics;\n\n  const delivery = extractDeliveryField($, node);\n  fieldDiagnostics.delivery_time = delivery.diagnostics;\n\n  const fields: Partial<ProductRow> = {};\n  if (name.value) fields.name = name.value;\n  if (price.value !== null) fields.price = price.value;\n  if (productUrl.value) fields.product_url = productUrl.value;\n  if (imageUrl.value) fields.image_url = imageUrl.value;\n  if (typeof stock.value === "boolean") fields.in_stock = stock.value;\n  if (delivery.value) fields.delivery_time = delivery.value;\n\n  const missingFields = ["name", "price", "product_url"].filter(\n    (field) => !(field in fields),\n  );\n\n  const reasons: string[] = [];\n  let score = 0;\n\n  if (fields.name) {\n    score += 120;\n    reasons.push("has_name");\n  }\n  if (fields.price) {\n    score += 140;\n    reasons.push("has_price");\n  }\n  if (fields.product_url) {\n    score += 130;\n    reasons.push("has_product_url");\n  }\n  if (fields.image_url) {\n    score += 30;\n    reasons.push("has_image");\n  }\n  if (typeof fields.in_stock === "boolean") {\n    score += 15;\n    reasons.push("has_stock");\n  }\n  if (fields.delivery_time) {\n    score += 10;\n    reasons.push("has_delivery");\n  }\n\n  const text = cleanText(node.text());\n  const html = $.html(node);\n  const repetition = repeatedAncestorGroupInfo($, node);\n\n  if (repetition.siblingCount >= 3) {\n    const bonus = Math.min(220, 40 + repetition.siblingCount * 8);\n    score += bonus;\n    reasons.push(`repeated_card_group:${repetition.siblingCount}`);\n  } else {\n    score -= 70;\n    reasons.push("isolated_candidate");\n  }\n\n  if (text.length < 12) {\n    score -= 80;\n    reasons.push("too_short");\n  }\n  if (text.length > 3500) {\n    score -= 180;\n    reasons.push("too_large");\n  }\n  if (node.find("form,input,select,textarea").length > 4) {\n    score -= 180;\n    reasons.push("too_many_controls");\n  }\n  if (looksLikeNavigationOrLayout(node)) {\n    score -= 260;\n    reasons.push("layout_or_navigation");\n  }\n  if (looksLikeHelpOrMarketing(node, text)) {\n    score -= 360;\n    reasons.push("help_or_marketing_block");\n  }\n  if (fields.product_url && looksLikeCategoryOnlyUrl(fields.product_url)) {\n    score -= 260;\n    reasons.push("category_url_not_product_url");\n  }\n\n  const explicitRejectReason = looksLikeHelpOrMarketing(node, text)\n    ? "help_or_marketing_block"\n    : fields.product_url && looksLikeCategoryOnlyUrl(fields.product_url)\n      ? "category_url_not_product_url"\n      : undefined;\n\n  const rejectReason =\n    explicitRejectReason ??\n    (missingFields.length > 0\n      ? `missing_${missingFields.join("_")}`\n      : undefined);\n  return {\n    index,\n    selector,\n    htmlPreview: html.slice(0, 3000),\n    textPreview: text.slice(0, 500),\n    score,\n    reasons,\n    fields,\n    fieldDiagnostics,\n    missingFields,\n    rejectReason,\n  };\n}\n\n/**\n * Выбирает наиболее правдоподобное название товара внутри карточки.\n *\n * @param $ Документ Cheerio.\n * @param node Корень карточки.\n * @returns Название и объяснение выбранного селектора либо диагностику\n * отсутствия кандидата.\n */\nfunction extractNameField(\n  $: cheerio.CheerioAPI,\n  node: cheerio.Cheerio<AnyNode>,\n): {\n  value: string | undefined;\n  diagnostics: ProductCandidateFieldDiagnostics;\n} {\n  const raw: string[] = [];\n\n  for (const selector of TITLE_SELECTORS) {\n    const target = node.find(selector).first();\n    const value =\n      cleanText(target.text()) ||\n      cleanText(target.attr("title")) ||\n      cleanText(target.attr("aria-label")) ||\n      cleanText(target.attr("alt"));\n\n    if (value) raw.push(value);\n\n    const normalized = cleanProductName(value);\n    if (looksLikeName(normalized)) {\n      return {\n        value: normalized.slice(0, 240),\n        diagnostics: {\n          value: normalized,\n          confidence: 0.9,\n          reason: `selected_by_selector:${selector}`,\n          raw: raw.slice(0, 8),\n        },\n      };\n    }\n  }\n\n  const linkText = cleanText(node.find("a[href]").first().text());\n  if (looksLikeName(linkText)) {\n    return {\n      value: cleanProductName(linkText).slice(0, 240),\n      diagnostics: {\n        value: linkText,\n        confidence: 0.65,\n        reason: "selected_from_first_link_text",\n        raw: [linkText],\n      },\n    };\n  }\n\n  const imageAlt = cleanText(node.find("img[alt]").first().attr("alt"));\n  if (looksLikeName(imageAlt)) {\n    return {\n      value: cleanProductName(imageAlt).slice(0, 240),\n      diagnostics: {\n        value: imageAlt,\n        confidence: 0.6,\n        reason: "selected_from_image_alt",\n        raw: [imageAlt],\n      },\n    };\n  }\n\n  return {\n    value: undefined,\n    diagnostics: {\n      confidence: 0,\n      reason: "no_name_candidate",\n      raw: raw.slice(0, 8),\n    },\n  };\n}\n\n/**\n * Извлекает текущую цену из специализированных узлов или общего текста.\n *\n * @param $ Документ Cheerio.\n * @param node Корень карточки.\n * @returns Числовую цену и диагностические исходные значения.\n */\nfunction extractPriceField(\n  $: cheerio.CheerioAPI,\n  node: cheerio.Cheerio<AnyNode>,\n): { value: number | null; diagnostics: ProductCandidateFieldDiagnostics } {\n  const raw: string[] = [];\n\n  for (const selector of PRICE_SELECTORS) {\n    const target = node.find(selector).first();\n    const text =\n      cleanText(target.attr("content")) ||\n      cleanText(target.attr("data-price")) ||\n      cleanText(target.attr("data-retail-price")) ||\n      cleanText(target.attr("data-meta-price")) ||\n      cleanText(target.text());\n\n    if (text) raw.push(text);\n\n    const price = coercePrice(text);\n    if (price !== null) {\n      return {\n        value: price,\n        diagnostics: {\n          value: price,\n          confidence: 0.9,\n          reason: `selected_by_selector:${selector}`,\n          raw: raw.slice(0, 8),\n        },\n      };\n    }\n  }\n\n  const fullText = ownOrDescendantText(node);\n  const priceMatch = fullText.match(PRICE_TEXT_RE)?.[0] ?? "";\n  const price = coercePrice(priceMatch);\n\n  return {\n    value: price,\n    diagnostics: {\n      value: price ?? undefined,\n      confidence: price !== null ? 0.55 : 0,\n      reason: price !== null ? "selected_from_full_text" : "no_price_candidate",\n      raw: priceMatch ? [priceMatch] : raw.slice(0, 8),\n    },\n  };\n}\n\n/**\n * Выбирает лучшую ссылку на карточку товара.\n *\n * Ссылки оцениваются по форме URL, подсказкам профиля, классу элемента и\n * содержательности текста.\n *\n * @param $ Документ Cheerio.\n * @param node Корень карточки.\n * @param pageUrl URL документа для разрешения относительных ссылок.\n * @param profile Профиль поставщика с характерными фрагментами URL.\n * @returns Абсолютный URL и диагностику либо отсутствие значения.\n */\nfunction extractProductUrlField(\n  $: cheerio.CheerioAPI,\n  node: cheerio.Cheerio<AnyNode>,\n  pageUrl: string,\n  profile?: SupplierProfile,\n): {\n  value: string | undefined;\n  diagnostics: ProductCandidateFieldDiagnostics;\n} {\n  const hints = profile?.productUrlHints ?? [];\n  let bestUrl = "";\n  let bestScore = Number.NEGATIVE_INFINITY;\n  const raw: string[] = [];\n\n  const links = node.is("a[href]")\n    ? node.add(node.find("a[href]"))\n    : node.find("a[href]");\n\n  links.each((_, element) => {\n    const link = $(element);\n    const href = cleanText(link.attr("href"));\n    if (!href) return;\n\n    raw.push(href);\n\n    const absolute = absoluteUrl(pageUrl, href);\n    const lowered = absolute.toLowerCase();\n\n    let score = 0;\n    if (looksLikeProductHref(href)) score += 10;\n    if (hints.some((hint) => lowered.includes(hint))) score += 12;\n    if (/title|name|product|goods|item/i.test(cleanText(link.attr("class"))))\n      score += 5;\n    if (cleanText(link.text()).length >= 8) score += 3;\n\n    if (score > bestScore) {\n      bestScore = score;\n      bestUrl = absolute;\n    }\n  });\n\n  if (bestUrl && bestScore > 0) {\n    return {\n      value: bestUrl,\n      diagnostics: {\n        value: bestUrl,\n        confidence: Math.min(1, 0.35 + bestScore / 20),\n        reason: "selected_best_product_link",\n        raw: raw.slice(0, 8),\n      },\n    };\n  }\n\n  return {\n    value: undefined,\n    diagnostics: {\n      confidence: 0,\n      reason: "no_product_url_candidate",\n      raw: raw.slice(0, 8),\n    },\n  };\n}\n\n/**\n * Извлекает первое настоящее изображение товара из карточки.\n *\n * @param $ Документ Cheerio.\n * @param node Корень карточки.\n * @param pageUrl Базовый URL.\n * @returns Абсолютный URL изображения и диагностику; placeholder-ы и\n * индикаторы загрузки пропускаются.\n */\nfunction extractImageField(\n  $: cheerio.CheerioAPI,\n  node: cheerio.Cheerio<AnyNode>,\n  pageUrl: string,\n): {\n  value: string | undefined;\n  diagnostics: ProductCandidateFieldDiagnostics;\n} {\n  for (const selector of IMAGE_SELECTORS) {\n    const image = node.find(selector).first();\n    const value =\n      cleanText(image.attr("src")) ||\n      cleanText(image.attr("data-src")) ||\n      cleanText(image.attr("data-original")) ||\n      cleanText(image.attr("srcset"))?.split(",", 1)[0]?.split(/\\s+/, 1)[0];\n\n    if (value && !/placeholder|loader|spinner/i.test(value)) {\n      const resolved = absoluteUrl(pageUrl, value);\n      return {\n        value: resolved,\n        diagnostics: {\n          value: resolved,\n          confidence: 0.8,\n          reason: `selected_by_selector:${selector}`,\n          raw: [value],\n        },\n      };\n    }\n  }\n\n  return {\n    value: undefined,\n    diagnostics: {\n      confidence: 0,\n      reason: "no_image_candidate",\n      raw: [],\n    },\n  };\n}\n\n/**\n * Определяет наличие товара по тексту карточки.\n *\n * @param node Корень карточки.\n * @returns Логическое значение с уверенностью либо `undefined`, если текст\n * неоднозначен.\n */\nfunction extractStockField(node: cheerio.Cheerio<AnyNode>): {\n  value: boolean | undefined;\n  diagnostics: ProductCandidateFieldDiagnostics;\n} {\n  const text = cleanText(node.text()).toLowerCase();\n\n  if (\n    /нет в наличии|нет на складе|out of stock|недоступен|нет в продаже/.test(\n      text,\n    )\n  ) {\n    return {\n      value: false,\n      diagnostics: {\n        value: false,\n        confidence: 0.75,\n        reason: "negative_stock_text",\n        raw: [],\n      },\n    };\n  }\n\n  if (\n    /есть в наличии|в наличии|на складе|купить|в корзину|доступно/.test(text)\n  ) {\n    return {\n      value: true,\n      diagnostics: {\n        value: true,\n        confidence: 0.65,\n        reason: "positive_stock_text",\n        raw: [],\n      },\n    };\n  }\n\n  return {\n    value: undefined,\n    diagnostics: {\n      confidence: 0,\n      reason: "no_stock_candidate",\n      raw: [],\n    },\n  };\n}\n\n/**\n * Ищет текст доставки или доступности в специализированных элементах.\n *\n * @param $ Документ Cheerio.\n * @param node Корень карточки.\n * @returns Короткий текст и диагностику либо отсутствие значения.\n */\nfunction extractDeliveryField(\n  $: cheerio.CheerioAPI,\n  node: cheerio.Cheerio<AnyNode>,\n): {\n  value: string | undefined;\n  diagnostics: ProductCandidateFieldDiagnostics;\n} {\n  const selectors = [\n    "[class*=\'delivery\' i]",\n    "[class*=\'availability\' i]",\n    "[class*=\'stock\' i]",\n    "[data-qa*=\'availability\' i]",\n  ];\n\n  for (const selector of selectors) {\n    const text = cleanText(node.find(selector).first().text());\n    if (text) {\n      return {\n        value: text.slice(0, 180),\n        diagnostics: {\n          value: text,\n          confidence: 0.65,\n          reason: `selected_by_selector:${selector}`,\n          raw: [text],\n        },\n      };\n    }\n  }\n\n  return {\n    value: undefined,\n    diagnostics: {\n      confidence: 0,\n      reason: "no_delivery_candidate",\n      raw: [],\n    },\n  };\n}\n\n/**\n * Поднимается по DOM к наиболее полному, но ещё локальному корню карточки.\n *\n * Обход прекращается перед контейнером нескольких товаров или слишком большим\n * блоком. Среди просмотренных предков возвращается узел с лучшей оценкой.\n *\n * @param $ Документ Cheerio.\n * @param node Первоначально найденный элемент.\n * @param profile Профиль поставщика.\n * @returns Наиболее вероятный корень одной товарной карточки.\n */\nfunction expandToCardRoot(\n  $: cheerio.CheerioAPI,\n  node: cheerio.Cheerio<AnyNode>,\n  profile?: SupplierProfile,\n): cheerio.Cheerio<AnyNode> {\n  let current = node;\n  let best = node;\n  let bestScore = scoreContainer($, node, profile);\n\n  for (let depth = 0; depth < 7; depth += 1) {\n    const parent = current.parent();\n    if (!parent.length || parent.is("body") || parent.is("html")) break;\n\n    const score = scoreContainer($, parent, profile) - depth * 5;\n    const textLength = cleanText(parent.text()).length;\n    const productChildCount = repeatedProductChildCount($, parent);\n\n    if (productChildCount >= 2 || textLength > 4500) break;\n\n    if (score > bestScore) {\n      best = parent;\n      bestScore = score;\n    }\n\n    current = parent;\n  }\n\n  return best;\n}\n\n/**\n * Оценивает пригодность DOM-контейнера как одной товарной карточки.\n *\n * @param $ Документ Cheerio.\n * @param node Проверяемый контейнер.\n * @param profile Профиль с подсказками товарных URL.\n * @returns Балл на основе атрибутов, цены, ссылок, изображения и размера.\n */\nfunction scoreContainer(\n  $: cheerio.CheerioAPI,\n  node: cheerio.Cheerio<AnyNode>,\n  profile?: SupplierProfile,\n): number {\n  const text = cleanText(node.text());\n  const attrs = [\n    node.attr("id"),\n    node.attr("class"),\n    node.attr("data-product-id"),\n    node.attr("data-offer-id"),\n    node.attr("data-sku"),\n    node.attr("data-qa"),\n    node.attr("data-testid"),\n  ]\n    .filter(Boolean)\n    .join(" ")\n    .toLowerCase();\n\n  let score = 0;\n\n  if (/product|goods|catalog|item|card|snippet|tile|offer|sku/.test(attrs))\n    score += 80;\n  if (PRICE_TEXT_RE.test(text)) score += 80;\n  if (node.find("a[href]").length > 0) score += 40;\n  if (node.find("img,source").length > 0) score += 25;\n  if (profile?.productUrlHints?.some((hint) => node.html()?.includes(hint)))\n    score += 60;\n\n  if (text.length < 15) score -= 80;\n  if (text.length > 3500) score -= 140;\n  if (looksLikeNavigationOrLayout(node)) score -= 180;\n\n  return score;\n}\n\n/**\n * Считает прямых потомков, каждый из которых сам похож на товар.\n *\n * @param $ Документ Cheerio.\n * @param node Родительский контейнер.\n * @returns Число вероятных карточек среди непосредственных детей.\n */\nfunction repeatedProductChildCount(\n  $: cheerio.CheerioAPI,\n  node: cheerio.Cheerio<AnyNode>,\n): number {\n  let count = 0;\n  node.children().each((_, element) => {\n    const child = $(element);\n    if (scoreContainer($, child) >= 100) count += 1;\n  });\n  return count;\n}\n\n/**\n * Проверяет, является ли узел навигацией или общим элементом макета.\n *\n * @param node DOM-контейнер.\n * @returns `true` для header/footer, меню, фильтров, боковых панелей и\n * похожих служебных блоков.\n */\nfunction looksLikeNavigationOrLayout(node: cheerio.Cheerio<AnyNode>): boolean {\n  const attrs = [\n    node.prop("tagName"),\n    node.attr("id"),\n    node.attr("class"),\n    node.attr("role"),\n  ]\n    .filter(Boolean)\n    .join(" ")\n    .toLowerCase();\n\n  const text = cleanText(node.text()).toLowerCase();\n\n  return (\n    /header|footer|nav|menu|filter|pagination|breadcrumb|modal|popup|layout|wrapper|sidebar|aside|catalog__aside|help-choosing/.test(\n      attrs,\n    ) ||\n    /личный кабинет|корзина|избранное|сравнение|сортировать|фильтр|затрудняетесь в выборе|помогут вам определиться/.test(\n      text,\n    )\n  );\n}\n\n/**\n * Измеряет повторяемость структуры узла среди соседей на нескольких уровнях.\n *\n * @param $ Документ Cheerio.\n * @param node Проверяемый кандидат.\n * @returns Максимальное число соседей с одинаковой структурной подписью и\n * саму подпись.\n */\nfunction repeatedAncestorGroupInfo(\n  $: cheerio.CheerioAPI,\n  node: cheerio.Cheerio<AnyNode>,\n): { siblingCount: number; signature: string } {\n  let current = node;\n  let best = {\n    siblingCount: 1,\n    signature: structuralSignature(node),\n  };\n\n  for (let depth = 0; depth < 6; depth += 1) {\n    const signature = structuralSignature(current);\n    if (signature) {\n      let count = 0;\n      const parent = current.parent();\n\n      parent.children().each((_, element) => {\n        if (structuralSignature($(element)) === signature) {\n          count += 1;\n        }\n      });\n\n      if (count > best.siblingCount) {\n        best = { siblingCount: count, signature };\n      }\n    }\n\n    const parent = current.parent();\n    if (!parent.length || parent.is("body") || parent.is("html")) break;\n    current = parent;\n  }\n\n  return best;\n}\n\n/**\n * Строит грубую подпись структуры товарного узла.\n *\n * @param node DOM-узел.\n * @returns Комбинацию тега, значимых классов и data-атрибутов либо пустую\n * строку для неинформативного узла.\n */\nfunction structuralSignature(node: cheerio.Cheerio<AnyNode>): string {\n  const tag = cleanText(node.prop("tagName")).toLowerCase();\n  const classTokens = cleanText(node.attr("class"))\n    .toLowerCase()\n    .split(/\\s+/)\n    .filter((token) =>\n      /product|goods|catalog|item|card|snippet|tile|offer/.test(token),\n    )\n    .sort()\n    .slice(0, 8)\n    .join(".");\n\n  const dataAttrs = [\n    "data-product-id",\n    "data-offer-id",\n    "data-sku",\n    "data-id",\n    "data-code",\n  ]\n    .filter((attr) => cleanText(node.attr(attr)))\n    .join("|");\n\n  if (!tag && !classTokens && !dataAttrs) return "";\n  return `${tag}|${classTokens}|${dataAttrs}`;\n}\n\n/**\n * Распознаёт консультационные, рекламные и вспомогательные блоки.\n *\n * @param node DOM-контейнер.\n * @param text Уже извлечённый текст контейнера.\n * @returns `true`, если атрибуты или формулировки указывают не на товар.\n */\nfunction looksLikeHelpOrMarketing(\n  node: cheerio.Cheerio<AnyNode>,\n  text: string,\n): boolean {\n  const attrs = [\n    node.prop("tagName"),\n    node.attr("id"),\n    node.attr("class"),\n    node.attr("role"),\n  ]\n    .filter(Boolean)\n    .join(" ")\n    .toLowerCase();\n\n  const normalizedText = cleanText(text).toLowerCase();\n\n  return (\n    /help-choosing|help|consult|banner|promo|sidebar|aside/.test(attrs) ||\n    /затрудняетесь в выборе|помогут вам определиться|материалы, которые помогут/.test(\n      normalizedText,\n    )\n  );\n}\n\n/**\n * Проверяет, является ли URL только числовой категорией каталога.\n *\n * @param value Абсолютная или относительная ссылка.\n * @returns `true` для маршрута вида `/catalog/1852/`, который недостаточно\n * конкретен для карточки товара.\n */\nfunction looksLikeCategoryOnlyUrl(value: string): boolean {\n  let pathname = "";\n  try {\n    pathname = new URL(value).pathname.toLowerCase();\n  } catch {\n    pathname = value.toLowerCase();\n  }\n\n  return /^\\/catalog\\/\\d+\\/?$/.test(pathname);\n}\n\n/**\n * Проверяет строку на пригодность в качестве названия товара.\n *\n * @param value Текстовый кандидат.\n * @returns `false` для слишком коротких, длинных, служебных и чисто ценовых\n * строк.\n */\nfunction looksLikeName(value: string): boolean {\n  const text = cleanText(value);\n  if (text.length < 4 || text.length > 260) return false;\n  if (BAD_TEXT_RE.test(text)) return false;\n  if (/^\\d[\\d\\s.,]*(?:₽|руб\\.?|р\\.?)?$/i.test(text)) return false;\n  return /[a-zа-яё]/i.test(text);\n}\n\n/**\n * Удаляет повторные кандидаты по URL или сочетанию имени и цены.\n *\n * @param candidates Кандидаты в порядке предпочтения.\n * @returns Первый экземпляр каждого уникального товара.\n */\nfunction dedupeCandidates(candidates: ProductCandidate[]): ProductCandidate[] {\n  const seen = new Set<string>();\n  const result: ProductCandidate[] = [];\n\n  for (const candidate of candidates) {\n    const key =\n      candidate.fields.product_url ||\n      `${candidate.fields.name ?? ""}|${candidate.fields.price ?? ""}`;\n\n    if (!key || seen.has(String(key))) continue;\n\n    seen.add(String(key));\n    result.push(candidate);\n  }\n\n  return result;\n}\n', 'src/utils.ts': 'import * as cheerio from "cheerio";\nimport type { AnyNode } from "domhandler";\n\n/**\n * Извлекает и нормализует домен из URL или похожей строки.\n *\n * @param value URL либо доменное имя.\n * @returns Домен в нижнем регистре без префикса `www.`. При неверном URL\n * применяется осторожная строковая очистка.\n */\nexport function normalizeHostname(value: string): string {\n  try {\n    return new URL(value).hostname.toLowerCase().replace(/^www\\./, "");\n  } catch {\n    return value\n      .toLowerCase()\n      .replace(/^https?:\\/\\//, "")\n      .replace(/^www\\./, "");\n  }\n}\n\n/**\n * Приводит произвольное значение к однострочному видимому тексту.\n *\n * @param value Значение из DOM, JSON или запроса.\n * @returns Строка без неразрывных пробелов и повторяющихся пробельных символов.\n */\nexport function cleanText(value: unknown): string {\n  if (value === null || value === undefined) return "";\n  return String(value)\n    .replace(/\\u00a0/g, " ")\n    .replace(/\\u202f/g, " ")\n    .replace(/\\u2009/g, " ")\n    .replace(/\\s+/g, " ")\n    .trim();\n}\n\n/**\n * Очищает название товара от приклеенного JSON виджета.\n *\n * @param value Сырой текст элемента карточки.\n * @returns Нормализованное название до первого похожего на JSON блока.\n */\nexport function cleanProductName(value: unknown): string {\n  const text = cleanText(value);\n  const widgetJsonStart = text.search(/\\s+\\{/);\n  return widgetJsonStart >= 0 ? text.slice(0, widgetJsonStart).trim() : text;\n}\n\n/**\n * Преобразует число или строку с валютой в положительную цену.\n *\n * @param value Сырое значение цены.\n * @returns Положительное число либо `null` для пустого, логического,\n * непарсируемого значения или цены «по запросу».\n */\nexport function coercePrice(value: unknown): number | null {\n  if (typeof value === "boolean" || value === null || value === undefined) {\n    return null;\n  }\n  if (typeof value === "number" && Number.isFinite(value)) {\n    return value > 0 ? value : null;\n  }\n  const text = cleanText(value)\n    .replace(/[₽]/g, " ")\n    .replace(/руб\\.?/gi, " ")\n    .replace(/р\\./gi, " ")\n    .replace(/\\bр\\b/gi, " ");\n  if (!text || /по запросу/i.test(text)) return null;\n  const match = text.match(\n    /\\d{1,3}(?:[ \\u00a0\\u202f\\u2009]\\d{3})+(?:[.,]\\d+)?|\\d+(?:[.,]\\d+)?/,\n  );\n  if (!match) return null;\n  const normalized = match[0]\n    .replace(/[ \\u00a0\\u202f\\u2009]/g, "")\n    .replace(",", ".")\n    .replace(/\\.$/, "");\n  const parsed = Number.parseFloat(normalized);\n  return Number.isFinite(parsed) && parsed > 0 ? parsed : null;\n}\n\n/**\n * Разрешает относительную ссылку относительно URL страницы.\n *\n * @param pageUrl Базовый URL документа.\n * @param value Сырое значение ссылки.\n * @returns Абсолютный URL, пустую строку для пустого значения либо исходный\n * текст, если конструктор URL его не принимает.\n */\nexport function absoluteUrl(pageUrl: string, value: unknown): string {\n  const text = cleanText(value);\n  if (!text) return "";\n  try {\n    return new URL(text, pageUrl).toString();\n  } catch {\n    return text;\n  }\n}\n\n/**\n * Возвращает первый непустой видимый текст по списку селекторов.\n *\n * @param $ Экземпляр Cheerio текущего документа.\n * @param root Корень поиска.\n * @param selectors Селекторы в порядке приоритета.\n * @returns Найденный текст либо пустую строку.\n */\nexport function firstText(\n  $: cheerio.CheerioAPI,\n  root: cheerio.Cheerio<AnyNode>,\n  selectors: string[],\n): string {\n  for (const selector of selectors) {\n    const value = ownOrDescendantText(root.find(selector).first());\n    if (value) return value;\n  }\n  return "";\n}\n\n/**\n * Возвращает первый непустой атрибут по списку селекторов.\n *\n * @param root Корень поиска Cheerio.\n * @param selectors Селекторы в порядке приоритета.\n * @param attrName Имя читаемого атрибута.\n * @returns Очищенное значение либо пустую строку.\n */\nexport function firstAttr(\n  root: cheerio.Cheerio<AnyNode>,\n  selectors: string[],\n  attrName: string,\n): string {\n  for (const selector of selectors) {\n    const value = cleanText(root.find(selector).first().attr(attrName));\n    if (value) return value;\n  }\n  return "";\n}\n\n/**\n * Извлекает видимый текст узла без содержимого служебных тегов.\n *\n * @param node Узел Cheerio.\n * @returns Нормализованный текст узла и его потомков.\n */\nexport function ownOrDescendantText(node: cheerio.Cheerio<AnyNode>): string {\n  if (node.length === 0) return "";\n  const visibleNode = node.clone();\n  visibleNode.find("script, style, noscript, noframes, template").remove();\n  return cleanText(visibleNode.text());\n}\n\n/**\n * Эвристически определяет, ведёт ли ссылка на отдельный товар.\n *\n * Служебные страницы корзины, поиска и справки отклоняются. Принимаются\n * характерные маршруты товара, параметры с идентификатором и достаточно\n * конкретные вложенные ссылки каталога.\n *\n * @param value Сырая ссылка.\n * @returns `true` только для ссылки, похожей на карточку товара.\n */\nexport function looksLikeProductHref(value: string): boolean {\n  const href = cleanText(value).toLowerCase();\n\n  if (!href || href === "/" || href.startsWith("#")) return false;\n  if (\n    href.startsWith("javascript:") ||\n    href.startsWith("mailto:") ||\n    href.startsWith("tel:")\n  ) {\n    return false;\n  }\n\n  if (\n    [\n      "/cart",\n      "/basket",\n      "/compare",\n      "/favorites",\n      "/favorite",\n      "/wishlist",\n      "/login",\n      "/auth",\n      "/search",\n      "/filter",\n      "/delivery",\n      "/payment",\n      "/help",\n      "/contacts",\n      "/about",\n      "/news",\n      "/blog",\n      "/reviews",\n      "/review",\n    ].some((part) => href.includes(part))\n  ) {\n    return false;\n  }\n\n  if (\n    [\n      "/product/",\n      "/products/",\n      "/goods/",\n      "/good/",\n      "/katalog/",\n      "/shop/",\n      "/item/",\n      "/items/",\n      "/sku/",\n      "/p/",\n      "/tovar/",\n      "/product-",\n      "/item-",\n      "/goods-",\n      "/offer/",\n      "/offers/",\n      "/books/",\n    ].some((part) => href.includes(part))\n  ) {\n    return true;\n  }\n\n  if (/[?&](product|sku|item|offer|good|book)[_-]?id=/.test(href)) {\n    return true;\n  }\n\n  // Важно: plain /catalog/1852/ — это часто категория, не товар.\n  // Товарные catalog-ссылки обычно содержат product/goods/item/tovar/sku/offer\n  // или имеют минимум два сегмента после /catalog/.\n  if (/\\/catalog\\/(?:product|goods|item|tovar|sku|offer)[^?#]*/.test(href)) {\n    return true;\n  }\n\n  if (\n    /\\/catalog\\/[^?#]+\\/(?:product|goods|item|tovar|sku|offer)[^?#]*/.test(href)\n  ) {\n    return true;\n  }\n\n  if (/\\/catalog\\/[^?#]+\\/[^?#]*\\d{3,}/.test(href)) {\n    return true;\n  }\n\n  return false;\n}\n\n/**\n * Разбивает строку CSS-селекторов, разделённых запятыми.\n *\n * @param value Исходная настройка или пустое значение.\n * @returns Непустые селекторы без окружающих пробелов.\n */\nexport function splitSelectorList(value: string | null | undefined): string[] {\n  if (!value) return [];\n  return value\n    .split(",")\n    .map((item) => item.trim())\n    .filter(Boolean);\n}\n\n/**\n * Удаляет поддерживаемые браузером псевдоэлементы из селектора Cheerio.\n *\n * `css-select` не умеет работать с `::before` и похожими конструкциями, а их\n * сгенерированное содержимое всё равно отсутствует в HTML.\n *\n * @param selector Исходный CSS-селектор.\n * @returns Селектор, пригодный для Cheerio.\n * @throws Error Если после очистки остался неизвестный псевдоэлемент.\n */\nexport function normalizeCssSelectorForCheerio(selector: string): string {\n  let value = selector.trim();\n\n  // css-select не поддерживает pseudo-elements.\n  // В extractor нам они не нужны: ::text/::attr обрабатываются выше,\n  // ::before/::after не содержат DOM-текста.\n  value = value.replace(\n    /::(?:before|after|marker|placeholder|selection|first-line|first-letter)\\b/gi,\n    "",\n  );\n\n  // На случай, если pseudo-element остался в середине выражения.\n  if (/::[a-z-]+/i.test(value)) {\n    throw new Error(`Unsupported pseudo-element selector: ${selector}`);\n  }\n\n  return value.trim();\n}\n', 'src/profiles.ts': 'import type { SupplierProfile } from "./contracts.js";\nimport { normalizeHostname } from "./utils.js";\n\n// Каталог хранит устойчивые селекторы карточек и характерные фрагменты\n// товарных URL для поставщиков, которым полезна предметная настройка.\nconst profiles: SupplierProfile[] = [\n  {\n    domain: "cnc.su",\n    displayName: "CNC",\n    itemSelector:\n      "td.wrapper_td, .wrapper_td, .catalog_item, .item_block, [data-entity=\'item\'], [class*=\'catalog-item\']",\n    productUrlHints: ["/catalog/"],\n  },\n  {\n    domain: "etm.ru",\n    displayName: "ETM",\n    itemSelector:\n      "[data-testid*=\'product\'], [data-test*=\'product\'], .product-card, [class*=\'product-card\'], [class*=\'ProductCard\']",\n    productUrlHints: ["/cat/", "/catalog/", "/goods/"],\n  },\n  {\n    domain: "hahn-kolb.ru",\n    displayName: "Hahn Kolb",\n    itemSelector:\n      ".product-layout, .product-grid > div, .product-list > div, .product, .product-item, .product-tile, .product-card, [class*=\'product-card\']",\n    productUrlHints: ["/katalog/", "/product/", "/catalog/"],\n  },\n  {\n    domain: "bigam.ru",\n    displayName: "Bigam",\n    itemSelector:\n      ".digi-product, .product-card, .catalog-item, [class*=\'product-card\'], [class*=\'productCard\']",\n    productUrlHints: ["/product/", "/catalog/"],\n  },\n  {\n    domain: "chipdip.ru",\n    displayName: "ChipDip",\n    itemSelector: "tr.with-hover, .itemlist tr",\n    productUrlHints: ["/product/"],\n  },\n  {\n    domain: "komus.ru",\n    displayName: "Komus",\n    itemSelector:\n      ".product-card-list, .product-card, [class*=\'product-card-list\']",\n    productUrlHints: ["/product/", "/catalog/"],\n  },\n  {\n    domain: "dns-shop.ru",\n    displayName: "DNS",\n    itemSelector:\n      ".catalog-product, [data-id=\'product\'], div:has(> .catalog-product__image):has(.catalog-product__name):has(.product-buy__price)",\n    fieldSelectors: {\n      name: "a.catalog-product__name::attr(title), a.catalog-product__name, .catalog-product__name::attr(title), .catalog-product__name, .catalog-product__image img::attr(alt)",\n      price:\n        ".product-buy__price, .catalog-product__buy .product-buy__price, [class*=\'price\']:not([class*=\'old\']):not([class*=\'credit\']):not([class*=\'month\'])",\n      product_url:\n        "a.catalog-product__name::attr(href), a.catalog-product__image-link::attr(href), a[href*=\'/product/\']::attr(href)",\n      image_url:\n        ".catalog-product__image img[data-src]::attr(data-src), .catalog-product__image img[src]::attr(src), .catalog-product__image source[srcset]::attr(srcset)",\n      delivery_time:\n        ".delivery-info-widget, .order-avail-wrap_main, .catalog-product__avails",\n      availability: ".order-avail-wrap_main, .catalog-product__avails",\n    },\n    productUrlHints: ["/product/"],\n  },\n  {\n    domain: "citilink.ru",\n    displayName: "Citilink",\n    itemSelector:\n      "[data-meta-name=\'SnippetProductVerticalLayout\'], [data-meta-name=\'ProductVerticalSnippet\'], [data-meta-product-id]",\n    productUrlHints: ["/product/"],\n  },\n  {\n    domain: "mvideo.ru",\n    displayName: "M.Video",\n    itemSelector: "a[mvid-product-card], [mvid-product-card]",\n    productUrlHints: ["/products/"],\n  },\n  {\n    domain: "oaopolimer.ru",\n    displayName: "OAO Polimer",\n    itemSelector:\n      ".product-item-container, li.product-item-big-card, .product-item-big-card",\n    productUrlHints: ["/catalog/"],\n  },\n  {\n    domain: "officemag.ru",\n    displayName: "Офисмаг",\n    itemSelector: "li.listItem.js-productListItem, .listItem",\n    productUrlHints: ["/catalog/goods/"],\n  },\n  {\n    domain: "lemanapro.ru",\n    displayName: "Lemana Pro",\n    itemSelector:\n      "article, [data-product-id], [data-qa*=\'product\'], [class*=\'product-card\'], [class*=\'productCard\']",\n    productUrlHints: ["/product/", "/p/"],\n  },\n  {\n    domain: "petrovich.ru",\n    displayName: "Petrovich",\n    itemSelector:\n      "[data-test*=\'product\'], [data-testid*=\'product\'], .product-card, [class*=\'product-card\'], [class*=\'ProductCard\']",\n    productUrlHints: ["/product/", "/catalog/"],\n  },\n  {\n    domain: "rs24.ru",\n    displayName: "RS24",\n    itemSelector:\n      ".search-results__item.js-product, .search-results__item, [class*=\'item-card\'], [class*=\'product-card\']",\n    productUrlHints: ["/product/"],\n  },\n  {\n    domain: "spb.tara.ru",\n    displayName: "SPB Tara",\n    itemSelector:\n      ".catalog-block-view__item, .catalog_item_wrapp, .catalog_item, .item_block, [class*=\'product\']",\n    productUrlHints: ["/catalog/", "/product/"],\n  },\n  {\n    domain: "tara.ru",\n    displayName: "Tara",\n    itemSelector:\n      ".catalog-block-view__item, .catalog_item_wrapp, .catalog_item, .item_block, [class*=\'product\']",\n    productUrlHints: ["/catalog/", "/product/"],\n  },\n  {\n    domain: "ximtek.ru",\n    displayName: "Ximtek",\n    itemSelector: ".product-item, [class*=\'product-item\']",\n    productUrlHints: ["/catalog/", "/product/"],\n  },\n  {\n    domain: "xn----7sbbnaebi2cxajxwo0b.xn--p1ai",\n    displayName: "Specodezhda Tula",\n    itemSelector: ".product-list-item",\n    productUrlHints: ["/catalog/"],\n  },\n  {\n    domain: "yandex.ru",\n    displayName: "Yandex Market",\n    itemSelector:\n      "[data-zone-name=\'productSnippet\'], [data-baobab-name=\'productSnippet\'], article, [class*=\'product-card\']",\n    productUrlHints: ["/product/", "/market/product", "/card/"],\n  },\n  {\n    domain: "market.yandex.ru",\n    displayName: "Yandex Market",\n    itemSelector:\n      "[data-zone-name=\'productSnippet\'], [data-baobab-name=\'productSnippet\'], article, [class*=\'product-card\']",\n    productUrlHints: ["/product/", "/market/product", "/card/"],\n  },\n  {\n    domain: "ozon.ru",\n    displayName: "Ozon",\n    itemSelector:\n      "[data-widget=\'tileV2\'], [data-widget=\'searchResultsV2\'] a[href*=\'/product/\']",\n    productUrlHints: ["/product/"],\n  },\n  {\n    domain: "vseinstrumenti.ru",\n    displayName: "Vseinstrumenti",\n    itemSelector:\n      "[data-qa=\'product-card\'], a[data-qa=\'product-name\'], a[data-qa=\'product-photo-click\']",\n    productUrlHints: ["/product/"],\n  },\n];\n\n/**\n * Подбирает профиль поставщика по домену страницы или исходного источника.\n *\n * @param url Фактический URL загруженной страницы.\n * @param sourceUrl Исходный URL источника, используемый как резерв.\n * @returns Профиль точного домена или его поддомена либо `undefined`.\n */\nexport function getSupplierProfile(\n  url: string,\n  sourceUrl?: string,\n): SupplierProfile | undefined {\n  const host = normalizeHostname(url || sourceUrl || "");\n  return profiles.find(\n    (profile) => host === profile.domain || host.endsWith(`.${profile.domain}`),\n  );\n}\n\n/**\n * Возвращает снимок всех встроенных профилей поставщиков.\n *\n * @returns Новый массив, который можно изменять без изменения каталога модуля.\n */\nexport function listSupplierProfiles(): SupplierProfile[] {\n  return [...profiles];\n}\n', 'src/blockDetection.ts': 'import { cleanText } from "./utils.js";\n\n/**\n * Определяет, содержит ли HTML признаки антибот-блокировки.\n *\n * Наличие убедительной товарной разметки имеет приоритет и отменяет\n * срабатывания на слова из скриптов. В остальных случаях проверяются CAPTCHA,\n * Cloudflare, Qrator, сообщения об ограничении доступа и характерные URL.\n *\n * @param html HTML загруженной страницы.\n * @param pageUrl Конечный URL после перенаправлений.\n * @returns Уникальные машинные маркеры блокировки; пустой список означает, что\n * блокировка не подтверждена.\n */\nexport function detectBlockedPage(html: string, pageUrl: string): string[] {\n  if (htmlHasProductEvidence(html)) {\n    return [];\n  }\n\n  const lowerHtml = html.toLowerCase();\n  const lowerUrl = pageUrl.toLowerCase();\n  const visible = cleanText(\n    html.replace(/<script\\b[^>]*>[\\s\\S]*?<\\/script>/gi, " "),\n  );\n  const visibleLower = visible.toLowerCase();\n  const markers: string[] = [];\n\n  const add = (marker: string, condition: boolean) => {\n    if (condition && !markers.includes(marker)) markers.push(marker);\n  };\n\n  add("captcha", /captcha|g-recaptcha|hcaptcha|sp_rotated_captcha/i.test(html));\n  add(\n    "cloudflare",\n    /cloudflare|cf-ray|cdn-cgi\\/challenge|__cf_chl/i.test(html),\n  );\n  add(\n    "qrator",\n    /qrator|qauth|xpvnsulc/i.test(html) || /xpvnsulc/.test(lowerUrl),\n  );\n  add(\n    "access_denied",\n    /доступ ограничен|access denied|forbidden|проблема с ip/i.test(\n      visibleLower,\n    ),\n  );\n  add(\n    "ozon_rr_redirect",\n    /ozon\\.ru/.test(lowerUrl) &&\n      /__rr=1/.test(lowerUrl) &&\n      !/tile-root|tile-clickable-element|href=["\']\\/product\\//i.test(html),\n  );\n  add(\n    "antibot_check",\n    /проверяем, что вы не робот|подозрительная активность|enable javascript/i.test(\n      visibleLower,\n    ),\n  );\n\n  return markers;\n}\n\n/**\n * Ищет явное сообщение о пустой поисковой выдаче.\n *\n * Проверяется только видимый текст без скриптов, причём наличие товарной\n * разметки отменяет результат, чтобы не принять служебную фразу за пустую\n * страницу.\n *\n * @param html HTML результатов поиска.\n * @returns Код найденной русской или английской формулировки либо `undefined`.\n */\nexport function detectEmptySearchResults(html: string): string | undefined {\n  const visible = cleanText(\n    html.replace(/<script\\b[^>]*>[\\s\\S]*?<\\/script>/gi, " "),\n  );\n  if (!visible) return undefined;\n  if (htmlHasProductEvidence(html)) return undefined;\n  const patterns: Array<[string, RegExp]> = [\n    [\n      "ru_no_products_matching_search",\n      /нет\\s+товаров[^.]{0,180}(?:поиск|запрос|критери)/i,\n    ],\n    [\n      "ru_products_not_found",\n      /(?:товары?|результаты?|предложения?)\\s+не\\s+найден/i,\n    ],\n    ["ru_nothing_found", /ничего\\s+не\\s+найдено/i],\n    [\n      "en_no_products_found",\n      /no\\s+(?:products?|items?|results?)\\s+(?:found|available|match)/i,\n    ],\n  ];\n  for (const [reason, pattern] of patterns) {\n    if (pattern.test(visible)) return reason;\n  }\n  return undefined;\n}\n\n/**\n * Быстро проверяет HTML на известные признаки карточек и товарных данных.\n *\n * @param html Исходный HTML.\n * @returns `true`, если найдены селекторы, микроразметка или URL-шаблоны,\n * характерные для товаров поддерживаемых магазинов.\n */\nexport function htmlHasProductEvidence(html: string): boolean {\n  return [\n    "digi-product",\n    "digi-product__price",\n    "mvid-product-card",\n    "current-price",\n    "itemlist",\n    "with-hover",\n    "price-main",\n    "product-item-container",\n    "product-item-big-card",\n    "product-list-item",\n    "search-results__item",\n    "js-product",\n    "data-retail-price",\n    "data-description",\n    \'data-entity="item"\',\n    \'data-testid="product\',\n    "data-testid=\'product",\n    \'data-test="product\',\n    \'data-zone-name="productSnippet"\',\n    \'data-baobab-name="productSnippet"\',\n    "price_value",\n    "js-productListItem",\n    "listItemBuy__price",\n    "product-card-list",\n    "product-card",\n    "product-title",\n    "v-product-price__value",\n    "catalog-product",\n    "catalog-block-view__item",\n    "item_block",\n    \'data-meta-name="SnippetProductVerticalLayout"\',\n    \'data-id="product"\',\n    // ozon\n    "tile-root",\n    "tile-clickable-element",\n    \'href="/product/\',\n    "href=\'/product/",\n    "searchResultsV2",\n    "tsHeadline500Medium",\n    "tsBody500Medium",\n\n    // vseinstrumenti\n    \'data-qa="product-name"\',\n    "data-qa=\'product-name\'",\n    \'data-qa="product-price-current"\',\n    "data-qa=\'product-price-current\'",\n    \'data-qa="product-add-to-cart-button"\',\n    "data-qa=\'product-add-to-cart-button\'",\n    \'data-qa="product-photo-click"\',\n    "data-qa=\'product-photo-click\'",\n    \'data-qa="product-availability"\',\n    "data-qa=\'product-availability\'",\n    "catalog__list-item snippet",\n    "promo-slider__item snippet",\n    "snippet__price",\n    "snippet-price__value",\n    "snippet__title",\n    "snippet__photo",\n    "/product-",\n    "itemListElement",\n    \'"@type":"Offer"\',\n    \'"priceCurrency":"RUB"\',\n  ].some((marker) => html.includes(marker));\n}\n', 'src/queryRelevance.ts': 'import type {\n  ProductRow,\n  QueryTag,\n  QueryRelevanceDiagnostics,\n} from "./contracts.js";\nimport { cleanText } from "./utils.js";\n\nconst BM25_K1 = 1.2;\nconst BM25_B = 0.75;\n// Порог пока сохраняется в диагностическом контракте; решение о принятии\n// строки принимает строгая проверка наличия всех обязательных тегов.\nconst DEFAULT_THRESHOLD = 0.16;\nconst MAX_REJECTED_DIAGNOSTICS = 24;\n\nconst stopWords = new Set([\n  "и",\n  "или",\n  "для",\n  "под",\n  "над",\n  "без",\n  "при",\n  "по",\n  "на",\n  "в",\n  "во",\n  "из",\n  "от",\n  "до",\n  "с",\n  "со",\n  "у",\n  "за",\n  "к",\n  "ко",\n  "а",\n  "но",\n  "же",\n  "ли",\n  "the",\n  "and",\n  "or",\n  "for",\n  "with",\n  "безнал",\n  "купить",\n  "цена",\n  "стоимость",\n  "товар",\n  "товары",\n  "заказать",\n]);\n\nconst measurementUnits = [\n  "л",\n  "литр",\n  "литра",\n  "литров",\n  "мл",\n  "мм",\n  "см",\n  "м",\n  "кг",\n  "г",\n  "в",\n  "v",\n  "вт",\n  "w",\n  "а",\n  "ah",\n  "ач",\n  "шт",\n];\n\n/**\n * Нормализует десятичный разделитель числовой части измерения.\n *\n * @param value Число в текстовом виде.\n * @returns То же значение с точкой вместо запятой.\n */\nfunction normalizeMeasureValue(value: string): string {\n  return value.replace(",", ".");\n}\n\n/**\n * Приводит распространённые единицы измерения к компактной форме.\n *\n * Например, «10 литров» становится `10л`, а «18 вольт» — `18в`. Это\n * позволяет одинаково сравнивать запрос и название товара.\n *\n * @param text Исходный запрос или текст карточки.\n * @returns Строка в нижнем регистре с нормализованными измерениями.\n */\nfunction normalizeMeasurements(text: string): string {\n  let normalized = text.toLocaleLowerCase("ru-RU");\n\n  normalized = normalized.replace(/ё/g, "е");\n\n  normalized = normalized.replace(\n    /(\\d+(?:[.,]\\d+)?)\\s*(литр(?:а|ов)?|л)\\b/gi,\n    (_match: string, value: string) => `${normalizeMeasureValue(value)}л`,\n  );\n\n  normalized = normalized.replace(\n    /(\\d+(?:[.,]\\d+)?)\\s*(миллиметр(?:а|ов)?|мм)\\b/gi,\n    (_match: string, value: string) => `${normalizeMeasureValue(value)}мм`,\n  );\n\n  normalized = normalized.replace(\n    /(\\d+(?:[.,]\\d+)?)\\s*(сантиметр(?:а|ов)?|см)\\b/gi,\n    (_match: string, value: string) => `${normalizeMeasureValue(value)}см`,\n  );\n\n  normalized = normalized.replace(\n    /(\\d+(?:[.,]\\d+)?)\\s*(килограмм(?:а|ов)?|кг)\\b/gi,\n    (_match: string, value: string) => `${normalizeMeasureValue(value)}кг`,\n  );\n\n  normalized = normalized.replace(\n    /(\\d+(?:[.,]\\d+)?)\\s*(ватт|ватта|ваттов|вт|w)\\b/gi,\n    (_match: string, value: string) => `${normalizeMeasureValue(value)}вт`,\n  );\n\n  normalized = normalized.replace(\n    /(\\d+(?:[.,]\\d+)?)\\s*(вольт|вольта|вольтов|в|v)\\b/gi,\n    (_match: string, value: string) => `${normalizeMeasureValue(value)}в`,\n  );\n\n  return normalized;\n}\n\n/**\n * Выполняет лёгкий русский стемминг отдельного токена.\n *\n * Числа и короткие слова сохраняются, у длинных слов удаляются наиболее\n * частые падежные и родовые окончания.\n *\n * @param token Нормализованный токен.\n * @returns Основа токена, пригодная для строгого сравнения.\n */\nfunction stemToken(token: string): string {\n  if (/^\\d/.test(token)) return token;\n\n  let normalized = token.replace(/ё/g, "е").toLocaleLowerCase("ru-RU");\n\n  if (normalized.length <= 5) return normalized;\n\n  normalized = normalized.replace(\n    /(иями|ями|ами|ого|его|ому|ему|ыми|ими|ая|яя|ое|ее|ые|ие|ый|ий|ой|ей|ую|юю|ам|ям|ах|ях|ов|ев|ом|ем|а|я|ы|и|у|ю|е)$/u,\n    "",\n  );\n\n  return normalized || token;\n}\n\n/**\n * Проверяет, является ли токен числом, объединённым с единицей измерения.\n *\n * @param token Проверяемый токен.\n * @returns `true` для значений вроде `10л` или `18в`.\n */\nfunction isMeasurementToken(token: string): boolean {\n  return new RegExp(\n    `^\\\\d+(?:[.,]\\\\d+)?(?:${measurementUnits.join("|")})$`,\n    "iu",\n  ).test(token);\n}\n\n/**\n * Проверяет, содержит ли токен только целое или дробное число.\n *\n * @param token Проверяемый токен.\n * @returns `true`, если в токене нет букв и посторонних символов.\n */\nfunction isNumberToken(token: string): boolean {\n  return /^\\d+(?:[.,]\\d+)?$/.test(token);\n}\n\n/**\n * Разбивает текст товара на значимые нормализованные токены.\n *\n * Стоп-слова удаляются, единицы измерения унифицируются, а слова приводятся к\n * основам.\n *\n * @param text Анализируемый текст.\n * @returns Последовательность токенов для расчёта релевантности.\n */\nfunction tokenize(text: string): string[] {\n  const normalized = normalizeMeasurements(cleanText(text))\n    .replace(/[^\\p{L}\\p{N}.,/-]+/gu, " ")\n    .replace(/\\s+/g, " ")\n    .trim();\n\n  const rawTokens =\n    normalized.match(/[\\p{L}\\p{N}]+(?:[.,/-][\\p{L}\\p{N}]+)*/gu) ?? [];\n\n  return rawTokens\n    .map((token) => token.trim().toLocaleLowerCase("ru-RU"))\n    .filter(Boolean)\n    .filter((token) => !stopWords.has(token))\n    .filter((token) => token.length >= 2 || isNumberToken(token))\n    .map(stemToken)\n    .filter(Boolean);\n}\n\n/**\n * Преобразует пользовательский запрос в обязательные смысловые теги.\n *\n * @param query Поисковый запрос или пустое значение.\n * @returns Уникальные слова, числа и измерения с исходной и нормализованной\n * формой.\n */\nexport function splitQueryIntoTags(\n  query: string | null | undefined,\n): QueryTag[] {\n  const raw = normalizeMeasurements(cleanText(query ?? ""));\n  if (!raw) return [];\n\n  const rawTokens = raw.match(/[\\p{L}\\p{N}]+(?:[.,/-][\\p{L}\\p{N}]+)*/gu) ?? [];\n  const tags: QueryTag[] = [];\n  const seen = new Set<string>();\n\n  for (const rawToken of rawTokens) {\n    const value = rawToken.trim().toLocaleLowerCase("ru-RU");\n    if (!value || stopWords.has(value)) continue;\n\n    const kind: QueryTag["kind"] = isMeasurementToken(value)\n      ? "measure"\n      : isNumberToken(value)\n        ? "number"\n        : "word";\n\n    if (kind === "word" && value.length < 3) continue;\n\n    const normalized = stemToken(value);\n    if (!normalized || seen.has(normalized)) continue;\n\n    seen.add(normalized);\n\n    tags.push({\n      value,\n      normalized,\n      kind,\n      required: true,\n    });\n  }\n\n  return tags;\n}\n\n/**\n * Собирает текстовые поля строки, участвующие в проверке релевантности.\n *\n * @param row Извлечённая товарная строка.\n * @returns Название, наличие и доставка, объединённые в одну строку.\n */\nfunction rowText(row: ProductRow): string {\n  return [\n    row.name,\n    row.availability,\n    row.delivery_time,\n    row.in_stock === false ? "нет в наличии" : "",\n  ]\n    .filter(Boolean)\n    .join(" ");\n}\n\n/**\n * Подсчитывает частоту каждого токена в документе.\n *\n * @param tokens Токены товарной строки.\n * @returns Карта `токен -> число вхождений`.\n */\nfunction termFrequency(tokens: string[]): Map<string, number> {\n  const map = new Map<string, number>();\n\n  for (const token of tokens) {\n    map.set(token, (map.get(token) ?? 0) + 1);\n  }\n\n  return map;\n}\n\n/**\n * Рассчитывает BM25-оценку товарной строки относительно запроса.\n *\n * @param params Токены запроса и документа, частоты по корпусу и средняя\n * длина документа.\n * @returns Ненормализованная BM25-оценка; ноль для пустых входных данных.\n */\nfunction bm25Score(params: {\n  queryTokens: string[];\n  documentTokens: string[];\n  documentFrequency: Map<string, number>;\n  averageDocumentLength: number;\n  documentCount: number;\n}): number {\n  const {\n    queryTokens,\n    documentTokens,\n    documentFrequency,\n    averageDocumentLength,\n    documentCount,\n  } = params;\n\n  if (!queryTokens.length || !documentTokens.length) return 0;\n\n  const tf = termFrequency(documentTokens);\n  const documentLength = documentTokens.length;\n  let score = 0;\n\n  for (const token of queryTokens) {\n    const frequency = tf.get(token) ?? 0;\n    if (frequency <= 0) continue;\n\n    const df = documentFrequency.get(token) ?? 0;\n    const idf = Math.log(1 + (documentCount - df + 0.5) / (df + 0.5));\n    const denominator =\n      frequency +\n      BM25_K1 *\n        (1 -\n          BM25_B +\n          BM25_B * (documentLength / Math.max(1, averageDocumentLength)));\n\n    score += idf * ((frequency * (BM25_K1 + 1)) / denominator);\n  }\n\n  return score;\n}\n\n/**\n * Отбрасывает товары, в названии и сопутствующих полях которых нет тегов запроса.\n *\n * Сейчас принятие строго требует присутствия каждого тега. BM25-оценка и\n * `threshold` сохраняются в диагностике для анализа качества, но порог не\n * заменяет проверку обязательных тегов.\n *\n * @param rows Извлечённые товарные строки.\n * @param query Пользовательский запрос.\n * @param threshold Диагностический порог релевантности.\n * @returns Принятые строки, теги запроса и ограниченный список причин отказа.\n */\nexport function filterRowsByQueryRelevance(\n  rows: ProductRow[],\n  query: string | null | undefined,\n  threshold = DEFAULT_THRESHOLD,\n): {\n  rows: ProductRow[];\n  queryTags: QueryTag[];\n  diagnostics: QueryRelevanceDiagnostics;\n} {\n  const queryTags = splitQueryIntoTags(query);\n  const queryTokens = queryTags.map((tag) => tag.normalized);\n\n  if (!rows.length || !queryTokens.length) {\n    return {\n      rows,\n      queryTags,\n      diagnostics: {\n        enabled: false,\n        queryTags,\n        beforeCount: rows.length,\n        afterCount: rows.length,\n        rejectedCount: 0,\n        threshold,\n        rejected: [],\n      },\n    };\n  }\n\n  const documents = rows.map((row) => tokenize(rowText(row)));\n  const averageDocumentLength =\n    documents.reduce((sum, tokens) => sum + tokens.length, 0) /\n    Math.max(1, documents.length);\n\n  const documentFrequency = new Map<string, number>();\n\n  for (const token of new Set(queryTokens)) {\n    documentFrequency.set(\n      token,\n      documents.filter((documentTokens) => documentTokens.includes(token))\n        .length,\n    );\n  }\n\n  const accepted: ProductRow[] = [];\n  const rejected: QueryRelevanceDiagnostics["rejected"] = [];\n\n  rows.forEach((row, index) => {\n    const documentTokens = documents[index] ?? [];\n    const tokenSet = new Set(documentTokens);\n\n    const missingTags = queryTags.filter(\n      (tag) => !tokenSet.has(tag.normalized),\n    );\n\n    const score = bm25Score({\n      queryTokens,\n      documentTokens,\n      documentFrequency,\n      averageDocumentLength,\n      documentCount: rows.length,\n    });\n\n    const normalizedScore = score / Math.max(1, queryTokens.length);\n\n    if (missingTags.length === 0) {\n      accepted.push(row);\n      return;\n    }\n\n    if (rejected.length < MAX_REJECTED_DIAGNOSTICS) {\n      rejected.push({\n        name: row.name,\n        score: Number(normalizedScore.toFixed(4)),\n        reason: `missing_query_tags:${missingTags\n          .map((tag) => tag.value)\n          .join(",")}`,\n      });\n    }\n  });\n\n  return {\n    rows: accepted,\n    queryTags,\n    diagnostics: {\n      enabled: true,\n      queryTags,\n      beforeCount: rows.length,\n      afterCount: accepted.length,\n      rejectedCount: rows.length - accepted.length,\n      threshold,\n      rejected,\n    },\n  };\n}\n', 'src/contracts.ts': 'export interface SourceInput {\r\n  url: string;\r\n  name?: string;\r\n  itemSelector?: string | null;\r\n  fieldSelectors?: Record<string, string>;\r\n  searchUrlTemplate?: string | null;\r\n  searchEntryUrlTemplate?: string | null;\r\n  searchInputSelector?: string | null;\r\n  searchSubmitSelector?: string | null;\r\n  searchSubmitAction?: string | null;\r\n}\r\n\r\nexport interface ExtractProductsRequest {\r\n  html: string;\r\n  pageUrl: string;\r\n  source?: SourceInput;\r\n  query?: string | null;\r\n  itemSelector?: string | null;\r\n  fieldSelectors?: Record<string, string>;\r\n  limit?: number;\r\n\r\n  /**\r\n   * Для обычного collection можно оставить false.\r\n   * Для onboarding true: вернуть topCandidates и частичные поля.\r\n   */\r\n  includeCandidates?: boolean;\r\n}\r\n\r\nexport interface ProductRow {\r\n  name: string;\r\n  price: number;\r\n  product_url: string;\r\n  delivery_time?: string;\r\n  rating?: number;\r\n  in_stock?: boolean;\r\n  image_url?: string;\r\n  availability?: string;\r\n}\r\n\r\nexport interface SearchProduct {\r\n  name: string;\r\n  price: number;\r\n  url: string;\r\n  delivery_time?: string;\r\n  rating?: number;\r\n  in_stock?: boolean;\r\n}\r\n\r\nexport interface ExtractionDiagnostics {\r\n  blockedMarkers: string[];\r\n  emptyReason?: string;\r\n  candidateCount: number;\r\n  rowCount: number;\r\n  rejectedRows: Record<string, number>;\r\n  debug: string[];\r\n  candidates?: CandidateDiagnostics;\r\n}\r\n\r\nexport interface QueryTag {\r\n  value: string;\r\n  normalized: string;\r\n  kind: "word" | "measure" | "number";\r\n  required: boolean;\r\n}\r\n\r\nexport interface QueryRelevanceDiagnostics {\r\n  enabled: boolean;\r\n  queryTags: QueryTag[];\r\n  beforeCount: number;\r\n  afterCount: number;\r\n  rejectedCount: number;\r\n  threshold: number;\r\n  rejected: Array<{\r\n    name: string;\r\n    score: number;\r\n    reason: string;\r\n  }>;\r\n}\r\n\r\nexport interface ExtractProductsResult {\r\n  status: "ok" | "empty" | "blocked";\r\n  route: string;\r\n  pageUrl: string;\r\n  queryTags?: QueryTag[];\r\n  rows: ProductRow[];\r\n  products: SearchProduct[];\r\n  diagnostics: ExtractionDiagnostics & {\r\n    queryRelevance?: QueryRelevanceDiagnostics;\r\n  };\r\n}\r\n\r\nexport interface SupplierProfile {\r\n  domain: string;\r\n  displayName: string;\r\n  itemSelector?: string;\r\n  fieldSelectors?: Record<string, string>;\r\n  productUrlHints?: string[];\r\n}\r\n\r\nexport interface SourceDraft {\r\n  url: string;\r\n  name: string;\r\n  is_available: boolean;\r\n  item_selector?: string | null;\r\n  field_selectors?: Record<string, string>;\r\n  search_url_template?: string | null;\r\n  search_entry_url_template?: string | null;\r\n  search_input_selector?: string | null;\r\n  search_submit_selector?: string | null;\r\n  search_submit_action?: string | null;\r\n}\r\n\r\nexport interface OnboardingRequest extends ExtractProductsRequest {\r\n  catalogUrl?: string | null;\r\n  sampleQuery?: string | null;\r\n  searchUrlTemplate?: string | null;\r\n}\r\n\r\nexport interface OnboardingResult {\r\n  status: "ready_for_save" | "manual_required" | "blocked" | "empty";\r\n  diagnosticStatus: number;\r\n  message: string;\r\n  route: string;\r\n  pageUrl: string;\r\n  sourceDraft: SourceDraft | null;\r\n  fields: Record<string, unknown>;\r\n  rows: ProductRow[];\r\n  diagnostics: ExtractionDiagnostics & {\r\n    selectedRowIndex?: number;\r\n  };\r\n}\r\n\r\nexport interface ProductCandidateFieldDiagnostics {\r\n  value?: unknown;\r\n  confidence: number;\r\n  reason: string;\r\n  raw: string[];\r\n}\r\n\r\nexport interface ProductCandidate {\r\n  index: number;\r\n  selector: string;\r\n  htmlPreview: string;\r\n  textPreview: string;\r\n  score: number;\r\n  reasons: string[];\r\n  fields: Partial<ProductRow>;\r\n  fieldDiagnostics: Record<string, ProductCandidateFieldDiagnostics>;\r\n  missingFields: string[];\r\n  rejectReason?: string;\r\n}\r\n\r\nexport interface CandidateDiagnostics {\r\n  candidateCount: number;\r\n  validRowCount: number;\r\n  rejectedCandidateCount: number;\r\n  rejectReasons: Record<string, number>;\r\n  topCandidates: ProductCandidate[];\r\n}\r\n'}



# Compiled original sources + exact installed npm dependencies; no runtime npm lookup.



# Optional historical baselines (not selected by default).
IN_STOCK_FALSE = ("outofstock", "out of stock", "unavailable", "sold out", "нет в наличии", "отсутств", "законч")
IN_STOCK_TRUE = ("instock", "in stock", "в наличии", "available", "есть", "доступ")


# ---- B1 core copied from parse_logs_to_excel_v2.py ----
def _iter_jsonld_nodes_reference(data):
    if isinstance(data, list):
        for x in data:
            yield from _iter_jsonld_nodes_reference(x)
    elif isinstance(data, dict):
        if "@graph" in data:
            yield from _iter_jsonld_nodes_reference(data["@graph"])
        yield data


def _node_to_product_reference(node: dict) -> dict:
    offers = node.get("offers") or node.get("Offer") or {}
    if isinstance(offers, list):
        offers = offers[0] if offers else {}
    price = None
    availability = None
    for src in (offers, node):
        if isinstance(src, dict):
            p = src.get("price") or src.get("lowPrice")
            if p is not None and price is None:
                try:
                    price = float(str(p).replace(",", ".").replace(" ", ""))
                except (ValueError, TypeError):
                    pass
            a = src.get("availability")
            if a and availability is None:
                availability = str(a)
    return {
        "name": node.get("name"),
        "url": node.get("url") or (offers.get("url") if isinstance(offers, dict) else None),
        "price": price,
        "availability": availability,
    }


def extract_jsonld_products_reference(html_text: str) -> list[dict]:
    products: list[dict] = []
    for m in JSONLD_RE.finditer(html_text):
        raw = m.group(1).strip()
        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            try:
                data = json.loads(re.sub(r",\s*([}\]])", r"\1", raw))
            except json.JSONDecodeError:
                continue
        for node in _iter_jsonld_nodes_reference(data):
            if not isinstance(node, dict):
                continue
            t = node.get("@type")
            types = [t] if isinstance(t, str) else (t or [])
            if not any("product" in str(x).lower() for x in types):
                continue
            products.append(_node_to_product_reference(node))
    return products

def parse_in_stock(value: object) -> bool | None:
    text = _clean_text(value).casefold().replace("https://schema.org/", "").replace("http://schema.org/", "")
    if not text:
        return None
    if any(x in text for x in IN_STOCK_FALSE):
        return False
    if any(x in text for x in IN_STOCK_TRUE):
        return True
    if text in {"0", "false", "no", "нет"}:
        return False
    if text in {"1", "true", "yes", "да"}:
        return True
    return None


def first_json_text(value: object) -> str:
    if isinstance(value, str):
        return value
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return str(value)
    if isinstance(value, list):
        for item in value:
            text = first_json_text(item)
            if text:
                return text
    if isinstance(value, dict):
        for key in ("@id", "url", "name", "text", "value"):
            text = first_json_text(value.get(key))
            if text:
                return text
    return ""


def is_product_type(raw_type: object) -> bool:
    values = raw_type if isinstance(raw_type, list) else [raw_type]
    for value in values:
        text = _clean_text(value).casefold().rstrip("/#")
        if text == "product" or text.endswith("/product") or text.endswith(":product"):
            return True
    return False


def walk_jsonld_products(value: object):
    if isinstance(value, list):
        for item in value:
            yield from walk_jsonld_products(item)
        return
    if not isinstance(value, dict):
        return
    if is_product_type(value.get("@type")):
        yield value
    for nested in value.values():
        if isinstance(nested, (dict, list)):
            yield from walk_jsonld_products(nested)


def select_priced_offer(value: object) -> dict | None:
    if isinstance(value, dict):
        if any(k in value for k in ("price", "lowPrice", "highPrice")):
            return value
        for key in ("offers", "priceSpecification"):
            found = select_priced_offer(value.get(key))
            if found is not None:
                return found
    elif isinstance(value, list):
        for item in value:
            found = select_priced_offer(item)
            if found is not None:
                return found
    return None


def document_base_url(soup: BeautifulSoup, fallback: str = "") -> str:
    """Choose a deterministic base URL for resolving relative product links.

    Prefer an explicit <base>; then a page canonical/og:url; finally the debug-log
    page/source URL. This compensates for older artifacts whose debug log did not
    persist the final search-page URL.
    """
    base = soup.select_one("base[href]")
    if base and _clean_text(base.get("href")):
        return urljoin(fallback, _clean_text(base.get("href")))
    for selector, attr in (
        ('link[rel="canonical"][href]', "href"),
        ('meta[property="og:url"][content]', "content"),
    ):
        node = soup.select_one(selector)
        if node and _clean_text(node.get(attr)):
            return urljoin(fallback, _clean_text(node.get(attr)))
    return fallback


def document_canonical_url(soup: BeautifulSoup, fallback: str = "") -> str:
    for selector, attr in (
        ('link[rel="canonical"][href]', "href"),
        ('meta[property="og:url"][content]', "content"),
    ):
        node = soup.select_one(selector)
        if node:
            value = _clean_text(node.get(attr))
            if value:
                return _resolve_norm_url(value, fallback)
    return _norm_url(fallback)


def extract_b1_jsonld(html_text: str, soup: BeautifulSoup, page_url: str) -> list[dict]:
    """B1: reference JSON-LD parser + only URL resolution/output adaptation."""
    base_url = document_base_url(soup, page_url)
    products = extract_jsonld_products_reference(html_text)
    canonical = document_canonical_url(soup, page_url)
    rows = []
    for p in products:
        name = _clean_text(p.get("name")) or None
        price = p.get("price")
        raw_url = _clean_text(p.get("url"))
        product_url = _norm_url(urljoin(base_url, raw_url)) if raw_url else None
        if not product_url and len(products) == 1:
            product_url = canonical
        if not product_url:
            continue
        if not name and price is None:
            continue
        rows.append({
            "product_url": product_url,
            "pred_name": name,
            "pred_price": price,
            "pred_in_stock": parse_in_stock(p.get("availability")),
            "extract_notes": "jsonld_product_reference_core",
        })
    return rows


B2_CARD_SELECTORS = (
    # schema/data attributes
    '[itemtype*="Product"]',
    '[itemscope][itemtype*="schema.org/Product"]',
    '[data-product-id]', '[data-meta-product-id]', '[data-product]', '[data-offer-id]',
    '[data-sku]', '[data-id="product"]', '[data-entity="item"]',
    '[data-testid*="product" i]', '[data-test*="product" i]', '[data-qa*="product" i]',
    # repeated generic ecommerce cards
    '.catalog__list-item.snippet', '.promo-slider__item.snippet', '.snippet', '.snippet__content',
    '[class~="snippet"]', '[class*="snippet" i]',
    '.product-card-list', '.product-card', '[class*="product-card" i]', '[class*="product_card" i]',
    '.catalog-product', '[class*="catalog-product" i]', '.catalog-item', '.catalog_item',
    '.product-item', '.product_item', '.goods-item', '.goods_item', '.item-card', '.card-item',
    '[class*="tile" i]',
    # broad structural containers are intentionally last
    'article', 'li', 'tr',
)

B2_NAME_SELECTORS = (
    '[itemprop="name"]',
    '.snippet__title', '.snippet-title',
    '.product-card__name', '.product-card__title',
    '[class*="product-name" i]', '[class*="product-title" i]',
    '[class*="title" i]', '[class*="name" i]',
    '[data-qa*="name" i]', '[data-testid*="name" i]',
    'h1', 'h2', 'h3', 'a[title]', 'img[alt]',
)

B2_PRICE_SELECTORS = (
    'meta[itemprop="price"]', '[content][itemprop="price"]', '[itemprop="price"]',
    '[data-price]', '[data-retail-price]', '[data-meta-price]',
    '.snippet-price__value', '.product-card__price-current', '.current-price', '.price-main',
    '[class*="price" i]:not([class*="old" i]):not([class*="credit" i]):not([class*="month" i]):not([class*="installment" i])',
    '[data-qa*="price" i]', '[data-testid*="price" i]',
)

B2_EXCLUDED_HREF_PARTS = (
    '/cart', '/basket', '/compare', '/favorites', '/favorite', '/wishlist', '/login', '/auth',
    '/search', '/filter', '/delivery', '/payment', '/help', '/contacts', '/about', '/news',
    '/blog', '/reviews', '/review',
)
B2_PRODUCT_HREF_PARTS = (
    '/product/', '/products/', '/goods/', '/good/', '/katalog/', '/shop/', '/item/', '/items/',
    '/sku/', '/p/', '/tovar/', '/product-', '/item-', '/goods-', '/offer/', '/offers/', '/books/',
)


def looks_like_product_href(value: object) -> bool:
    href = _clean_text(value).casefold()
    if not href or href == '/' or href.startswith(('#', 'javascript:', 'mailto:', 'tel:')):
        return False
    if any(part in href for part in B2_EXCLUDED_HREF_PARTS):
        return False
    if any(part in href for part in B2_PRODUCT_HREF_PARTS):
        return True
    if re.search(r'[?&](?:product|sku|item|offer|good|book)[_-]?id=', href):
        return True
    if re.search(r'/catalog/(?:product|goods|item|tovar|sku|offer)[^?#]*', href):
        return True
    if re.search(r'/catalog/[^?#]+/(?:product|goods|item|tovar|sku|offer)[^?#]*', href):
        return True
    if re.search(r'/catalog/[^?#]+/[^?#]*\d{3,}', href):
        return True
    return False


def _tag_value(node, attrs=("content", "data-price", "data-retail-price", "data-meta-price", "value")) -> str:
    for attr in attrs:
        value = node.get(attr) if hasattr(node, "get") else None
        if _clean_text(value):
            return _clean_text(value)
    return _clean_text(node.get_text(" ", strip=True)) if hasattr(node, "get_text") else ""


def _first_selector_value(root, selectors, *, attr_names=("content", "value")) -> tuple[str, str]:
    for selector in selectors:
        try:
            nodes = root.select(selector)
        except Exception:
            continue
        for node in nodes:
            value = ""
            for attr in attr_names:
                value = _clean_text(node.get(attr))
                if value:
                    break
            if not value:
                value = _clean_text(node.get_text(" ", strip=True))
            if value:
                return value, selector
    return "", ""


def _collect_unique_nodes(soup: BeautifulSoup, selectors, limit: int) -> list:
    nodes = []
    seen = set()
    for selector in selectors:
        try:
            found = soup.select(selector)
        except Exception:
            continue
        for node in found:
            ident = id(node)
            if ident in seen:
                continue
            seen.add(ident)
            nodes.append(node)
            if len(nodes) >= limit:
                return nodes
    return nodes


def _b2_candidate_nodes(soup: BeautifulSoup) -> tuple[list, str]:
    # Collect from the whole fixed selector set rather than stopping at the first
    # selector. This avoids a stray matching widget suppressing real product cards.
    nodes = _collect_unique_nodes(soup, B2_CARD_SELECTORS, MAX_CANDIDATES_PER_HTML)
    if nodes:
        return nodes, "generic_card_union"

    # Product-link fallback: climb a few levels and retain the nearest ancestor
    # that carries a price signal. Link classification is source-agnostic.
    links = [a for a in soup.find_all("a", href=True) if looks_like_product_href(a.get("href"))]
    if links:
        nodes = []
        seen = set()
        for link in links[:MAX_CANDIDATES_PER_HTML]:
            candidate = link
            current = link
            for _ in range(5):
                parent = getattr(current, "parent", None)
                if parent is None or not hasattr(parent, "select"):
                    break
                if any(parent.select_one(sel) is not None for sel in B2_PRICE_SELECTORS):
                    candidate = parent
                    break
                current = parent
            ident = id(candidate)
            if ident not in seen:
                seen.add(ident)
                nodes.append(candidate)
        if nodes:
            return nodes[:MAX_CANDIDATES_PER_HTML], "product_link_parent"

    # Product-detail fallback: one document-level candidate only when a title and a price signal exist.
    has_name = bool(soup.select_one('[itemprop="name"], h1, title'))
    has_price = any(soup.select_one(sel) is not None for sel in B2_PRICE_SELECTORS)
    if has_name and has_price and soup.body is not None:
        return [soup.body], "document_detail_fallback"
    return [], "none"


def _extract_b2_name(root) -> tuple[str, str]:
    for selector in B2_NAME_SELECTORS:
        try:
            nodes = root.select(selector)
        except Exception:
            nodes = []
        for node in nodes:
            value = _first_nonempty(
                node.get("content"), node.get("title"), node.get("aria-label"), node.get("alt"),
                node.get_text(" ", strip=True),
            )
            # Mirrors the production cleanProductName idea: discard appended widget JSON.
            value = re.split(r"\s+\{", value, maxsplit=1)[0].strip()
            value = re.sub(r"^(?:открыть карточку товара|купить|в корзину|подробнее)\s+", "", value, flags=re.I)
            if 3 <= len(value) <= 240:
                return value, selector

    # Production also falls back to the selected product-link label. Keep this
    # generic: only links that pass looks_like_product_href are considered.
    try:
        links = root.find_all("a", href=True)
    except Exception:
        links = []
    for link in links:
        if not looks_like_product_href(link.get("href")):
            continue
        value = _first_nonempty(link.get_text(" ", strip=True), link.get("title"), link.get("aria-label"))
        value = re.split(r"\s+\{", value, maxsplit=1)[0].strip()
        if 3 <= len(value) <= 240:
            return value, "product_link_text"
    return "", ""


def _extract_b2_price(root) -> tuple[float | None, str]:
    for selector in B2_PRICE_SELECTORS:
        try:
            nodes = root.select(selector)
        except Exception:
            nodes = []
        for node in nodes:
            classes = " ".join(node.get("class") or []).casefold()
            if any(bad in classes for bad in ("old", "credit", "month", "installment")):
                continue
            raw = _tag_value(node)
            if re.search(r"артикул|article|код товара", raw, flags=re.I):
                continue
            price = parse_price(raw)
            if price is not None:
                return price, selector
    return None, ""


def _extract_b2_url(root, soup: BeautifulSoup, base_url: str, card_mode: str) -> tuple[str, str]:
    # Explicit schema.org URL has highest priority.
    try:
        node = root.select_one('a[itemprop="url"][href]')
    except Exception:
        node = None
    if node and _clean_text(node.get("href")):
        return _resolve_norm_url(node.get("href"), base_url), 'a[itemprop="url"]'

    best = None
    best_score = -1
    try:
        links = root.find_all("a", href=True)
    except Exception:
        links = []
    for link in links:
        href = _clean_text(link.get("href"))
        if not looks_like_product_href(href):
            continue
        label = _first_nonempty(link.get_text(" ", strip=True), link.get("title"), link.get("aria-label"))
        cls = " ".join(link.get("class") or []).casefold()
        score = 1 + int(len(label) >= 8) * 2 + int(bool(re.search(r"title|name|product|goods|item", cls))) * 2
        if score > best_score:
            best = link
            best_score = score
    if best is not None:
        return _resolve_norm_url(best.get("href"), base_url), "best_product_link"

    if card_mode == "document_detail_fallback":
        url = document_canonical_url(soup, base_url)
        if url:
            return url, "canonical_or_page_url"
    return "", ""


def _extract_b2_stock(root) -> bool | None:
    node = root.select_one('[itemprop="availability"]') if hasattr(root, "select_one") else None
    if node:
        value = _first_nonempty(node.get("href"), node.get("content"), node.get_text(" ", strip=True))
        parsed = parse_in_stock(value)
        if parsed is not None:
            return parsed
    text = _clean_text(root.get_text(" ", strip=True)) if hasattr(root, "get_text") else ""
    return parse_in_stock(text)


def extract_b2_css(html_text: str, soup: BeautifulSoup, page_url: str) -> list[dict]:
    del html_text  # B2 intentionally uses only DOM/CSS heuristics.
    base_url = document_base_url(soup, page_url)
    nodes, card_mode = _b2_candidate_nodes(soup)
    rows = []
    for node in nodes:
        name, name_sel = _extract_b2_name(node)
        price, price_sel = _extract_b2_price(node)
        product_url, url_sel = _extract_b2_url(node, soup, base_url, card_mode)
        if not product_url:
            continue
        if not name and price is None:
            continue
        rows.append({
            "product_url": product_url,
            "pred_name": name or None,
            "pred_price": price,
            "pred_in_stock": _extract_b2_stock(node),
            "extract_notes": f"card={card_mode};name={name_sel or '-'};price={price_sel or '-'};url={url_sel or '-'}",
        })
    return rows


B3_PRICE_RE = re.compile(
    r"(?<!\d)(\d{1,3}(?:[ \u00a0\u202f\u2009]\d{3})+(?:[.,]\d{1,2})?|\d{1,9}(?:[.,]\d{1,2})?)\s*(?:₽|руб(?:\.|лей|ля)?|р\.)",
    re.I,
)


def extract_b3_regex(html_text: str, soup: BeautifulSoup, page_url: str) -> list[dict]:
    del html_text
    h1 = soup.find("h1")
    title = soup.find("title")
    name = _first_nonempty(
        h1.get_text(" ", strip=True) if h1 else "",
        title.get_text(" ", strip=True) if title else "",
    )
    visible_text = _clean_text(soup.get_text(" ", strip=True))
    m = B3_PRICE_RE.search(visible_text)
    price = parse_price(m.group(1)) if m else None
    product_url = document_canonical_url(soup, page_url)
    if not product_url or (not name and price is None):
        return []
    return [{
        "product_url": product_url,
        "pred_name": name or None,
        "pred_price": price,
        "pred_in_stock": parse_in_stock(visible_text),
        "extract_notes": "document_h1_or_title+currency_regex",
    }]


EXTRACTORS = {
    "b1_jsonld": extract_b1_jsonld,
    "b2_css": extract_b2_css,
    "b3_regex": extract_b3_regex,
}
# Frozen original parser payloads are stored as benchmark assets instead of notebook literals.
ORIGINAL_PARSER_ASSET_DIR = BENCHMARK_ROOT / "config" / "original_fast_renderer"
ORIGINAL_PARSER_META = json.loads((ORIGINAL_PARSER_ASSET_DIR / "provenance.json").read_text(encoding="utf-8"))



## 5. Run identical extraction over the selected HTML corpora

Every enabled baseline receives exactly the same saved HTML file within a dataset. No GroundTruth information is used during extraction or candidate deduplication.


In [7]:
def candidate_completeness(row: dict) -> tuple[int, int, int]:
    return (int(bool(_clean_text(row.get("pred_name")))),
            int(parse_price(row.get("pred_price")) is not None),
            int(not _is_blank(row.get("pred_in_stock"))))


def deduplicate_predictions(rows: list[dict]) -> list[dict]:
    best = {}
    for row in rows:
        key = _norm_url(row.get("product_url"))
        if not key:
            continue
        row = dict(row, product_url=key)
        if key not in best or candidate_completeness(row) > candidate_completeness(best[key]):
            best[key] = row
    return list(best.values())


EXTRACTION_LOG_COLUMNS = [
    "benchmark_run_id", "dataset", "variant", "page_id", "html_key", "html_file", "source",
    "extraction_status", "skip_reason", "html_read_ms", "html_parse_ms", "extract_ms",
    "candidate_count", "deduplicated_count", "returned_count", "candidate_limit_hit",
    "error_type", "error_message", "error_traceback",
]


def run_legacy_baselines(manifest: pd.DataFrame, variants=VARIANTS) -> pd.DataFrame:
    out, log = [], []
    for rec in manifest.to_dict("records"):
        common = {k: rec.get(k) for k in ["dataset", "page_id", "html_key", "html_file", "source", "skip_reason"]}
        common.update(benchmark_run_id=BENCHMARK_RUN_ID, html_read_ms=None, html_parse_ms=None)
        if not _is_blank(rec.get("skip_reason")):
            log.extend(dict(common, variant=v, extraction_status="skipped") for v in variants)
            continue
        stage = "read_error"
        try:
            started = perf_counter()
            html_text = Path(rec["html_path"]).read_text(encoding="utf-8", errors="replace")
            common["html_read_ms"] = (perf_counter() - started) * 1000
            stage = "parse_error"
            started = perf_counter()
            soup = BeautifulSoup(html_text, "lxml")
            page_url = _first_nonempty(rec.get("page_url"), document_canonical_url(soup, rec.get("source_url") or ""))
            common["html_parse_ms"] = (perf_counter() - started) * 1000
        except Exception as exc:
            log.extend(dict(common, variant=v, extraction_status=stage, error_type=type(exc).__name__,
                            error_message=str(exc), error_traceback=traceback.format_exc()) for v in variants)
            print(f"[warn] {rec['html_file']}: {type(exc).__name__}: {exc}")
            continue
        for variant in variants:
            started = perf_counter()
            record = dict(common, variant=variant, extraction_status="ok", error_type="", error_message="", error_traceback="")
            try:
                candidates = EXTRACTORS[variant](html_text, soup, page_url)
                record["candidate_count"] = len(candidates)
                rows = deduplicate_predictions(candidates)
                record["deduplicated_count"] = len(rows)
                record["candidate_limit_hit"] = len(rows) > MAX_CANDIDATES_PER_HTML
                pending = []
                for row in rows[:MAX_CANDIDATES_PER_HTML]:
                    url = row.get("product_url")
                    if not url:
                        continue
                    resolved = urljoin(page_url or rec.get("source_url") or "", str(url))
                    product_url = _norm_url(resolved)
                    if not product_url:
                        continue
                    pending.append({
                        "dataset": rec["dataset"], "page_id": rec["page_id"], "html_file": rec["html_file"],
                        "html_key": rec["html_key"], "run_id": rec["run_id"], "source": rec["source"],
                        "product_url": product_url, "variant": variant,
                        "pred_name": _clean_text(row.get("pred_name")) or None,
                        "pred_price": parse_price(row.get("pred_price")), "pred_in_stock": row.get("pred_in_stock"),
                        "extract_notes": row.get("extract_notes", ""), "html_path": rec["html_path"],
                    })
                out.extend(pending)
                record["returned_count"] = len(pending)
            except Exception as exc:
                record.update(extraction_status="extract_error", returned_count=0, error_type=type(exc).__name__,
                              error_message=str(exc), error_traceback=traceback.format_exc())
                print(f"[warn] {rec['html_file']} {variant}: {type(exc).__name__}: {exc}")
            record["extract_ms"] = (perf_counter() - started) * 1000
            log.append(record)
    columns = ["dataset", "page_id", "html_file", "html_key", "run_id", "source", "product_url", "variant",
               "pred_name", "pred_price", "pred_in_stock", "extract_notes", "html_path"]
    pred = pd.DataFrame(out, columns=columns)
    if not pred.empty:
        pred = pred.sort_values(["dataset", "variant", "html_key", "product_url"]).reset_index(drop=True)
    pred.attrs["extraction_log"] = log
    return pred


ORIGINAL_VARIANTS = ("original_fast", "original_fast_query")


def original_parser_contract(variants):
    if not set(variants) & set(ORIGINAL_VARIANTS):
        return None
    return {"bundle_sha256": ORIGINAL_PARSER_META["bundle_sha256"],
            "worker_sha256": ORIGINAL_PARSER_META["worker_sha256"],
            "limit": 500, "source_selectors": "built-in profiles only", "variants": list(variants)}


def prepare_original_runtime():
    import os
    import shutil
    import subprocess
    import zlib

    node = shutil.which("node")
    if not node:
        raise RuntimeError("Original parser requires Node.js (reference: 24.6.0). Install Node.js and restart the kernel. npm and PriceTracker are not required.")
    env = {k: v for k, v in os.environ.items() if k.upper() not in {"NODE_PATH", "NODE_OPTIONS"}}
    version = subprocess.check_output([node, "--version"], text=True, env=env).strip()
    if int(version.lstrip("v").split(".")[0]) < 24:
        raise RuntimeError(f"Use Node.js 24 or newer; reference runtime is 24.6.0, found {version}")
    bundle = zlib.decompress((ORIGINAL_PARSER_ASSET_DIR / "original_fast.cjs.zlib").read_bytes())
    if hashlib.sha256(bundle).hexdigest() != ORIGINAL_PARSER_META["bundle_sha256"]:
        raise ValueError("Embedded original parser bundle failed SHA256 verification")
    worker = (ORIGINAL_PARSER_ASSET_DIR / "worker.cjs").read_bytes()
    if hashlib.sha256(worker).hexdigest() != ORIGINAL_PARSER_META["worker_sha256"]:
        raise ValueError("Embedded original worker failed SHA256 verification")
    token = ORIGINAL_PARSER_META["bundle_sha256"][:16] + "-" + ORIGINAL_PARSER_META["worker_sha256"][:12]
    runtime = Path(ORIGINAL_RUNTIME_DIR) / token
    runtime.mkdir(parents=True, exist_ok=True)
    payloads = {"original_fast.cjs": bundle, "worker.cjs": worker}
    for name, source in ORIGINAL_SOURCE_FILES.items():
        data = source.encode("utf-8")
        if hashlib.sha256(data).hexdigest() != ORIGINAL_PARSER_META["source_sha256"][name]:
            raise ValueError(f"Embedded original source failed SHA256 verification: {name}")
        payloads[name] = data
    payloads["THIRD_PARTY_LICENSES.json"] = (ORIGINAL_PARSER_ASSET_DIR / "THIRD_PARTY_LICENSES.json").read_bytes()
    for name, data in payloads.items():
        path = runtime / name
        path.parent.mkdir(parents=True, exist_ok=True)
        if not path.is_file() or path.read_bytes() != data:
            path.write_bytes(data)
    return node, version, runtime, env


def run_original_baselines(manifest: pd.DataFrame, variants=ORIGINAL_VARIANTS) -> pd.DataFrame:
    import shutil
    import subprocess

    columns = ["dataset", "page_id", "html_file", "html_key", "run_id", "source", "product_url", "variant",
               "pred_name", "pred_price", "pred_in_stock", "extract_notes", "html_path"]
    if manifest.empty:
        result = pd.DataFrame(columns=columns)
        result.attrs["extraction_log"] = []
        return result
    if manifest.dataset.nunique() != 1 or manifest.page_id.duplicated().any():
        raise ValueError("Original extraction needs one dataset and unique page IDs")
    dataset = str(manifest.dataset.iloc[0])
    node, version, runtime, env = prepare_original_runtime()
    pages = []
    for page in manifest.to_dict("records"):
        if not _is_blank(page.get("skip_reason")):
            continue
        request = {k: _clean_text(page.get(k)) for k in ["page_id", "html_key", "source", "page_url", "source_url", "query"]}
        request["html_path"] = str(Path(page["html_path"]).resolve())
        request["content_sha256"] = _clean_text(page.get("content_sha256")) or hashlib.sha256(Path(request["html_path"]).read_bytes()).hexdigest()
        pages.append(request)
    contract = original_parser_contract(variants)
    signature = hashlib.sha256(json.dumps({"requests": pages, "contract": contract, "node": version},
                                          sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    checkpoint = Path(ORIGINAL_RUNTIME_DIR) / "checkpoints" / dataset / signature
    checkpoint.mkdir(parents=True, exist_ok=True)
    inputs, responses = checkpoint / "inputs.json", checkpoint / "responses.jsonl"
    inputs.write_text(json.dumps(pages, ensure_ascii=False), encoding="utf-8")
    cached_ids = set()
    if RESUME_EXTRACTION and responses.is_file():
        for line in responses.read_text(encoding="utf-8").splitlines():
            try:
                cached_ids.add(json.loads(line)["page_id"])
            except json.JSONDecodeError:
                break  # Worker recovers an incomplete final write; earlier corruption is an error.
    command = [node, str(runtime / "worker.cjs"), str(inputs.resolve()), str(responses.resolve())]
    if RESUME_EXTRACTION:
        command.append("--resume")
    with subprocess.Popen(command, cwd=runtime, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(f"[{dataset}] {line.rstrip()}", flush=True)
        if process.wait() != 0:
            raise RuntimeError(f"Original parser process failed; checkpoints: {checkpoint}")
    records = [json.loads(line) for line in responses.read_text(encoding="utf-8").splitlines() if line]
    if len(records) != len(pages) or {r["page_id"] for r in records} != {p["page_id"] for p in pages}:
        raise ValueError("Original parser returned incomplete or duplicate page results")
    out = OUTPUT_DIR / dataset
    out.mkdir(parents=True, exist_ok=True)
    response_path = out / "original_parser_responses.jsonl"
    shutil.copyfile(responses, response_path)
    errors = [r for r in records if r["status"] != "ok"]
    if errors:
        raise RuntimeError(f"Original parser failed on {len(errors)} pages; see {response_path}: {errors[0].get('error')}")
    by_page = {r["page_id"]: r for r in records}
    predictions, logs = [], []
    for page in manifest.to_dict("records"):
        common = {k: page.get(k) for k in ["dataset", "page_id", "html_key", "html_file", "source", "skip_reason"]}
        common["benchmark_run_id"] = BENCHMARK_RUN_ID
        if not _is_blank(page.get("skip_reason")):
            logs.extend(dict(common, variant=v, extraction_status="skipped") for v in variants)
            continue
        record = by_page[page["page_id"]]
        cached = page["page_id"] in cached_ids
        for variant in variants:
            rows = record["result"]["rows"] if variant == "original_fast" else record["queried"]["rows"]
            candidates = [dict(product_url=r["product_url"], pred_name=r["name"], pred_price=r["price"],
                               pred_in_stock=r.get("in_stock"), extract_notes=record["result"]["route"]) for r in rows]
            deduplicated = deduplicate_predictions(candidates)
            for candidate in deduplicated:
                predictions.append(dict({k: page[k] for k in ["dataset", "page_id", "html_file", "html_key", "run_id", "source"]},
                                        variant=variant, html_path=page["html_path"], **candidate))
            logs.append(dict(common, variant=variant, extraction_status="ok", original_status=record["result"]["status"],
                             replay_cached=cached, html_read_ms=None if cached else record["html_read_ms"], html_parse_ms=None,
                             extract_ms=None if cached else record["extract_ms"] + (record["query_filter_ms"] if variant.endswith("_query") else 0),
                             shared_extraction_ms=None if cached else record["extract_ms"],
                             query_filter_ms=None if cached else record["query_filter_ms"], timing_scope="HTML parse + extraction; shared across modes",
                             candidate_count=len(rows), deduplicated_count=len(deduplicated), returned_count=len(deduplicated),
                             candidate_limit_hit=len(record["result"]["rows"]) >= 500, error_type="", error_message="", error_traceback=""))
    result = pd.DataFrame(predictions, columns=columns)
    result.attrs["extraction_log"] = logs
    result.attrs["original_responses_path"] = str(response_path)
    result.attrs["original_parser"] = dict(ORIGINAL_PARSER_META, node_version=version,
                                          input_fingerprint=signature, resumed_pages=len(cached_ids),
                                          parser_contract=contract)
    return result


def run_baselines(manifest: pd.DataFrame, variants=VARIANTS) -> pd.DataFrame:
    variants = tuple(variants)
    original = tuple(v for v in variants if v in ORIGINAL_VARIANTS)
    legacy = tuple(v for v in variants if v not in ORIGINAL_VARIANTS)
    if not original:
        return run_legacy_baselines(manifest, legacy)
    result = run_original_baselines(manifest, original)
    if legacy:
        old = run_legacy_baselines(manifest, legacy)
        attributes = dict(result.attrs)
        attributes["extraction_log"] = [*attributes["extraction_log"], *old.attrs.get("extraction_log", [])]
        result = pd.concat([result, old], ignore_index=True)
        result.attrs = attributes
    return result


EXTRACTION_LOG_COLUMNS += ["original_status", "replay_cached", "shared_extraction_ms", "query_filter_ms", "timing_scope"]


## 6. Page-scoped evaluation

The matching key is (relative HTML path, normalized product URL) within a dataset.
Both predictions and GT are restricted to available, fully annotated pages in
the supplied manifest, so evaluating a class/cluster slice does not add misses
from other pages. Empty pages require a negative GT annotation (is_product=0).

Results include matched_gt, missing_prediction and unmatched_prediction rows.
GT identifiers remain present for misses; unmatched predictions have no GT ID.
result_id identifies every result row; pred_id is empty for a missing prediction.
Uniqueness is checked per method. Additional classifications can be joined using
(dataset, entity_id), or (dataset, page_id) for page-level classes.

Known absent prices are not numeric price targets. Predicting a price for such
a matched product counts as price_FP/unexpected_price_FP. Unknown GT prices
remain unscored. price_presence_ok additionally checks presence/absence on
matched products. Per-row entity/name/price TP/FP/FN support grouped metrics.


In [8]:
def name_matches(pred: object, truth: object, threshold: float = NAME_THRESHOLD) -> bool:
    return False if _is_blank(pred) or _is_blank(truth) else _name_match(str(pred), str(truth), threshold)


def price_matches(pred: object, truth: object, rel_tol: float = PRICE_REL_TOL) -> bool:
    p = parse_price(pred)
    return False if p is None or _is_blank(truth) else _price_match(p, truth, rel_tol)


def stable_pred_id(dataset: str, variant: str, html_key: str, product_key: str) -> str:
    raw = "\x1f".join([dataset, variant, html_key, product_key]).encode("utf-8")
    return f"{variant}-{hashlib.sha1(raw).hexdigest()[:14]}"


def collapse_prediction_keys(pred: pd.DataFrame) -> pd.DataFrame:
    if pred.empty:
        return pred.copy()
    pred = pred.copy()
    # Also normalize cached/external predictions before joining to the current GT.
    pred["html_key"] = pred.html_key.map(normalize_html_key)
    pred["product_url"] = pred.product_url.map(_norm_url)
    pred = pred[pred.product_url.notna()]
    rows = []
    for _, group in pred.groupby(["variant", "html_key", "product_url"], sort=False):
        records = sorted(group.to_dict("records"), key=candidate_completeness, reverse=True)
        rows.append(records[0])
    return pd.DataFrame(rows, columns=pred.columns)


def _prf(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, 2 * p * r / (p + r) if p + r else 0.0


COUNT_COLUMNS = [f"{field}_{outcome}" for field in ("entity", "name", "price") for outcome in ("TP", "FP", "FN")]
EVALUATION_COLUMNS = [
    "benchmark_run_id", "result_id", "pred_id", "dataset", "variant", "page_id", "gt_id", "entity_id",
    "html_file", "html_key", "run_id", "source", "query", "product_url", "match_status",
    "gt_name", "gt_price", "gt_price_state", "pred_name", "pred_price", "name_ok", "price_ok",
    "price_presence_ok", "unexpected_price_FP", "name_similarity", "price_error_signed", "price_error_absolute",
    "price_error_relative", "price_error_relative_absolute", "error_labels", "notes", *COUNT_COLUMNS,
    "_gt_exists", "_gt_name_present", "_gt_price_present", "_pred_name_present", "_pred_price_present",
]
EXPORT_EVALUATION_COLUMNS = [c for c in EVALUATION_COLUMNS if not c.startswith("_")]
PAGE_RESULT_COLUMNS = [
    "benchmark_run_id", "dataset", "variant", "page_id", "html_file", "query", "page_status", "is_scored",
    "gt_positive_count", "prediction_count", "matched_gt", "missing_prediction", "unmatched_prediction",
    "page_has_products_ok", *COUNT_COLUMNS,
]


def evaluate_predictions(raw_pred: pd.DataFrame, gt: pd.DataFrame, manifest: pd.DataFrame,
                         variants=VARIANTS) -> tuple[pd.DataFrame, pd.DataFrame]:
    variants = tuple(variants)
    if not variants or len(set(variants)) != len(variants):
        raise ValueError("Evaluation variants must be nonempty and unique")
    if not manifest.empty and manifest.dataset.nunique() != 1:
        raise ValueError("Evaluate one dataset at a time")
    dataset_name = str(manifest.dataset.iloc[0]) if not manifest.empty else (
        str(raw_pred.dataset.iloc[0]) if not raw_pred.empty else "dataset")
    gt = align_groundtruth_to_manifest(gt, manifest)
    scoped_gt, scored_manifest = evaluation_scope(gt, manifest)
    positive = scoped_gt[scoped_gt._positive].copy()
    for column in ["gt_id", "entity_id"]:
        if positive[column].map(_is_blank).any() or positive[column].astype(str).duplicated().any():
            raise ValueError(f"Positive GT requires unique nonempty {column}")
    if positive.duplicated(["html_key", "_eval_product_key"]).any():
        raise ValueError("Duplicate positive GroundTruth page/URL key after URL normalization")
    negative_keys = set(scoped_gt.loc[~scoped_gt._positive, "html_key"])
    if negative_keys & set(positive.html_key):
        raise ValueError("A page has both positive and negative GT rows")
    pred = collapse_prediction_keys(raw_pred)
    if not pred.empty:
        pred = pred[pred.html_key.isin(scored_manifest.html_key) & pred.variant.isin(variants)
                    & pred.dataset.eq(dataset_name)]
    gt_map = {(r.html_key, str(r._eval_product_key)): r for _, r in positive.iterrows()}
    page_map = scored_manifest.set_index("html_key").to_dict("index")
    rows = []
    for variant in variants:
        pv = pred[pred.variant == variant] if not pred.empty else pred
        pred_map = {(r["html_key"], str(r["product_url"])): r for r in pv.to_dict("records")}
        for html_key, product_key in sorted(set(gt_map) | set(pred_map)):
            p = pred_map.get((html_key, product_key))
            g = gt_map.get((html_key, product_key))
            meta = page_map[html_key]
            gt_exists, pred_exists = g is not None, p is not None
            pred_name = (_clean_text(p.get("pred_name")) or None) if pred_exists else None
            pred_price = parse_price(p.get("pred_price")) if pred_exists else None
            gt_name = g.gt_name if gt_exists else None
            gt_price = g.gt_price if gt_exists else None
            state = g.gt_price_state if gt_exists else None
            has_gt_name = gt_exists and not _is_blank(gt_name)
            has_pred_name = not _is_blank(pred_name)
            has_pred_price = pred_price is not None
            name_ok = int(name_matches(pred_name, gt_name)) if has_gt_name else (0 if has_pred_name and not gt_exists else None)
            price_ok = int(price_matches(pred_price, gt_price)) if state == "present" else (
                0 if has_pred_price and (not gt_exists or state == "absent") else None)
            presence_ok = (int(has_pred_price == (state == "present"))
                           if gt_exists and pred_exists and state in {"present", "absent"} else None)
            status = "matched_gt" if gt_exists and pred_exists else (
                "missing_prediction" if gt_exists else "unmatched_prediction")
            result_id = stable_pred_id(dataset_name, variant, html_key, product_key)
            notes = status
            if pred_exists and _clean_text(p.get("extract_notes")):
                notes += "; " + _clean_text(p["extract_notes"])
            rows.append({
                "benchmark_run_id": BENCHMARK_RUN_ID, "result_id": result_id,
                "pred_id": result_id if pred_exists else None, "dataset": dataset_name, "variant": variant,
                "page_id": meta["page_id"], "gt_id": g.gt_id if gt_exists else None,
                "entity_id": g.entity_id if gt_exists else None,
                "html_file": meta["html_file"], "html_key": html_key,
                "run_id": meta["run_id"], "source": meta["source"],
                "query": _first_nonempty(g.query if gt_exists else None, meta.get("query")),
                "product_url": g.product_url if gt_exists else p["product_url"], "match_status": status,
                "gt_name": gt_name, "gt_price": gt_price, "gt_price_state": state,
                "pred_name": pred_name, "pred_price": pred_price, "name_ok": name_ok, "price_ok": price_ok,
                "price_presence_ok": presence_ok,
                "unexpected_price_FP": int(gt_exists and state == "absent" and has_pred_price), "notes": notes,
                "entity_TP": int(gt_exists and pred_exists), "entity_FP": int(not gt_exists),
                "entity_FN": int(gt_exists and not pred_exists),
                "name_TP": int(has_gt_name and name_ok == 1),
                "name_FP": int(has_pred_name and (not gt_exists or name_ok == 0)),
                "name_FN": int(has_gt_name and name_ok != 1),
                "price_TP": int(state == "present" and price_ok == 1),
                "price_FP": int(has_pred_price and (not gt_exists or state == "absent" or price_ok == 0)),
                "price_FN": int(state == "present" and price_ok != 1),
                "_gt_exists": gt_exists, "_gt_name_present": has_gt_name, "_gt_price_present": state == "present",
                "_pred_name_present": has_pred_name, "_pred_price_present": has_pred_price,
            })
    for row in rows:
        name_pair = row["_gt_name_present"] and row["_pred_name_present"]
        row["name_similarity"] = (SequenceMatcher(None, row["pred_name"].lower().strip(),
                                                  row["gt_name"].lower().strip()).ratio() if name_pair else None)
        price_pair = row["_gt_price_present"] and row["_pred_price_present"]
        error = row["pred_price"] - row["gt_price"] if price_pair else None
        relative = error / abs(row["gt_price"]) if price_pair and row["gt_price"] != 0 else None
        row.update(price_error_signed=error, price_error_absolute=abs(error) if error is not None else None,
                   price_error_relative=relative,
                   price_error_relative_absolute=abs(relative) if relative is not None else None)
        labels = []
        if row["entity_FN"]: labels.append("missing_entity")
        if row["entity_FP"]: labels.append("unexpected_entity")
        if row["match_status"] == "matched_gt":
            if row["name_ok"] == 0: labels.append("wrong_name" if row["_pred_name_present"] else "missing_name")
            if row["gt_price_state"] == "present" and row["price_ok"] == 0:
                labels.append("wrong_price" if row["_pred_price_present"] else "missing_price")
            if row["unexpected_price_FP"]: labels.append("unexpected_price")
        row["error_labels"] = ";".join(labels)
    evaluation = pd.DataFrame(rows, columns=EVALUATION_COLUMNS)
    for col in [c for c in EVALUATION_COLUMNS if c.startswith("_")]:
        evaluation[col] = evaluation[col].astype(bool)
    for col in [*COUNT_COLUMNS, "unexpected_price_FP"]:
        evaluation[col] = evaluation[col].astype("int64")
    # Nullable integer GT IDs must not become "123.0" when mixed with unmatched predictions.
    evaluation["gt_id"] = pd.Series([r["gt_id"] for r in rows], dtype=object)
    metric_rows = []
    for variant in variants:
        ev = evaluation[evaluation.variant == variant]
        counts = {c: int(ev[c].sum()) if len(ev) else 0 for c in COUNT_COLUMNS}
        row = {"benchmark_run_id": BENCHMARK_RUN_ID, "dataset": dataset_name, "variant": variant,
               "n_pages": len(scored_manifest), "n": len(positive),
               "n_priced": int(positive.gt_price_state.eq("present").sum()),
               "n_price_absent": int(positive.gt_price_state.eq("absent").sum()),
               "n_price_unannotated": int(positive.gt_price_state.eq("unannotated").sum()),
               "unexpected_price_FP": int(ev.unexpected_price_FP.sum()) if len(ev) else 0, **counts}
        for field in ["entity", "name", "price"]:
            p, r, f1 = _prf(*(counts[f"{field}_{outcome}"] for outcome in ["TP", "FP", "FN"]))
            row.update({f"{field}_P": p, f"{field}_R": r, f"{field}_F1": f1})
        metric_rows.append(row)
    return evaluation, pd.DataFrame(metric_rows)


def build_page_results(evaluation: pd.DataFrame, gt: pd.DataFrame, manifest: pd.DataFrame,
                       variants=VARIANTS) -> pd.DataFrame:
    scoped_gt, scored = evaluation_scope(gt, manifest)
    counts = scoped_gt[scoped_gt._positive].groupby("html_key").size().to_dict()
    scored_keys = set(scored.html_key)
    excluded = set(gt.loc[gt._excluded, "html_key"])
    grouped = {(v, key): group for (v, key), group in evaluation.groupby(["variant", "html_key"])}
    rows = []
    for variant in variants:
        for meta in manifest.to_dict("records"):
            key = meta["html_key"]
            ev = grouped.get((variant, key), evaluation.iloc[:0])
            is_scored = key in scored_keys
            n_gt = int(counts.get(key, 0))
            n_pred = int(ev.match_status.ne("missing_prediction").sum())
            status = _first_nonempty(meta.get("skip_reason")) or (
                "excluded_gt" if key in excluded else "unannotated" if not is_scored else
                "positive" if n_gt else "negative")
            rows.append({
                "benchmark_run_id": BENCHMARK_RUN_ID, "dataset": meta["dataset"], "variant": variant,
                "page_id": meta["page_id"], "html_file": meta["html_file"], "query": meta.get("query", ""),
                "page_status": status, "is_scored": is_scored,
                "gt_positive_count": n_gt if is_scored else None, "prediction_count": n_pred if is_scored else None,
                "matched_gt": int(ev.match_status.eq("matched_gt").sum()),
                "missing_prediction": int(ev.match_status.eq("missing_prediction").sum()),
                "unmatched_prediction": int(ev.match_status.eq("unmatched_prediction").sum()),
                "page_has_products_ok": int((n_gt > 0) == (n_pred > 0)) if is_scored else None,
                **{c: int(ev[c].sum()) if len(ev) else 0 for c in COUNT_COLUMNS},
            })
    pages = pd.DataFrame(rows, columns=PAGE_RESULT_COLUMNS)
    pages["is_scored"] = pages.is_scored.astype(bool)
    return pages


## 7. Exports

Outputs are written to baseline_results/determined/<dataset>/:

- baseline_predictions.csv: explicit GT IDs, match_status and per-row counters;
- baseline_page_results.csv: all manifest pages, including negative/unscored/missing pages;
- baseline_metrics.csv: entity, name and price metrics;
- baseline_entity_analytics.csv: GT/Pages metadata, class/cluster labels and numeric errors;
- baseline_group_metrics.csv: per-site/query/page-type/class metrics and page confusion;
- baseline_error_breakdown.csv, baseline_error_distributions.csv: chart-ready error counts;
- baseline_extraction_log.csv: measured timings, candidate counts, exceptions and skipped files;
- baseline_analysis_profile.json: distributions, table schemas and metric definitions;
- baseline_raw_predictions.csv, groundtruth_loaded.csv, html_manifest.csv;
- groundtruth_audit.json and benchmark_run.json: scope, GT fingerprint and settings.

Additional GT classification columns are preserved in groundtruth_loaded.csv.
Join by dataset/entity_id for products and dataset/page_id for pages. Include
benchmark_run_id when combining multiple executions. Empty exports keep their
column schema. See BENCHMARK.md for the join contract and price-state semantics.

- baseline_review.xlsx: GT and method names/prices side by side, extra predictions, metrics and page diagnostics.
- baseline_review_all.xlsx in the output root combines all included datasets.


In [9]:

DISPLAY_NAMES = {
    "original_fast": "Original fast-renderer",
    "original_fast_query": "Original + query filter",
    "b1_jsonld": "B1 JSON-LD",
    "b2_css": "B2 CSS heuristics",
    "b3_regex": "B3 Regex/text",
}

METRIC_COLUMNS = [
    "benchmark_run_id", "dataset", "variant", "n_pages", "n", "n_priced", "n_price_absent", "n_price_unannotated",
    "entity_TP", "entity_FP", "entity_FN", "entity_P", "entity_R", "entity_F1", "unexpected_price_FP",
    "name_TP", "name_FP", "name_FN", "name_P", "name_R", "name_F1",
    "price_TP", "price_FP", "price_FN", "price_P", "price_R", "price_F1",
]


def export_dataset_outputs(
    dataset_name: str,
    manifest: pd.DataFrame,
    raw_pred: pd.DataFrame,
    gt: pd.DataFrame | None,
    audit: dict | None,
    evaluation: pd.DataFrame | None,
    metrics: pd.DataFrame | None,
) -> dict[str, Path]:
    out = OUTPUT_DIR / dataset_name
    out.mkdir(parents=True, exist_ok=True)

    paths = {}
    manifest_path = out / "html_manifest.csv"
    raw_path = out / "baseline_raw_predictions.csv"
    manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
    raw_pred.to_csv(raw_path, index=False, encoding="utf-8-sig")
    paths["manifest"] = manifest_path
    paths["raw_predictions"] = raw_path

    if gt is not None:
        gt_export = gt.drop(columns=[c for c in gt.columns if c.startswith("_")], errors="ignore")
        gt_path = out / "groundtruth_loaded.csv"
        gt_export.to_csv(gt_path, index=False, encoding="utf-8-sig")
        paths["groundtruth_loaded"] = gt_path
    else:
        gt_path = out / "groundtruth_loaded.csv"
        pd.DataFrame(columns=["gt_id", "entity_id", "page_id", "html_file"]).to_csv(gt_path, index=False, encoding="utf-8-sig")
        paths["groundtruth_loaded"] = gt_path

    if audit is not None:
        audit_path = out / "groundtruth_audit.json"
        audit_path.write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
        paths["audit"] = audit_path
    else:
        audit_path = out / "groundtruth_audit.json"
        audit_path.write_text(json.dumps({"evaluation_status": "not_scored", "reason": "audit not supplied"}), encoding="utf-8")
        paths["audit"] = audit_path

    if evaluation is not None:
        pred_path = out / "baseline_predictions.csv"
        evaluation.reindex(columns=EXPORT_EVALUATION_COLUMNS).to_csv(pred_path, index=False, encoding="utf-8-sig", na_rep="")
        paths["predictions"] = pred_path
    else:
        pred_path = out / "baseline_predictions.csv"
        pd.DataFrame(columns=EXPORT_EVALUATION_COLUMNS).to_csv(pred_path, index=False, encoding="utf-8-sig")
        paths["predictions"] = pred_path

    if metrics is not None:
        metrics_path = out / "baseline_metrics.csv"
        metrics[METRIC_COLUMNS].to_csv(metrics_path, index=False, encoding="utf-8-sig")
        paths["metrics"] = metrics_path
    else:
        metrics_path = out / "baseline_metrics.csv"
        pd.DataFrame(columns=METRIC_COLUMNS).to_csv(metrics_path, index=False, encoding="utf-8-sig")
        paths["metrics"] = metrics_path

    if raw_pred.attrs.get("original_responses_path"):
        paths["original_responses"] = Path(raw_pred.attrs["original_responses_path"])
    paths.update(export_analysis_outputs(out, manifest, raw_pred, gt, evaluation, metrics))
    paths["review_excel"] = export_review_workbook(
        out / "baseline_review.xlsx",
        evaluation if evaluation is not None else pd.DataFrame(columns=EXPORT_EVALUATION_COLUMNS),
        metrics if metrics is not None else pd.DataFrame(columns=METRIC_COLUMNS),
        pd.read_csv(paths["page_results"]), manifest,
        provenance={"dataset": dataset_name, "benchmark_run_id": BENCHMARK_RUN_ID, "original_parser": raw_pred.attrs.get("original_parser")})
    metadata = {
        "schema_version": 2, "benchmark_run_id": BENCHMARK_RUN_ID,
        "dataset": dataset_name, "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "reference_file": gt.attrs.get("source_path") if gt is not None else None,
        "reference_sha256": gt.attrs.get("reference_sha256") if gt is not None else None,
        "annotation_policy": gt.attrs.get("annotation_policy") if gt is not None else ANNOTATION_POLICY,
        "industrial_zero_means_absent_price": gt.attrs.get("zero_price_is_missing") if gt is not None else None,
        "variants": list(metrics.variant) if metrics is not None else list(VARIANTS),
        "name_threshold": NAME_THRESHOLD, "price_relative_tolerance": PRICE_REL_TOL,
        "url_policy": "lowercase; strip query, fragment, trailing slash and www host prefix",
        "manifest_selection": manifest.attrs.get("selection"),
        "manifest_sha256": hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
        "raw_predictions_sha256": hashlib.sha256(raw_path.read_bytes()).hexdigest(),
        "pandas_version": pd.__version__,
        "python_version": __import__("platform").python_version(),
        "parser_contract": original_parser_contract(tuple(metrics.variant) if metrics is not None else VARIANTS),
        "original_parser": raw_pred.attrs.get("original_parser"),
        "evaluation_status": "scored" if evaluation is not None else "not_scored",
        "files": {key: {"file": path.name, "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
                  for key, path in paths.items()},
    }
    notebook_path = BENCHMARK_ROOT / "baseline_determined.ipynb"
    if notebook_path.exists():
        metadata["notebook_sha256"] = hashlib.sha256(notebook_path.read_bytes()).hexdigest()
    metadata_path = out / "benchmark_run.json"
    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    paths["run_metadata"] = metadata_path

    return paths


def paper_table(all_metrics: pd.DataFrame) -> pd.DataFrame:
    if all_metrics.empty:
        return all_metrics
    out = all_metrics.copy()
    out["Method"] = out["variant"].map(DISPLAY_NAMES).fillna(out["variant"])
    return out[
        ["dataset", "Method", "n_pages", "n",
         "name_P", "name_R", "name_F1", "price_P", "price_R", "price_F1"]
    ].rename(columns={"dataset": "Dataset"})


def extraction_log_frame(raw_pred: pd.DataFrame, manifest: pd.DataFrame, variants=VARIANTS) -> pd.DataFrame:
    records = raw_pred.attrs.get("extraction_log")
    if records is None:
        # Cached predictions do not contain measured timings; never invent them.
        records = [dict(benchmark_run_id=BENCHMARK_RUN_ID, dataset=m["dataset"], variant=v,
                        page_id=m["page_id"], html_key=m["html_key"], html_file=m["html_file"], source=m["source"],
                        skip_reason=m.get("skip_reason", ""),
                        extraction_status="skipped" if not _is_blank(m.get("skip_reason")) else "not_recorded")
                   for m in manifest.to_dict("records") for v in variants]
    return pd.DataFrame(records, columns=EXTRACTION_LOG_COLUMNS)


def _page_metadata(manifest: pd.DataFrame) -> pd.DataFrame:
    return manifest.rename(columns={c: c if c == "page_id" or c.startswith("page_meta__") else "manifest__" + c
                                    for c in manifest.columns})


def build_entity_analytics(evaluation: pd.DataFrame, gt: pd.DataFrame, manifest: pd.DataFrame) -> pd.DataFrame:
    result = evaluation.reindex(columns=EXPORT_EVALUATION_COLUMNS).copy()
    labels = gt.loc[gt._positive.astype(bool), [c for c in gt.columns if not c.startswith("_")]].copy()
    labels = labels.rename(columns={c: "gt_meta__" + c for c in labels.columns if c != "entity_id"})
    # Remove unscored duplicate IDs from unrelated pages by restricting to the actual evaluation scope.
    labels = labels[labels.entity_id.isin(result.entity_id.dropna())]
    result = result.merge(labels, on="entity_id", how="left", validate="many_to_one")
    return result.merge(_page_metadata(manifest), on="page_id", how="left", validate="many_to_one")


def build_page_analytics(evaluation: pd.DataFrame, gt: pd.DataFrame, manifest: pd.DataFrame,
                         raw_pred: pd.DataFrame, variants=VARIANTS) -> pd.DataFrame:
    pages = build_page_results(evaluation, gt, manifest, variants)
    pages = pages.merge(_page_metadata(manifest), on="page_id", how="left", validate="many_to_one")
    for col in ["source", "run_id", "location", "route"]:
        pages[col] = pages.get("manifest__" + col, "")
    log = extraction_log_frame(raw_pred, manifest, variants)
    log_columns = ["variant", "page_id", *[c for c in EXTRACTION_LOG_COLUMNS if c not in
                   {"variant", "page_id", "benchmark_run_id", "dataset", "html_key", "html_file", "source", "skip_reason"}]]
    pages = pages.merge(log[log_columns], on=["variant", "page_id"], how="left", validate="one_to_one")
    scoped_gt, _ = evaluation_scope(gt, manifest)
    positive = scoped_gt[scoped_gt._positive]
    for state in ["present", "absent", "unannotated"]:
        count = positive[positive.gt_price_state.eq(state)].groupby("html_key").size()
        pages["gt_price_" + state + "_count"] = pages.manifest__html_key.map(count).fillna(0).astype(int)
    matched = evaluation[evaluation.match_status.eq("matched_gt")]
    diagnostics = matched.groupby(["variant", "page_id"]).agg(
        matched_name_similarity_mean=("name_similarity", "mean"),
        matched_price_MAE=("price_error_absolute", "mean"),
        matched_price_MAPE=("price_error_relative_absolute", "mean"),
        unexpected_price_FP=("unexpected_price_FP", "sum"),
    ).reset_index()
    pages = pages.merge(diagnostics, on=["variant", "page_id"], how="left", validate="one_to_one")
    pages["unexpected_price_FP"] = pages.unexpected_price_FP.fillna(0).astype(int)
    actual = pages.gt_positive_count.fillna(0).gt(0)
    predicted = pages.prediction_count.fillna(0).gt(0)
    for label, mask in {"TP": actual & predicted, "FP": ~actual & predicted,
                        "FN": actual & ~predicted, "TN": ~actual & ~predicted}.items():
        pages["page_" + label] = (pages.is_scored & mask).astype(int)
    for field in ["entity", "name", "price"]:
        values = [_prf(int(tp), int(fp), int(fn)) for tp, fp, fn in
                  zip(pages[field + "_TP"], pages[field + "_FP"], pages[field + "_FN"])]
        for i, metric in enumerate(["P", "R", "F1"]):
            pages[field + "_" + metric] = [v[i] if scored else None for v, scored in zip(values, pages.is_scored)]
    return pages


def build_group_metrics(entities: pd.DataFrame, pages: pd.DataFrame) -> pd.DataFrame:
    columns = ["benchmark_run_id", "dataset", "variant", "group_level", "group_dimension", "group_value",
               "n_pages", "n_entities", "n_predictions", "n_priced", "n_price_absent", "unexpected_price_FP",
               *COUNT_COLUMNS,
               *[f"{f}_{m}" for f in ["entity", "name", "price"] for m in ["P", "R", "F1"]],
               "page_TP", "page_FP", "page_FN", "page_TN", "page_accuracy",
               "macro_page_entity_F1", "macro_page_name_F1", "macro_page_price_F1"]
    rows = []

    def class_dimension(name):
        return any(part in name.casefold() for part in ["category", "class", "cluster", "label"])

    page_dims = [c for c in ["source", "page_status", "query", "location", "route"] if c in pages]
    page_dims += [c for c in pages if c.startswith("page_meta__") and (
        class_dimension(c) or c in {"page_meta__page_kind", "page_meta__metadata_is_blocked", "page_meta__empty_result_reason", "page_meta__mutation_axis", "page_meta__base_template", "page_meta__variant_idx"})]
    entity_dims = ["gt_price_state"] + [c for c in entities if c.startswith("gt_meta__") and class_dimension(c)]
    for level, frame, dimensions in [("page", pages[pages.is_scored], page_dims), ("entity", entities, entity_dims)]:
        for dim in dimensions:
            grouped_frame = frame.assign(_group_value=frame[dim].map(
                lambda value: "<unmatched_or_unlabeled>" if _is_blank(value) else str(value)))
            for (dataset, variant, value), group in grouped_frame.groupby(["dataset", "variant", "_group_value"], sort=False):
                counts = {c: int(group[c].sum()) for c in COUNT_COLUMNS}
                row = dict(benchmark_run_id=BENCHMARK_RUN_ID, dataset=dataset, variant=variant, group_level=level,
                           group_dimension=dim, group_value="<unmatched_or_unlabeled>" if _is_blank(value) else str(value),
                           n_pages=group.page_id.nunique(), n_entities=counts["entity_TP"] + counts["entity_FN"],
                           n_predictions=counts["entity_TP"] + counts["entity_FP"],
                           n_priced=counts["price_TP"] + counts["price_FN"],
                           n_price_absent=int(group.gt_price_absent_count.sum()) if level == "page" else int(group.gt_price_state.eq("absent").sum()),
                           unexpected_price_FP=int(group.unexpected_price_FP.sum()), **counts)
                for field in ["entity", "name", "price"]:
                    values = _prf(*(counts[f"{field}_{m}"] for m in ["TP", "FP", "FN"]))
                    row.update({f"{field}_{m}": v for m, v in zip(["P", "R", "F1"], values)})
                if level == "page":
                    row.update({"page_" + c: int(group["page_" + c].sum()) for c in ["TP", "FP", "FN", "TN"]})
                    row["page_accuracy"] = (row["page_TP"] + row["page_TN"]) / len(group)
                    row.update({"macro_page_" + f + "_F1": float(group[f + "_F1"].mean()) for f in ["entity", "name", "price"]})
                rows.append(row)
    return pd.DataFrame(rows, columns=columns)


def build_error_distributions(evaluation: pd.DataFrame, variants=VARIANTS) -> pd.DataFrame:
    rows = []
    for variant in variants:
        ev = evaluation[evaluation.variant.eq(variant)]
        for measure, column, edges in [
            ("name_similarity", "name_similarity", [i / 10 for i in range(11)]),
            ("absolute_relative_price_error", "price_error_relative_absolute", [0, .01, .05, .1, .25, .5, 1, float("inf")]),
        ]:
            values = pd.to_numeric(ev[column], errors="coerce").dropna()
            for i, (lower, upper) in enumerate(zip(edges, edges[1:])):
                mask = (values.ge(lower) if i == 0 else values.gt(lower)) & values.le(upper)
                rows.append(dict(benchmark_run_id=BENCHMARK_RUN_ID, variant=variant, measure=measure,
                                 bin_lower=lower, bin_upper=upper if math.isfinite(upper) else None,
                                 lower_inclusive=i == 0, upper_inclusive=True, count=int(mask.sum()),
                                 n_comparable_pairs=len(values)))
    return pd.DataFrame(rows, columns=["benchmark_run_id", "variant", "measure", "bin_lower", "bin_upper",
                                      "lower_inclusive", "upper_inclusive", "count", "n_comparable_pairs"])


def export_analysis_outputs(out: Path, manifest: pd.DataFrame, raw_pred: pd.DataFrame,
                            gt: pd.DataFrame | None, evaluation: pd.DataFrame | None,
                            metrics: pd.DataFrame | None) -> dict[str, Path]:
    variants = tuple(metrics.variant) if metrics is not None else VARIANTS
    ev = evaluation if evaluation is not None else pd.DataFrame(columns=EVALUATION_COLUMNS)
    if gt is None:
        gt = pd.DataFrame(columns=["html_key", "gt_price_state", "gt_price", "entity_id", "gt_id"])
        for col in ["_positive", "_excluded", "_annotated"]:
            gt[col] = pd.Series(dtype=bool)
    log = extraction_log_frame(raw_pred, manifest, variants)
    entities = build_entity_analytics(ev, gt, manifest)
    pages = build_page_analytics(ev, gt, manifest, raw_pred, variants)
    groups = build_group_metrics(entities, pages)
    errors = ev.loc[ev.error_labels.fillna("").ne(""), ["dataset", "variant", "source", "error_labels"]].copy()
    errors["error_type"] = errors.error_labels.str.split(";")
    errors = errors.explode("error_type").groupby(["dataset", "variant", "source", "error_type"]).size().rename("count").reset_index()
    distributions = build_error_distributions(ev, variants)
    distributions.insert(1, "dataset", str(manifest.dataset.iloc[0]) if not manifest.empty else str(out.name))
    tables = {
        "entity_analytics": ("baseline_entity_analytics.csv", entities),
        "page_results": ("baseline_page_results.csv", pages),
        "group_metrics": ("baseline_group_metrics.csv", groups),
        "error_breakdown": ("baseline_error_breakdown.csv", errors),
        "error_distributions": ("baseline_error_distributions.csv", distributions),
        "extraction_log": ("baseline_extraction_log.csv", log),
    }
    paths = {}
    for key, (name, frame) in tables.items():
        path = out / name
        frame.to_csv(path, index=False, encoding="utf-8-sig", na_rep="")
        paths[key] = path
    scoped, _ = evaluation_scope(gt, manifest)
    positive = scoped[scoped._positive]
    price_description = json.loads(pd.to_numeric(positive.gt_price, errors="coerce").describe(
        percentiles=[.1, .25, .5, .75, .9, .95, .99]).to_json())
    timings = log.copy()
    timings["extract_ms"] = pd.to_numeric(timings.extract_ms, errors="coerce")
    timing_summary = json.loads(timings.groupby("variant").extract_ms.describe().reset_index().to_json(orient="records"))
    profile = {
        "benchmark_run_id": BENCHMARK_RUN_ID, "gt_loaded_rows": len(gt), "scored_gt_entities": len(positive),
        "price_description": price_description,
        "scored_page_statuses": pages.loc[pages.is_scored].groupby("variant").page_status.value_counts().rename("count").reset_index().to_dict("records"),
        "extraction_statuses": log.groupby(["variant", "extraction_status"]).size().rename("count").reset_index().to_dict("records"),
        "extraction_time_ms_by_variant": timing_summary,
        "tables": {name: {"rows": len(frame), "columns": list(frame.columns)} for _, (name, frame) in tables.items()},
        "semantics": {
            "name_similarity": "SequenceMatcher on lowercase stripped names; comparable GT/prediction pairs only",
            "price_errors": "Predicted minus GT; relative error is a fraction of abs(GT), undefined for GT=0",
            "conditional_MAE_MAPE": "Computed only where both matched prices exist; use TP/FP/FN for overall quality",
            "error_breakdown": "Error categories can overlap for the same entity; counts are not additive across categories",
            "page_confusion": "Presence of at least one product on a scored page; independent of field correctness",
            "macro_page_F1": "Arithmetic mean over all scored pages in the group; undefined per-page P/R/F1 use zero",
            "gt_meta__": "Original GT columns joined by entity_id; unavailable for unmatched predictions",
            "page_meta__": "Original Pages metadata; manifest__ holds normalized manifest fields",
            "unmatched_or_unlabeled": "Retained group; never silently discard false positives without a GT class",
            "timings": "Milliseconds. HTML read/parse is shared and repeated for each variant; do not sum it across variants. Cached timings remain missing.",
        },
    }
    profile_path = out / "baseline_analysis_profile.json"
    profile_path.write_text(json.dumps(profile, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    paths["analysis_profile"] = profile_path
    return paths


def load_verified_saved_results(dataset_name: str, cfg: dict):
    """Reuse completed unselected datasets, never stale GT or edited result CSVs."""
    folder = OUTPUT_DIR / dataset_name
    metadata_path = folder / "benchmark_run.json"
    if not metadata_path.is_file():
        return None, {"dataset": dataset_name, "status": "missing_saved_results"}
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    reference = cfg.get("reference_file") or discover_dataset_reference(Path(cfg["root"]))
    expected = {
        "schema_version": 2, "variants": list(VARIANTS), "annotation_policy": ANNOTATION_POLICY,
        "parser_contract": original_parser_contract(VARIANTS),
        "name_threshold": NAME_THRESHOLD, "price_relative_tolerance": PRICE_REL_TOL,
        "url_policy": "lowercase; strip query, fragment, trailing slash and www host prefix",
    }
    for key, value in expected.items():
        if metadata.get(key) != value:
            return None, {"dataset": dataset_name, "status": "incompatible_settings", "field": key}
    if reference is None or not Path(reference).is_file() or metadata.get("reference_sha256") != hashlib.sha256(Path(reference).read_bytes()).hexdigest():
        return None, {"dataset": dataset_name, "status": "stale_groundtruth"}
    tables = {}
    for key in ["metrics", "predictions", "raw_predictions", "page_results", "manifest"]:
        item = metadata.get("files", {}).get(key)
        if not item:
            return None, {"dataset": dataset_name, "status": "missing_file_fingerprint", "field": key}
        saved = folder / item["file"]
        if not saved.is_file() or hashlib.sha256(saved.read_bytes()).hexdigest() != item["sha256"]:
            return None, {"dataset": dataset_name, "status": "modified_saved_results", "field": key}
        tables[key] = pd.read_csv(saved, low_memory=False)
    # Filename restoration or edits to any scored snapshot require a fresh run.
    for rec in tables["manifest"].to_dict("records"):
        if not _is_blank(rec.get("skip_reason")):
            continue
        html_path = Path(cfg["html_dir"]) / rec["html_key"]
        if not html_path.is_file() or hashlib.sha256(html_path.read_bytes()).hexdigest() != rec.get("content_sha256"):
            return None, {"dataset": dataset_name, "status": "stale_html", "html_file": rec["html_file"]}
    # The companion manifest defines the split even when the physical directory is shared.
    current_pages = read_pages_metadata(Path(reference))
    if current_pages is not None and set(current_pages) != set(tables["manifest"].html_key):
        return None, {"dataset": dataset_name, "status": "changed_page_selection"}
    return tables, {"dataset": dataset_name, "status": "reused", "benchmark_run_id": metadata["benchmark_run_id"],
                    "reference_sha256": metadata["reference_sha256"], "run_metadata": str(metadata_path)}


# Embedded in notebook cell 16; the notebook has no dependency on this staging file.
def build_review_comparison(evaluation: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """One GT entity per row, methods side by side; unmatched predictions stay separate."""
    identity = ["dataset", "entity_id"]
    base_columns = ["dataset", "entity_id", "gt_id", "page_id", "source", "query", "html_file",
                    "product_url", "gt_name", "gt_price", "gt_price_state"]
    method_columns = ["benchmark_run_id", "pred_name", "pred_price", "match_status", "name_similarity",
                      "name_ok", "price_ok", "price_presence_ok", "error_labels", "notes"]
    if evaluation.empty:
        return pd.DataFrame(columns=base_columns), evaluation.copy()
    known = evaluation.loc[evaluation.entity_id.map(lambda v: not _is_blank(v))].copy()
    # CSV reloads turn blank text into NaN and numeric GT IDs into floats.
    # Preserve textual IDs (including leading zeroes), normalizing numeric IDs only.
    for column in base_columns:
        if column in known:
            known[column] = known[column].map(lambda v: None if _is_blank(v) else v)
    for column in ["gt_id", "entity_id", "page_id"]:
        if column in known:
            known[column] = known[column].map(
                lambda v: None if v is None else str(int(v)) if isinstance(v, (int, float)) and not isinstance(v, bool) and float(v).is_integer() else str(v))
    if known.duplicated(["dataset", "variant", "entity_id"]).any():
        raise ValueError("Excel comparison requires one run per dataset/method and unique GT entity IDs")
    # GT fields must agree across methods; otherwise side-by-side comparison is misleading.
    base = known.reindex(columns=base_columns).drop_duplicates()
    if base.duplicated(identity).any():
        raise ValueError("Conflicting GT fields across methods in Excel comparison")
    for variant in evaluation.variant.drop_duplicates():
        part = known.loc[known.variant.eq(variant)].reindex(columns=identity + method_columns)
        part = part.rename(columns={c: f"{variant}__{c}" for c in method_columns})
        base = base.merge(part, on=identity, how="left", validate="one_to_one")
    base = base.sort_values(["dataset", "source", "page_id", "entity_id"], kind="stable").reset_index(drop=True)
    # Put business values first; keep technical join keys and diagnostics to the right.
    leading = ["source", "gt_name", "gt_price", "gt_price_state"]
    for variant in evaluation.variant.drop_duplicates():
        leading += [f"{variant}__pred_name", f"{variant}__pred_price"]
    base = base.reindex(columns=leading + [c for c in base if c not in leading])
    extras = evaluation.loc[evaluation.match_status.eq("unmatched_prediction")].copy()
    return base, extras


def export_review_workbook(path: Path, evaluation: pd.DataFrame, metrics: pd.DataFrame,
                           pages: pd.DataFrame | None = None, manifest: pd.DataFrame | None = None,
                           *, provenance: dict | None = None) -> Path:
    """Readable review artifact; numbers and evaluation semantics are unchanged."""
    from openpyxl import Workbook
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.formatting.rule import CellIsRule
    from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
    from openpyxl.utils import get_column_letter

    if evaluation.empty:
        evaluation = evaluation.reindex(columns=EXPORT_EVALUATION_COLUMNS)
    if metrics.empty:
        metrics = metrics.reindex(columns=METRIC_COLUMNS)

    comparison, extras = build_review_comparison(evaluation)

    matched = (
        evaluation.loc[evaluation.match_status.eq("matched_gt")]
        if not evaluation.empty else evaluation
    )
    conditional = []
    for (dataset, variant), group in matched.groupby(["dataset", "variant"], sort=False):
        conditional.append({
            "dataset": dataset,
            "variant": variant,
            "matched_entities": len(group),
            "correct_names_on_matched": int(group.name_TP.sum()),
            "name_accuracy_on_matched": float(group.name_TP.mean()),
        })

    summary = metrics.copy()
    if conditional:
        summary = summary.merge(
            pd.DataFrame(conditional),
            on=["dataset", "variant"],
            how="left",
            validate="one_to_one",
        )

    guide = [
        (
            "Comparison",
            "One row per ground-truth product. Reference values are shown on the left, "
            "with the name and price produced by each method on the right. Filter by "
            "source and the __match_status / __name_ok / __price_ok fields.",
        ),
        (
            "Unmatched predictions",
            "Predictions with no corresponding ground-truth entity. They are not "
            "attached to another product and are counted as false positives (FP).",
        ),
        (
            "All results",
            "Detailed raw results in long format: one row per method and "
            "entity/prediction, including errors and identifiers.",
        ),
        (
            "Metrics",
            "name_F1 measures quality over the full corpus, including missing and "
            "unmatched predictions. name_accuracy_on_matched is the proportion of "
            "correct names only among products with a matched URL; it does not replace F1.",
        ),
        (
            "Names",
            f"name_ok: SequenceMatcher applied to lowercased strings with threshold "
            f"{NAME_THRESHOLD}. A human-readable name does not by itself imply a match "
            "with the ground-truth name.",
        ),
        (
            "Price",
            f"price_ok: relative tolerance {PRICE_REL_TOL:.0%}. "
            "gt_price_state=absent means that the absence of a price was explicitly "
            "confirmed; unannotated means that the price was not annotated. "
            "An empty field is not equivalent to the numeric value 0.",
        ),
        (
            "Statuses",
            "matched_gt — URL matched the ground truth; missing_prediction — the product "
            "was missed; unmatched_prediction — an extra prediction. An empty method "
            "field means that the method produced no result for that dataset.",
        ),
        (
            "Dataset linkage",
            "Product key: (dataset, entity_id); page key: (dataset, page_id). "
            "benchmark_run_id is stored separately for each method. gt_id identifies "
            "the original ground-truth row.",
        ),
        (
            "Pages",
            "This sheet shows empty, excluded, and missing pages. They are not converted "
            "into artificial product entities on the Comparison sheet.",
        ),
        (
            "Safe display",
            "Text is written literally, including strings beginning with =. Invalid XML "
            "characters are removed only from the Excel representation, and long cells "
            "are limited to 32,767 characters. Complete data remain available in the CSV files.",
        ),
        (
            "Raw results",
            "The Excel export does not modify extraction results or recompute stored "
            "metrics. Formatting highlights FALSE values and missing entries; filters "
            "are available in the headers.",
        ),
    ]

    sheets = [
        ("Instructions", pd.DataFrame(guide, columns=["Section", "Description"])),
        ("Metrics", summary),
        ("Comparison", comparison),
        ("UnmatchedPredictions", extras),
        ("AllResults", evaluation),
        ("Pages", pages if pages is not None else pd.DataFrame()),
        ("Manifest", manifest if manifest is not None else pd.DataFrame()),
        (
            "Provenance",
            pd.DataFrame(
                [
                    {
                        "key": k,
                        "value": (
                            json.dumps(v, ensure_ascii=False, default=str)
                            if isinstance(v, (dict, list, tuple))
                            else v
                        ),
                    }
                    for k, v in (provenance or {}).items()
                ],
                columns=["key", "value"],
            ),
        ),
    ]

    workbook = Workbook()
    workbook.remove(workbook.active)
    red = PatternFill("solid", fgColor="FCE4D6")

    for title, frame in sheets:
        # Split only if an actual Excel sheet limit is reached; never silently discard rows.
        chunk_size = 1_048_575
        for chunk_index, start in enumerate(range(0, max(1, len(frame)), chunk_size)):
            sheet = workbook.create_sheet(
                title if chunk_index == 0 else f"{title}_{chunk_index + 1}"
            )
            sheet.sheet_view.zoomScale = 85
            sheet.freeze_panes = "E2" if title == "Comparison" else "A2"

            columns = list(frame.columns)
            if not columns:
                sheet.append(["No data"])
                continue

            sheet.append(columns)

            for row_index, values in enumerate(
                frame.iloc[start:start + chunk_size].itertuples(index=False, name=None),
                2,
            ):
                cleaned = []
                for value in values:
                    if isinstance(value, (dict, list, tuple)):
                        value = json.dumps(value, ensure_ascii=False, default=str)
                    elif _is_blank(value):
                        value = None
                    elif hasattr(value, "item"):
                        value = value.item()

                    if isinstance(value, str):
                        value = ILLEGAL_CHARACTERS_RE.sub("", value)[:32767]
                    cleaned.append(value)

                sheet.append(cleaned)

                for column_index in range(1, len(columns) + 1):
                    cell = sheet.cell(row_index, column_index)
                    if isinstance(cell.value, str):
                        cell.data_type = "s"
                    cell.alignment = Alignment(vertical="top", wrap_text=True)

                sheet.row_dimensions[row_index].height = 45

            sheet.auto_filter.ref = sheet.dimensions

            for i, column in enumerate(columns, 1):
                cell = sheet.cell(1, i)
                cell.font = Font(bold=True, color="FFFFFF")
                cell.fill = PatternFill("solid", fgColor="24476A")
                cell.alignment = Alignment(wrap_text=True, vertical="top")

                key = str(column)
                width = 20

                if key.endswith(("gt_name", "pred_name")):
                    width = 55
                elif key in {"Description", "value"}:
                    width = 115
                elif key.endswith(("notes", "error_labels")):
                    width = 44
                elif "url" in key or key.endswith(
                    ("entity_id", "page_id", "benchmark_run_id")
                ):
                    width = 32
                elif "price" in key or "similarity" in key:
                    width = 17

                letter = get_column_letter(i)
                sheet.column_dimensions[letter].width = width

                if sheet.max_row > 1 and key.endswith(
                    ("name_ok", "price_ok", "price_presence_ok")
                ):
                    sheet.conditional_formatting.add(
                        f"{letter}2:{letter}{sheet.max_row}",
                        CellIsRule(operator="equal", formula=["FALSE"], fill=red),
                    )

                if sheet.max_row > 1 and key.endswith("match_status"):
                    sheet.conditional_formatting.add(
                        f"{letter}2:{letter}{sheet.max_row}",
                        CellIsRule(
                            operator="equal",
                            formula=['"missing_prediction"'],
                            fill=red,
                        ),
                    )

            sheet.row_dimensions[1].height = 44

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pending = path.with_name(path.stem + ".pending.xlsx")
    workbook.save(pending)
    pending.replace(path)
    return path


## 8. Self-test

In [ ]:

def _self_test():
    html_doc = r"""
    <html><head>
      <link rel="canonical" href="https://demo.ru/search?q=drill">
      <script type="application/ld+json">
      {"@context":"https://schema.org","@graph":[
        {"@type":"Product","name":"Дрель Alpha 500","url":"https://demo.ru/product/1001","offers":{"@type":"Offer","price":"10000"}},
        {"@type":"Product","name":"Дрель Beta 700","url":"https://demo.ru/product/1002","offers":{"@type":"Offer","price":"12500"}}
      ]}
      </script>
    </head><body>
      <div class="product-card"><a href="https://demo.ru/product/1001" class="name">Дрель Alpha 500</a><span class="price">10 000 ₽</span></div>
      <div class="product-card"><a href="https://demo.ru/product/1002" class="name">Дрель Beta 700</a><span class="price">12 500 ₽</span></div>
    </body></html>
    """
    soup = BeautifulSoup(html_doc, "lxml")
    assert len(extract_b1_jsonld(html_doc, soup, "https://demo.ru/search")) == 2
    assert len(extract_b2_css(html_doc, soup, "https://demo.ru/search")) == 2

    assert parse_benchmark_filename("dns-shop.ru-run-none2.html") == {
        "source": "dns-shop.ru", "run_id": "run-none2"
    }
    assert parse_benchmark_filename("bigam.ru-run-ab12-2.html") == {
        "source": "bigam.ru", "run_id": "run-ab12"
    }
    assert _name_match("Дрель Alpha 500", "дрель alpha 500", 0.85)
    assert _price_match(10050, 10000, 0.01)
    assert not _price_match(10101, 10000, 0.01)
    # Regression: pending/empty GT and negative-only pages must work with pandas 3.
    from tempfile import TemporaryDirectory

    manifest = pd.DataFrame([{
        "dataset": "self-test", "html_key": "demo.html", "html_file": "demo.html",
        "page_id": "demo", "run_id": "run-none1", "source": "demo.ru", "skip_reason": "",
    }])
    cases = [
        ([], 0, 0, 0, 0),
        ([{"html_file": "demo.html", "annotation_status": "pending"}], 0, 0, 0, 0),
        ([{"html_file": "demo.html", "is_product": 0}], 1, 0, 0, 0),
        ([{"html_file": "demo.html", "is_product": 1}], 1, 1, 1, 0),
        ([{"html_file": "demo.html", "gt_name": "Product"}], 1, 1, 0, 1),
    ]
    with TemporaryDirectory() as tmp:
        reference = Path(tmp) / "groundtruth.csv"
        for records, annotated, positive, missing_name, name_fn in cases:
            pd.DataFrame(records, columns=[
                "html_file", "annotation_status", "is_product", "gt_name", "gt_price", "product_url",
            ]).to_csv(reference, index=False)
            gt = align_groundtruth_to_manifest(read_groundtruth(reference), manifest)
            # Explicit StringDtype exercises the empty-string reduction on pandas 2 as well.
            gt["gt_name"] = gt["gt_name"].astype("string")
            audit = groundtruth_audit(gt, manifest)
            assert audit["annotated_pages"] == annotated
            assert audit["positive_entities"] == positive
            assert audit["positive_missing_name_gt"] == missing_name
            assert audit["positive_missing_price_gt"] == positive
            assert audit["positive_missing_product_url_gt"] == positive
            _, metrics = evaluate_predictions(pd.DataFrame(), gt, manifest)
            assert metrics["name_FN"].eq(name_fn).all()
            assert metrics["price_FN"].eq(0).all()
        pd.DataFrame([{"html_file": "demo.html", "is_product": 1,
                       "gt_name": "Product", "gt_price": 0}]).to_csv(reference, index=False)
        industrial_gt = read_groundtruth(reference, zero_price_is_missing=True)
        assert industrial_gt["gt_price"].isna().all()
        assert read_groundtruth(reference)["gt_price"].eq(0).all()
        _, metrics = evaluate_predictions(pd.DataFrame(), industrial_gt, manifest)
        assert metrics["price_FN"].eq(0).all()
    assert all(_is_blank(value) for value in (None, float("nan"), pd.NA, "", "  "))
    return "OK"


print("Self-test:", _self_test())

# Reporting-only adapters (separate from the frozen cache contract)
REPORTING_CODE_SHA256 = '9e45b59978320713d494504cdce6add29c0341ba6d2a6f21467c8063814da731'
"""Reporting-only adapters, embedded after the frozen benchmark definitions.

They do not alter extraction, prompts, scoring or cache signatures. The adapter's
own hash is recorded separately from the frozen extraction/evaluation contract.
Pandas propagates attrs by deepcopy to each group/column: keep bulky response
ledgers on the original frame, never on the disposable deduplication workspace.
"""
import functools as _reporting_functools
import threading as _reporting_threading
import time as _reporting_time


def _reporting_original(function):
    return getattr(function, '_reporting_original_function', function)


def _reporting_collapse_adapter(function):
    original = _reporting_original(function)

    @_reporting_functools.wraps(original)
    def collapse(pred):
        # DataFrame construction preserves values/index/dtypes without copying attrs.
        # Do not clear pred.attrs: exports still require logs, records and provenance.
        if pred.empty:
            return original(pred)
        return original(pd.DataFrame(pred, copy=True))

    collapse._reporting_original_function = original
    return collapse


def _reporting_provenance():
    return dict(version=1, implementation_sha256=REPORTING_CODE_SHA256,
                source='scripts/benchmark_reporting.py; embedded in notebook',
                scope='attrs-free deduplication workspace and reporting progress only; frozen extraction/scoring/cache contract unchanged')


def _reporting_event(stage, scope, status, elapsed=None, error_type=None):
    event = dict(benchmark_run_id=BENCHMARK_RUN_ID, stage=stage, scope=scope, status=status,
                 at_utc=datetime.now(timezone.utc).isoformat(), elapsed_seconds=elapsed,
                 reporting_implementation_sha256=REPORTING_CODE_SHA256)
    if error_type:
        event['error_type'] = error_type
    folder = Path(OUTPUT_DIR)
    folder.mkdir(parents=True, exist_ok=True)
    with (folder / 'reporting_progress.jsonl').open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(event, ensure_ascii=False) + '\n')


def _reporting_stage_adapter(function, stage, scope_function, annotate_metadata=False):
    original = _reporting_original(function)

    @_reporting_functools.wraps(original)
    def timed(*args, **kwargs):
        scope = str(scope_function(args, kwargs))
        started = _reporting_time.perf_counter()
        stopped = _reporting_threading.Event()
        print(f'[{scope}] {stage}: started', flush=True)
        _reporting_event(stage, scope, 'started')

        def heartbeat():
            while not stopped.wait(30):
                print(f'[{scope}] {stage}: still running ({_reporting_time.perf_counter() - started:.0f}s)', flush=True)

        worker = _reporting_threading.Thread(target=heartbeat, daemon=True)
        worker.start()
        try:
            result = original(*args, **kwargs)
            if annotate_metadata:
                path = result['run_metadata']
                metadata = json.loads(path.read_text(encoding='utf-8'))
                metadata['reporting_implementation'] = _reporting_provenance()
                path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
            elapsed = _reporting_time.perf_counter() - started
            _reporting_event(stage, scope, 'complete', elapsed)
            print(f'[{scope}] {stage}: complete ({elapsed:.2f}s)', flush=True)
            return result
        except BaseException as exc:
            _reporting_event(stage, scope, 'interrupted' if isinstance(exc, KeyboardInterrupt) else 'failed',
                             _reporting_time.perf_counter() - started, type(exc).__name__)
            raise
        finally:
            stopped.set()
            worker.join(timeout=1)

    timed._reporting_original_function = original
    return timed


def _reporting_manifest_scope(args, kwargs):
    manifest = args[2] if len(args) > 2 else kwargs['manifest']
    return str(manifest.dataset.iloc[0]) if len(manifest) else 'empty dataset'


def _reporting_dataset_scope(args, kwargs):
    return args[0] if args else kwargs.get('dataset', kwargs.get('dataset_name', 'dataset'))


def _reporting_workbook_scope(args, kwargs):
    return Path(args[0] if args else kwargs['path']).name


collapse_prediction_keys = _reporting_collapse_adapter(collapse_prediction_keys)
evaluate_predictions = _reporting_stage_adapter(evaluate_predictions, 'Evaluate predictions', _reporting_manifest_scope)
export_dataset_outputs = _reporting_stage_adapter(export_dataset_outputs, 'Export CSV / Excel', _reporting_dataset_scope, True)
export_review_workbook = _reporting_stage_adapter(export_review_workbook, 'Write workbook', _reporting_workbook_scope)
if 'export_llm_analysis' in globals():
    export_llm_analysis = _reporting_stage_adapter(export_llm_analysis, 'Paper metrics / cost analysis', _reporting_dataset_scope)
if 'llm_compare_determined' in globals():
    llm_compare_determined = _reporting_stage_adapter(llm_compare_determined, 'Compare determined baseline', _reporting_dataset_scope)


## 9. Run the original baseline and export all analytics

Run the cells in order. By default the exact embedded original parser runs on all three datasets.
Use `RUN_DATASETS = ("industrial",)` to reproduce the reference industrial comparison only.
Keep both `original_fast` and `original_fast_query` enabled to reproduce both original rows in Excel.

`RESUME_EXTRACTION = True` resumes interrupted work only for identical HTML, requests, parser and Node
version. Cached timings are left empty. Normally keep it False for a fresh reproducible run.
The query mode shares the original extraction; its elapsed time includes extraction + query filtering.
Do not sum shared timings across the two modes.

Original response JSONL, code/dependency hashes, Node/Python versions, predictions, metrics, page/entity
analytics and Excel are saved under `baseline_results/determined/<dataset>/`.
The combined Excel is `baseline_results/determined/baseline_review_all.xlsx`.


In [11]:

results = {}
all_metrics_parts = []
all_eval_parts = []
all_raw_parts = []
all_page_parts = []

selected_datasets = list(DATASETS) if RUN_DATASETS is None else list(RUN_DATASETS)
if not selected_datasets or len(set(selected_datasets)) != len(selected_datasets) or set(selected_datasets) - set(DATASETS):
    raise ValueError(f"RUN_DATASETS must contain distinct configured names: {list(DATASETS)}")
combined_sources = []

for dataset_name in selected_datasets:
    cfg = DATASETS[dataset_name]
    dataset_root = Path(cfg["root"])
    html_dir = Path(cfg["html_dir"])
    reference_file = cfg.get("reference_file")
    if reference_file is not None:
        reference_file = Path(reference_file)
    else:
        reference_file = discover_dataset_reference(dataset_root)

    print("\n" + "=" * 80)
    print(dataset_name.upper())
    print("HTML_DIR =", html_dir)
    print("REFERENCE =", reference_file or "<not found>")

    manifest = build_html_manifest(dataset_name, html_dir, reference_file)
    print(f"HTML files: {len(manifest)}; usable: {(manifest['skip_reason'] == '').sum() if len(manifest) else 0}")

    raw_pred = run_baselines(manifest, VARIANTS)
    print(f"Raw predictions: {len(raw_pred)}")

    gt = None
    audit = None
    evaluation = None
    metrics = None

    if reference_file is not None and Path(reference_file).exists():
        gt = read_groundtruth(Path(reference_file), "GroundTruth",
                              zero_price_is_missing=(dataset_name == "industrial"),
                              annotation_policy=ANNOTATION_POLICY)

        gt = align_groundtruth_to_manifest(gt, manifest)
        audit = groundtruth_audit(gt, manifest)
        print("GroundTruth audit:")
        print(json.dumps(audit, ensure_ascii=False, indent=2))

        if FAIL_ON_INCOMPLETE_GT and not audit["paper_ready_closed_world"]:
            raise RuntimeError(f"{dataset_name}: GroundTruth is not paper-ready.")

        if audit["annotated_pages"] > 0:
            evaluation, metrics = evaluate_predictions(raw_pred, gt, manifest, VARIANTS)
            display(metrics[METRIC_COLUMNS])
            all_metrics_parts.append(metrics)
            all_eval_parts.append(evaluation)
        else:
            print("No annotated pages yet; metrics skipped.")
    else:
        print("GroundTruth not found; extraction completed, metrics skipped.")

    paths = export_dataset_outputs(
        dataset_name, manifest, raw_pred, gt, audit, evaluation, metrics
    )
    for label, path in paths.items():
        print(f"  {label}: {path}")
    all_page_parts.append(pd.read_csv(paths["page_results"]))

    all_raw_parts.append(raw_pred)
    results[dataset_name] = {
        "reference_file": str(reference_file) if reference_file else None,
        "manifest": manifest,
        "raw_predictions": raw_pred,
        "groundtruth": gt,
        "audit": audit,
        "evaluation": evaluation,
        "metrics": metrics,
        "paths": paths,
    }

# Reuse unselected datasets only if their inputs, settings and CSV fingerprints match.
for dataset_name, cfg in DATASETS.items():
    if dataset_name in selected_datasets:
        combined_sources.append({"dataset": dataset_name, "status": "computed", "benchmark_run_id": BENCHMARK_RUN_ID})
        continue
    saved, provenance = load_verified_saved_results(dataset_name, cfg)
    combined_sources.append(provenance)
    if saved is None:
        print(f"Combined tables omit {dataset_name}: {provenance['status']}. Select it in RUN_DATASETS to refresh.")
        continue
    all_metrics_parts.append(saved["metrics"])
    all_eval_parts.append(saved["predictions"])
    all_raw_parts.append(saved["raw_predictions"])
    all_page_parts.append(saved["page_results"])
    print(f"Reusing verified {dataset_name} results from {provenance['benchmark_run_id']}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "baseline_combined_manifest.json").write_text(
    json.dumps({"aggregation_run_id": BENCHMARK_RUN_ID, "datasets": combined_sources}, ensure_ascii=False, indent=2), encoding="utf-8")

all_raw_predictions = pd.concat(all_raw_parts, ignore_index=True) if all_raw_parts else pd.DataFrame()
all_metrics = pd.concat(all_metrics_parts, ignore_index=True) if all_metrics_parts else pd.DataFrame()
all_evaluation = pd.concat(all_eval_parts, ignore_index=True) if all_eval_parts else pd.DataFrame()

all_raw_predictions.to_csv(OUTPUT_DIR / "baseline_raw_predictions_all.csv", index=False, encoding="utf-8-sig")
all_evaluation.reindex(columns=EXPORT_EVALUATION_COLUMNS).to_csv(
    OUTPUT_DIR / "baseline_predictions_all.csv", index=False, encoding="utf-8-sig")
all_pages = pd.concat(all_page_parts, ignore_index=True) if all_page_parts else pd.DataFrame(columns=PAGE_RESULT_COLUMNS)
all_pages.to_csv(OUTPUT_DIR / "baseline_page_results_all.csv", index=False, encoding="utf-8-sig")
if not all_metrics.empty:
    all_metrics[METRIC_COLUMNS].to_csv(OUTPUT_DIR / "baseline_metrics_all.csv", index=False, encoding="utf-8-sig")
    # Display rounding needs no optional Jinja2 dependency; CSV keeps full precision.
    display(paper_table(all_metrics).round(3))
else:
    pd.DataFrame(columns=METRIC_COLUMNS).to_csv(OUTPUT_DIR / "baseline_metrics_all.csv", index=False, encoding="utf-8-sig")

review_path = export_review_workbook(
    OUTPUT_DIR / "baseline_review_all.xlsx", all_evaluation, all_metrics, all_pages,
    provenance={"datasets": combined_sources})
print("Excel review:", review_path)



SYNTHETIC
HTML_DIR = D:\Pet_Codes\science\article-price-tracker\synthetic\html
REFERENCE = D:\Pet_Codes\science\article-price-tracker\synthetic\synthetic_ground_truth.csv
HTML files: 20; usable: 20
[synthetic] Original parser: 0/20 saved checkpoints
[synthetic] Original parser: 10/20; ok
[synthetic] Original parser: 20/20; ok
Raw predictions: 504
GroundTruth audit:
{
  "reference_rows": 420,
  "manifest_pages": 20,
  "available_manifest_pages": 20,
  "annotated_pages": 20,
  "unannotated_manifest_pages": 0,
  "reference_pages_not_in_html_dir": 0,
  "positive_entities": 420,
  "negative_or_empty_rows": 0,
  "positive_missing_name_gt": 0,
  "positive_missing_price_gt": 0,
  "positive_confirmed_absent_price_gt": 0,
  "positive_unannotated_price_gt": 0,
  "positive_missing_product_url_gt": 0,
  "duplicate_positive_page_url_keys": 0,
  "missing_html_files": [],
  "html_hash_mismatches": [],
  "excluded_gt_pages": 0,
  "annotation_policy": "labeled",
  "annotation_status_counts": {
    "": 

,benchmark_run_id,dataset,variant,n_pages,n,n_priced,n_price_absent,n_price_unannotated,entity_TP,entity_FP,entity_FN,entity_P,entity_R,entity_F1,unexpected_price_FP,name_TP,name_FP,name_FN,name_P,name_R,name_F1,price_TP,price_FP,price_FN,price_P,price_R,price_F1
0,20261003T090603Z-d84a171c,synthetic,original_fast,20,420,420,0,0,252,0,168,1.0,0.6,0.75,0,252,0,168,1.0,0.6,0.75,252,0,168,1.0,0.6,0.75
1,20261003T090603Z-d84a171c,synthetic,original_fast_query,20,420,420,0,0,252,0,168,1.0,0.6,0.75,0,252,0,168,1.0,0.6,0.75,252,0,168,1.0,0.6,0.75


  manifest: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\synthetic\html_manifest.csv
  raw_predictions: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\synthetic\baseline_raw_predictions.csv
  groundtruth_loaded: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\synthetic\groundtruth_loaded.csv
  audit: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\synthetic\groundtruth_audit.json
  predictions: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\synthetic\baseline_predictions.csv
  metrics: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\synthetic\baseline_metrics.csv
  original_responses: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\synthetic\original_parser_responses.jsonl
  entity_analytics: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\synthetic\baseline_entity_analytics.csv
  page_results: D:\Pe

,benchmark_run_id,dataset,variant,n_pages,n,n_priced,n_price_absent,n_price_unannotated,entity_TP,entity_FP,entity_FN,entity_P,entity_R,entity_F1,unexpected_price_FP,name_TP,name_FP,name_FN,name_P,name_R,name_F1,price_TP,price_FP,price_FN,price_P,price_R,price_F1
0,20261003T090603Z-d84a171c,robustness,original_fast,200,4200,4200,0,0,2394,0,1806,1.0,0.57,0.726115,0,2394,0,1806,1.0,0.57,0.726115,2394,0,1806,1.0,0.57,0.726115
1,20261003T090603Z-d84a171c,robustness,original_fast_query,200,4200,4200,0,0,2394,0,1806,1.0,0.57,0.726115,0,2394,0,1806,1.0,0.57,0.726115,2394,0,1806,1.0,0.57,0.726115


  manifest: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\robustness\html_manifest.csv
  raw_predictions: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\robustness\baseline_raw_predictions.csv
  groundtruth_loaded: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\robustness\groundtruth_loaded.csv
  audit: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\robustness\groundtruth_audit.json
  predictions: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\robustness\baseline_predictions.csv
  metrics: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\robustness\baseline_metrics.csv
  original_responses: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\robustness\original_parser_responses.jsonl
  entity_analytics: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\robustness\baseline_entity_analytics.csv
  page_result

,Dataset,Method,n_pages,n,name_P,name_R,name_F1,price_P,price_R,price_F1
0,synthetic,Original fast-renderer,20,420,1.000,0.600,0.750,1.000,0.600,0.750
1,synthetic,Original + query filter,20,420,1.000,0.600,0.750,1.000,0.600,0.750
2,robustness,Original fast-renderer,200,4200,1.000,0.570,0.726,1.000,0.570,0.726
3,robustness,Original + query filter,200,4200,1.000,0.570,0.726,1.000,0.570,0.726
4,industrial,Original fast-renderer,239,6317,0.953,0.916,0.935,0.899,0.887,0.893
5,industrial,Original + query filter,239,6317,0.938,0.490,0.643,0.873,0.468,0.609


Excel review: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\baseline_review_all.xlsx


## 10. Scope of the paper baseline

This is the original deterministic HTML extraction mechanism, frozen in the notebook.
It includes built-in supplier profiles, but no live browser, external source configurations, database
or LLM fallback. Names/prices are evaluated with the same GT, URL normalization, thresholds and
absence-of-price policy as `original_comparison.xlsx`. LLM and hybrid runs belong in separate result folders.
